In [ ]:
# ===============================================
# CELL 1 — CONFIG + LOGGING
# ===============================================

import logging
from pathlib import Path

CONFIG = {
    "RMS_URL": "https://rms.koenig-solutions.com/Manager/frmPopularCourses.aspx",
    "TOP_N": 3,
    "WEIGHTS": {
        "feedback": 0.3,
        "availability": 0.2,
        "skill_match": 0.3,
        "demand": 0.2
    }
}

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(asctime)s - %(message)s"
)

logger = logging.getLogger("TrainerEngine")

In [ ]:
# ===============================================
# LEGACY COMPATIBILITY DEFAULTS
# ===============================================
# These defaults prevent the old prototype cells from failing if they are run
# before actual RMS/reportee data is loaded. The production flow is still main().

if "team_profiles" not in globals():
    team_profiles = []

if "course_skill_map" not in globals():
    course_skill_map = {}

print("Compatibility defaults loaded. Use main() for the real menu workflow.")


In [ ]:
# ===============================================
# CELL 2 — VALIDATION LAYER
# ===============================================

import pandas as pd

def validate_dataframe(df: pd.DataFrame, required_cols: list):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

In [ ]:
# ===============================================
# CELL 3 — SKILL GAP ENGINE
# ===============================================

def calculate_skill_gap(trainer_skills, target_skills):
    trainer_set = set(map(str.lower, trainer_skills))
    target_set = set(map(str.lower, target_skills))

    if not target_set:
        return [], 0

    gap = list(target_set - trainer_set)
    match_score = 1 - (len(gap) / len(target_set))

    return gap, round(match_score, 2)


def analyze_team_gaps(team_profiles, course_skill_map):
    records = []

    for trainer in team_profiles:
        for course, skills in course_skill_map.items():
            gap, score = calculate_skill_gap(trainer["skills"], skills)

            records.append({
                "trainer": trainer["name"],
                "course": course,
                "match_score": score,
                "missing_skills": gap
            })

    return pd.DataFrame(records)

In [ ]:
# ===============================================
# CELL 4 - RMS INGESTION (SAFE / OPTIONAL)
# ===============================================
# bs4 is optional. If it is not installed, this cell still loads and the
# production menu continues to work. RMS demand becomes an empty dataframe.

try:
    import requests
except Exception:
    requests = None

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None

from html.parser import HTMLParser


class _SimpleTableParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_cell = False
        self.in_row = False
        self.current_cell = []
        self.current_row = []
        self.rows = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "tr":
            self.in_row = True
            self.current_row = []
        elif tag.lower() in {"td", "th"} and self.in_row:
            self.in_cell = True
            self.current_cell = []

    def handle_data(self, data):
        if self.in_cell:
            self.current_cell.append(data)

    def handle_endtag(self, tag):
        if tag.lower() in {"td", "th"} and self.in_cell:
            text = " ".join("".join(self.current_cell).split())
            self.current_row.append(text)
            self.in_cell = False
        elif tag.lower() == "tr" and self.in_row:
            if self.current_row:
                self.rows.append(self.current_row)
            self.in_row = False


def _parse_rms_course_rows(html: str) -> list[dict]:
    courses = []
    if BeautifulSoup is not None:
        soup = BeautifulSoup(html, "html.parser")
        rows = []
        for row in soup.find_all("tr"):
            rows.append([c.get_text(" ", strip=True) for c in row.find_all(["td", "th"])])
    else:
        parser = _SimpleTableParser()
        parser.feed(html)
        rows = parser.rows

    for cols in rows:
        if len(cols) >= 2 and cols[0].strip():
            courses.append({
                "course_name": cols[0].strip(),
                "category": cols[1].strip(),
                "demand_score": 0.7,
            })
    return courses


def fetch_rms_courses():
    if requests is None:
        logger.warning("requests is not installed. RMS course demand skipped.")
        return pd.DataFrame(columns=["course_name", "category", "demand_score"])

    try:
        logger.info("Fetching RMS courses...")
        response = requests.get(CONFIG["RMS_URL"], timeout=10)
        response.raise_for_status()
        courses = _parse_rms_course_rows(response.text)
        if not courses:
            logger.warning("No RMS course rows parsed. Continuing with empty RMS demand.")
            return pd.DataFrame(columns=["course_name", "category", "demand_score"])
        return pd.DataFrame(courses)
    except Exception as e:
        logger.warning(f"RMS fetch skipped: {e}")
        return pd.DataFrame(columns=["course_name", "category", "demand_score"])

In [ ]:
# ===============================================
# CELL 5 — DEMAND MERGE
# ===============================================

def merge_with_demand(gap_df, rms_df):

    if rms_df.empty:
        gap_df["demand_score"] = 0.7
        return gap_df

    merged = gap_df.merge(
        rms_df[["course_name", "demand_score"]],
        left_on="course",
        right_on="course_name",
        how="left"
    )

    merged["demand_score"] = merged["demand_score"].fillna(0.65)

    return merged.drop(columns=["course_name"])

In [ ]:
# ===============================================
# CELL 6 — SCORING ENGINE
# ===============================================

def simulate_scores(df, team_profiles):

    profile_map = {t["name"]: t for t in team_profiles}

    def calc(row):
        t = profile_map.get(row["trainer"])

        if not t:
            return 0

        return round(
            t["feedback_score"] * CONFIG["WEIGHTS"]["feedback"] +
            t["availability"] * CONFIG["WEIGHTS"]["availability"] +
            row["match_score"] * CONFIG["WEIGHTS"]["skill_match"] +
            row["demand_score"] * CONFIG["WEIGHTS"]["demand"],
            3
        )

    df["assignment_probability"] = df.apply(calc, axis=1)

    return df.sort_values(by="assignment_probability", ascending=False)

In [ ]:
# ===============================================
# CELL 7 — DASHBOARD
# ===============================================

def generate_dashboard(df):

    print("\n" + "="*50)
    print("TEAM PERFORMANCE INTELLIGENCE")
    print("="*50)

    top = df.groupby("trainer").head(CONFIG["TOP_N"])

    display(top)

    print("\nACTIONABLE INSIGHTS:\n")

    for trainer in df["trainer"].unique():
        subset = df[df["trainer"] == trainer].iloc[0]

        print(f"{trainer}")
        print(f"  Target Course: {subset['course']}")
        print(f"  Probability: {subset['assignment_probability']}")
        print(f"  Skill Gap: {subset['missing_skills'] if subset['missing_skills'] else 'None'}\n")

In [ ]:
# ===============================================
# CELL 8 — ACTION ENGINE
# ===============================================

def generate_weekly_actions(df):

    actions = []

    for trainer in df["trainer"].unique():
        top = df[df["trainer"] == trainer].iloc[0]

        actions.append({
            "trainer": trainer,
            "focus_course": top["course"],
            "actions": [
                f"Complete skills: {', '.join(top['missing_skills'])}" if top["missing_skills"] else "Revise skills",
                "Update availability",
                "Align certification",
                "Target high-demand batch"
            ]
        })

    return pd.DataFrame(actions)

In [ ]:
# ===============================================
# CELL 9 — MASTER PIPELINE
# ===============================================

def run_full_pipeline(team_profiles, course_skill_map):

    logger.info("Starting legacy Trainer Intelligence Pipeline")

    if not team_profiles or not course_skill_map:
        print("Legacy pipeline skipped: team_profiles/course_skill_map are empty. Use main() for the production menu.")
        return pd.DataFrame(), pd.DataFrame()

    rms_df = fetch_rms_courses()

    gap_df = analyze_team_gaps(team_profiles, course_skill_map)

    merged_df = merge_with_demand(gap_df, rms_df)

    scored_df = simulate_scores(merged_df, team_profiles)

    generate_dashboard(scored_df)

    action_df = generate_weekly_actions(scored_df)

    display(action_df)

    return scored_df, action_df

In [ ]:
# Duplicate legacy run_full_pipeline cell removed. Use the first definition or main().


In [ ]:
"""
Cell 1 — Path configuration for SkillEdge Governance utilities.

Exposes:
- ONEDRIVE_BASE
- SKILLEDGE_ROOT
- MY_REPORTEES_FOLDER
- sanity_check()
"""

from __future__ import annotations

import os
import errno
from dataclasses import dataclass
from pathlib import Path
from typing import Final, Dict, Any


def _resolve_onedrive_base() -> Path:
    """
    Resolve OneDrive base directory with precedence:
    1) SKILLEDGE_ONEDRIVE_BASE
    2) "OneDrive - Koenig Solutions Ltd" under home
    3) "OneDrive" under home
    """
    env_override = os.getenv("SKILLEDGE_ONEDRIVE_BASE")
    if env_override:
        return Path(env_override).expanduser().resolve()

    org_specific = Path.home() / "OneDrive - Koenig Solutions Ltd"
    if org_specific.exists():
        return org_specific.resolve()

    return (Path.home() / "OneDrive").resolve()


def _ensure_dir(path: Path) -> Path:
    """Create directory if missing; return the path."""
    try:
        path.mkdir(parents=True, exist_ok=True)
        return path
    except PermissionError as exc:
        raise PermissionError(f"Insufficient permissions to create: {path}") from exc
    except FileExistsError as exc:
        raise FileExistsError(f"Path exists but is not a directory: {path}") from exc
    except OSError as exc:
        if exc.errno == errno.ENAMETOOLONG:
            raise OSError(f"Path is too long: {path}") from exc
        raise OSError(f"Failed to ensure directory: {path} ({exc})") from exc


ONEDRIVE_BASE: Final[Path] = _resolve_onedrive_base()

SKILLEDGE_ROOT: Final[Path] = Path(
    os.getenv("SKILLEDGE_ROOT", (ONEDRIVE_BASE / "SkillEdge").as_posix())
).expanduser().resolve()

MY_REPORTEES_FOLDER: Final[Path] = Path(
    os.getenv("SKILLEDGE_REPORTEES", (SKILLEDGE_ROOT / "myReportees").as_posix())
).expanduser().resolve()

_ensure_dir(SKILLEDGE_ROOT)
_ensure_dir(MY_REPORTEES_FOLDER)


@dataclass(frozen=True)
class PathStatus:
    path: Path
    exists: bool
    is_dir: bool
    readable: bool
    writable: bool
    creatable: bool
    notes: str


def _check_permissions(path: Path) -> tuple[bool, bool]:
    return (os.access(path, os.R_OK), os.access(path, os.W_OK))


def _check_creatable(path: Path) -> bool:
    try:
        test_file = path / ".sanity_check.tmp"
        with open(test_file, "w") as fh:
            fh.write("ok")
        test_file.unlink(missing_ok=True)
        return True
    except Exception:
        return False


def sanity_check() -> Dict[str, Any]:
    """
    Validate resolved paths and return a structured report.
    """
    statuses: Dict[str, PathStatus] = {}

    def assess(label: str, path: Path) -> None:
        exists = path.exists()
        is_dir = path.is_dir() if exists else False
        readable, writable = (False, False)
        creatable = False
        notes = []

        if exists and is_dir:
            readable, writable = _check_permissions(path)
            creatable = _check_creatable(path)
        elif exists and not is_dir:
            notes.append("exists-but-not-directory")
        else:
            parent = path.parent
            if parent.exists() and parent.is_dir():
                p_read, p_write = _check_permissions(parent)
                readable, writable = p_read, p_write
                creatable = _check_creatable(parent)
            else:
                notes.append("parent-missing-or-not-directory")

        statuses[label] = PathStatus(
            path=path,
            exists=exists,
            is_dir=is_dir,
            readable=readable,
            writable=writable,
            creatable=creatable,
            notes=";".join(notes),
        )

    assess("ONEDRIVE_BASE", ONEDRIVE_BASE)
    assess("SKILLEDGE_ROOT", SKILLEDGE_ROOT)
    assess("MY_REPORTEES_FOLDER", MY_REPORTEES_FOLDER)

    ok = all(s.is_dir and s.readable and (s.writable or s.creatable) for s in statuses.values())

    return {
        "env": {
            "SKILLEDGE_ONEDRIVE_BASE": os.getenv("SKILLEDGE_ONEDRIVE_BASE"),
            "SKILLEDGE_ROOT": os.getenv("SKILLEDGE_ROOT"),
            "SKILLEDGE_REPORTEES": os.getenv("SKILLEDGE_REPORTEES"),
        },
        "statuses": {k: vars(v) for k, v in statuses.items()},
        "ok": ok,
    }


# Optional: quick one-line report (comment out if too noisy)
report = sanity_check()
print("Paths OK:", report["ok"])

In [ ]:
"""
Cell 2 — Dependency bootstrap (idempotent), imports, and Chrome driver factory.
"""

from __future__ import annotations

import sys
import os
import importlib.util
import subprocess
import logging
from pathlib import Path
from typing import Optional

# ---------- Logging ----------
logger = logging.getLogger("bootstrap")
if not logger.handlers:
    handler = logging.StreamHandler(stream=sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)
logger.setLevel(os.getenv("BOOTSTRAP_LOGLEVEL", "INFO").upper())

# ---------- Version pins ----------
PINS: dict[str, str] = {
    "selenium": ">=4.18,<5",
    "webdriver-manager": ">=4,<5",
    "pandas": ">=2.0,<3",
    # Optional (if you want interactive dashboard)
    # "ipywidgets": ">=8,<9",
}

def _ensure_package(
    module_name: str,
    pip_name: Optional[str] = None,
    *,
    upgrade: bool = False,
    version_spec: Optional[str] = None,
    pip_proxy: Optional[str] = None,
) -> None:
    """Ensure 'module_name' is importable; if not, install via pip."""
    if importlib.util.find_spec(module_name) is not None:
        logger.debug(f"Package present: {module_name}")
        return

    pkg = pip_name or module_name
    pkg_spec = f"{pkg}{version_spec or ''}"

    proxy = (
        pip_proxy
        or os.getenv("PIP_PROXY")
        or os.getenv("HTTPS_PROXY")
        or os.getenv("HTTP_PROXY")
    )

    args = [
        sys.executable, "-m", "pip", "install", pkg_spec,
        "--disable-pip-version-check", "-qq"
    ]
    if upgrade:
        args.append("--upgrade")
    if proxy:
        args.extend(["--proxy", proxy])

    logger.info(f"Installing missing package: {pkg_spec}" + (f" via proxy={proxy}" if proxy else ""))
    subprocess.check_call(args, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    logger.info(f"Installed: {pkg_spec}")


def ensure_dependencies(*, upgrade: bool = False, pip_proxy: Optional[str] = None) -> None:
    """One-shot installer using PINS and proxy (idempotent)."""
    _ensure_package("selenium", "selenium", upgrade=upgrade, version_spec=PINS.get("selenium"), pip_proxy=pip_proxy)
    _ensure_package("webdriver_manager", "webdriver-manager", upgrade=upgrade, version_spec=PINS.get("webdriver-manager"), pip_proxy=pip_proxy)
    _ensure_package("pandas", "pandas", upgrade=upgrade, version_spec=PINS.get("pandas"), pip_proxy=pip_proxy)
    # Optional:
    # _ensure_package("ipywidgets", "ipywidgets", upgrade=upgrade, version_spec=PINS.get("ipywidgets"), pip_proxy=pip_proxy)


# ---- Dependency install is opt-in to avoid blocking normal notebook startup. ----
# To auto-install Selenium dependencies, set environment variable SKILLEDGE_AUTO_INSTALL_DEPS=1
# before running this notebook, or install manually in Anaconda.
if os.getenv("SKILLEDGE_AUTO_INSTALL_DEPS", "0") == "1":
    try:
        ensure_dependencies(upgrade=False)
    except Exception as exc:
        logger.warning(f"Dependency install step failed or was skipped: {exc}")
else:
    logger.info("Dependency auto-install skipped. Existing packages will be used if available.")

# ---- Imports after ensuring presence ----
import pandas as pd

SELENIUM_AVAILABLE = False
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC  # noqa: F401
    from selenium.webdriver.common.by import By  # noqa: F401
    from selenium.webdriver.common.keys import Keys  # noqa: F401
    from selenium.common.exceptions import TimeoutException, StaleElementReferenceException  # noqa: F401
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except Exception as exc:
    logger.warning(f"Selenium/webdriver imports unavailable. RMS refresh will be disabled until dependencies are installed: {exc}")
    webdriver = None
    Service = None
    WebDriverWait = None
    EC = None
    By = None
    Keys = None
    ChromeDriverManager = None
    class TimeoutException(Exception):
        pass
    class StaleElementReferenceException(Exception):
        pass

# Notebook-friendly clear_output (safe fallback outside IPython)
try:
    from IPython.display import clear_output  # noqa: F401
except Exception:
    def clear_output(*args, **kwargs):
        pass

# ---- Chrome options + driver factory ----
def build_chrome_options(
    headless: bool = False,
    download_dir: Path | None = None,
    detach: bool = True
):
    """Create a ChromeOptions instance with sensible defaults."""
    if not SELENIUM_AVAILABLE or webdriver is None:
        raise RuntimeError("Selenium is not available. Install selenium and webdriver-manager, then rerun this cell before RMS refresh.")
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
        opts.add_argument("--no-sandbox")
        opts.add_argument("--disable-gpu")
        opts.add_argument("--disable-dev-shm-usage")
    if detach:
        opts.add_experimental_option("detach", True)
    if download_dir:
        prefs = {
            "download.default_directory": str(Path(download_dir).resolve()),
            "download.prompt_for_download": False,
            "download.directory_upgrade": True,
        }
        opts.add_experimental_option("prefs", prefs)
    return opts


def make_driver(timeout_sec: int = 15, headless: bool = False):
    """Construct a Chrome WebDriver and a WebDriverWait."""
    if not SELENIUM_AVAILABLE or webdriver is None:
        raise RuntimeError("Selenium is not available. Install selenium and webdriver-manager, then rerun this cell before RMS refresh.")
    options = build_chrome_options(headless=headless)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_page_load_timeout(timeout_sec)
    wait = WebDriverWait(driver, timeout_sec)
    return driver, wait

In [ ]:
"""
Cell 3 — Text normalization, fuzzy matching utilities, and robust person-name matcher.
"""

from __future__ import annotations

import re
import unicodedata
from typing import Optional, Iterable, Sequence, Tuple, List, Dict

# Optional fast fuzzy engine
try:
    from rapidfuzz import fuzz, process  # type: ignore
    _RAPIDFUZZ = True
except Exception:
    _RAPIDFUZZ = False

# =========================
# Base normalization helpers
# =========================
_ZERO_WIDTH = re.compile(r"[\u200B-\u200D\uFEFF]")
_WHITESPACE = re.compile(r"\s+")
_NON_ALPHA_SPACE = re.compile(r"[^a-z\s]+")

def _strip_zero_width(s: str) -> str:
    return _ZERO_WIDTH.sub("", s)

def _fold_accents_to_ascii(s: str) -> str:
    nfkd = unicodedata.normalize("NFKD", s)
    return nfkd.encode("ascii", "ignore").decode("ascii")

def normalize_text(text: Optional[str]) -> str:
    """
    Normalize trainer/manager/course names:
    - remove zero-width/control artifacts
    - fold accents to ASCII
    - lowercase
    - remove non-letters (keep spaces)
    - collapse whitespace & trim
    """
    if not text:
        return ""
    s = _strip_zero_width(text)
    s = _fold_accents_to_ascii(s)
    s = s.lower()
    s = s.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    s = _NON_ALPHA_SPACE.sub(" ", s)
    s = _WHITESPACE.sub(" ", s).strip()
    return s

def _difflib_ratio(a: str, b: str) -> float:
    import difflib
    return difflib.SequenceMatcher(a=a, b=b).ratio()

def fuzzy_score(a: Optional[str], b: Optional[str]) -> int:
    """
    Return similarity score in [0, 100] after normalization.
    Uses rapidfuzz if available; falls back to difflib.
    """
    na, nb = normalize_text(a), normalize_text(b)
    if not na and not nb:
        return 100
    if not na or not nb:
        return 0
    if _RAPIDFUZZ:
        # token_sort_ratio is fine for general text
        return int(fuzz.token_sort_ratio(na, nb))
    return int(round(100 * _difflib_ratio(na, nb)))

def is_fuzzy_match(a: Optional[str], b: Optional[str], threshold: int = 90) -> bool:
    return fuzzy_score(a, b) >= threshold

def best_fuzzy_match(
    query: Optional[str],
    candidates: Iterable[str],
    *,
    threshold: int = 80,
    return_index: bool = False,
) -> Optional[Tuple[str, int] | Tuple[str, int, int]]:
    norm_query = normalize_text(query)
    if not norm_query:
        return None

    if _RAPIDFUZZ:
        cand_list = list(candidates)
        if not cand_list:
            return None
        norm_cands = [normalize_text(c) for c in cand_list]
        result = process.extractOne(norm_query, norm_cands, scorer=fuzz.token_sort_ratio)
        if not result:
            return None
        idx = result[2]
        score = int(result[1])
        if score < threshold:
            return None
        if return_index:
            return cand_list[idx], score, idx
        return cand_list[idx], score

    # Fallback
    best_val = None
    best_score = -1
    best_idx = -1
    cand_list = list(candidates)
    for i, c in enumerate(cand_list):
        s = fuzzy_score(norm_query, c)
        if s > best_score:
            best_score, best_val, best_idx = s, c, i
    if best_score >= threshold and best_val is not None:
        return (best_val, best_score, best_idx) if return_index else (best_val, best_score)
    return None

# =====================================
# Robust person-name matching extensions
# =====================================

# Common honorifics / suffixes to strip
_TITLES = {
    "mr", "mrs", "ms", "miss", "mx", "sir", "madam", "dr", "prof", "shri", "smt"
}
_SUFFIXES = {"jr", "sr", "i", "ii", "iii", "iv", "v"}

_PARENS_RE = re.compile(r"\s*\([^)]*\)\s*")  # remove "(...)" such as locations/notes

def _strip_titles_suffixes(name: str) -> str:
    """
    Remove parentheses, titles, suffixes, then normalize to 'a-z '.
    Relies on normalize_text() for final cleanup.
    """
    s = _PARENS_RE.sub(" ", name or "")
    s = normalize_text(s)
    tokens = [t for t in s.split() if t]
    if not tokens:
        return ""
    # drop leading titles
    while tokens and tokens[0] in _TITLES:
        tokens.pop(0)
    # drop trailing suffixes
    while tokens and tokens[-1] in _SUFFIXES:
        tokens.pop()
    return " ".join(tokens)

def _initial_of(tok: str) -> str:
    return tok[0] if tok else ""

def _variants_from_tokens(tokens: List[str]) -> List[str]:
    """
    Generate robust name variants:
      - First Last
      - First Middle(s) Last (full / initials)
      - First + FirstMiddleInitial + Last
      - Drop middle(s)
      - Swap order: Last First (and with middles)
      - Initials for first/last
    """
    if not tokens:
        return []

    tokens = [t for t in tokens if t]
    if len(tokens) == 1:
        return [tokens[0]]

    first = tokens[0]
    last = tokens[-1] if len(tokens) >= 2 else ""
    middles = tokens[1:-1] if len(tokens) > 2 else []

    variants = set()

    # 1) Baselines
    variants.add(f"{first} {last}".strip())
    if middles:
        full_mid = " ".join(middles)
        variants.add(f"{first} {full_mid} {last}".strip())
        mid_inits = " ".join(_initial_of(m) for m in middles if m)
        if mid_inits:
            variants.add(f"{first} {mid_inits} {last}".strip())
        # also try first middle only (full + initial)
        variants.add(f"{first} {middles[0]} {last}".strip())
        variants.add(f"{first} {_initial_of(middles[0])} {last}".strip())

    # 2) Initials combinations
    variants.add(f"{_initial_of(first)} {last}".strip())
    variants.add(f"{first} {_initial_of(last)}".strip())

    # 3) Swap order
    variants.add(f"{last} {first}".strip())
    if middles:
        variants.add(f"{last} {first} {' '.join(middles)}".strip())
        variants.add(f"{last} {first} {' '.join(_initial_of(m) for m in middles)}".strip())

    out = list(variants)
    # Prefer richer strings (longer) first (useful when logging/debug)
    out.sort(key=lambda s: (-len(s), s))
    return out

def _all_variants(name: str) -> List[str]:
    base = _strip_titles_suffixes(name)
    if not base:
        return []
    toks = base.split()
    return _variants_from_tokens(toks)

def _score_pair(a: str, b: str) -> int:
    """
    Robust multi-scorer for human names. Takes the max across several RapidFuzz scorers,
    or falls back to difflib when RapidFuzz isn't available.
    """
    if _RAPIDFUZZ:
        s1 = fuzz.token_set_ratio(a, b)
        s2 = fuzz.token_sort_ratio(a, b)
        s3 = fuzz.partial_ratio(a, b)
        try:
            s4 = fuzz.WRatio(a, b)  # weighted combo
        except Exception:
            s4 = 0
        return int(max(s1, s2, s3, s4))
    # Fallback to difflib
    return int(round(100 * _difflib_ratio(normalize_text(a), normalize_text(b))))

def match_person_name(
    cell_text: str,
    canonical_name: str,
    *,
    threshold: int = 80,
    return_debug: bool = False
) -> Tuple[bool, int, Tuple[str, str], Optional[Dict[str, int]]]:
    """
    Decide if 'cell_text' (from page) matches 'canonical_name' (e.g., RMS manager).
    - Generates variants for both sides (middle names, initials, order changes, title/paren stripping).
    - Scores all cross-pairs with multiple fuzzy scorers.
    Returns:
      (matched: bool, best_score: int, best_pair: (cell_variant, canonical_variant), debug_scores?: dict)
    """
    cell_base = _strip_titles_suffixes(cell_text)
    canon_base = _strip_titles_suffixes(canonical_name)
    if not cell_base or not canon_base:
        return False, 0, (cell_text, canonical_name), None

    cell_variants  = _all_variants(cell_base)
    canon_variants = _all_variants(canon_base)

    best_score = -1
    best_pair: Tuple[str, str] = (cell_text, canonical_name)
    debug_scores: Dict[str, int] = {}

    for v1 in cell_variants:
        for v2 in canon_variants:
            s = _score_pair(v1, v2)
            if return_debug:
                debug_scores[f"{v1}  ⇔  {v2}"] = s
            if s > best_score:
                best_score = s
                best_pair = (v1, v2)

    matched = best_score >= threshold
    if return_debug:
        # keep top 10 for readability
        top10 = dict(sorted(debug_scores.items(), key=lambda kv: kv[1], reverse=True)[:10])
        return matched, best_score, best_pair, top10
    return matched, best_score, best_pair, None

def is_manager_match(mgr_cell_text: str, rms_manager_name: str, threshold: int = 85) -> Tuple[bool, int]:
    """
    Convenience wrapper for the Trainer step.
    Returns (match_bool, score) using robust person-name matching.
    """
    ok, score, _pair, _dbg = match_person_name(mgr_cell_text, rms_manager_name, threshold=threshold, return_debug=False)
    return ok, score

In [ ]:
"""
Cell 4 — Login to RMS and read canonical manager name from <span id="lblUserName"> on Default.aspx.
"""

from __future__ import annotations

from typing import Optional, Tuple
try:
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.common.keys import Keys
except Exception:
    # Option 1 (RMS refresh) will raise a clear dependency message if Selenium is unavailable.
    By = globals().get("By")
    EC = globals().get("EC")
    Keys = globals().get("Keys")

LOGIN_URL = "https://rms.koenig-solutions.com/"
DEFAULT_URL = "https://rms.koenig-solutions.com/Default.aspx"
TRAINER_INDEX_URL = "https://rms.koenig-solutions.com/HR/trainerUtilization.aspx"
TRAINER_KPI_URL = "https://rms.koenig-solutions.com/BadgeProject/frmTrainerKpi.aspx"


def _wait_dom_ready(driver, timeout: int = 10) -> None:
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )


def _extract_text_safe(el) -> str:
    """Try .text, then textContent/innerText; return first non-empty."""
    try:
        txt = (el.text or "").strip()
        if txt:
            return txt
    except Exception:
        pass
    for attr in ("textContent", "innerText"):
        try:
            txt = (el.get_attribute(attr) or "").strip()
            if txt:
                return txt
        except Exception:
            pass
    return ""


def get_manager_name_from_default(
    driver,
    wait,
    *,
    default_url: str = DEFAULT_URL,
    timeout_sec: int = 12,
    require_visible: bool = False
) -> Optional[str]:
    """Ensure we are on Default.aspx and return the manager name from <span id="lblUserName">."""
    if "default.aspx" not in driver.current_url.lower():
        driver.get(default_url)
        _wait_dom_ready(driver, timeout=min(10, timeout_sec))

    condition = EC.visibility_of_element_located if require_visible else EC.presence_of_element_located
    locators = [
        (By.ID, "lblUserName"),
        (By.CSS_SELECTOR, "#lblUserName"),
        (By.XPATH, "//span[@id='lblUserName']"),
        (By.XPATH, "//*[contains(@id,'lblUserName')]"),
        (By.CSS_SELECTOR, "[id$='lblUserName']"),
    ]
    for by, sel in locators:
        try:
            el = WebDriverWait(driver, timeout_sec).until(condition((by, sel)))
            name = _extract_text_safe(el)
            if name:
                return name
        except Exception:
            continue

    # Last-chance: JS querySelector
    try:
        name = driver.execute_script(
            "var el = document.querySelector('#lblUserName,[id$=\"lblUserName\"]');"
            "return el ? (el.textContent||el.innerText||'').trim() : '';"
        )
        if isinstance(name, str) and name.strip():
            return name.strip()
    except Exception:
        pass
    return None


def login_and_get_driver(
    *,
    username: Optional[str] = None,
    password: Optional[str] = None,
    manager_name: Optional[str] = None,
    headless: bool = False,
    timeout_sec: int = 30,
    threshold: int = 90,
):
    """
    Login to RMS, ensure Default.aspx, read <span id="lblUserName"> as canonical manager.

    Returns:
    (driver, wait, manager_name_final, meta)
    """
    from getpass import getpass

    #user = username or os.getenv("RMS_USERNAME") or input("Enter RMS Username: ").strip()
    #pwd = password or os.getenv("RMS_PASSWORD") or getpass("Enter RMS Password: ").strip()

    user = "AISHWAR"
    #pwd = "1t5M3e+A!5w4rDx8@0609"
    pwd = "MicroLogin@94"

    print("🔐 Logging into RMS…")
    driver, wait = make_driver(timeout_sec=timeout_sec, headless=headless)
    driver.get(LOGIN_URL)

    # Login form
    username_input = wait.until(EC.presence_of_element_located((By.XPATH, "(//input[@type='text' or @type='email'])[1]")))
    password_input = wait.until(EC.presence_of_element_located((By.XPATH, "//input[@type='password']")))
    username_input.clear(); username_input.send_keys(user)
    password_input.clear(); password_input.send_keys(pwd)
    password_input.send_keys(Keys.ENTER)

    # Ensure Default.aspx (or label present)
    try:
        wait.until(EC.any_of(
            EC.url_contains("Default.aspx"),
            EC.presence_of_element_located((By.ID, "lblUserName"))
        ))
    except Exception:
        driver.get(DEFAULT_URL)

    # Read RMS display name (priority)
    rms_display_name = get_manager_name_from_default(driver, wait, require_visible=False, timeout_sec=12)

    # Canonical manager (used downstream)
    canonical = (rms_display_name or manager_name or "").strip()

    # Fuzzy validation (if user gave an expected value)
    score = None
    matched = None
    try:
        if rms_display_name and manager_name:
            score = fuzzy_score(manager_name, rms_display_name)
            matched = is_fuzzy_match(manager_name, rms_display_name, threshold=threshold)
            print(f"👤 Using RMS manager: '{rms_display_name}' (provided='{manager_name}', score={score}, match={matched})")
        elif rms_display_name:
            print(f"👤 Using RMS manager: '{rms_display_name}'")
        elif canonical:
            print(f"👤 Using provided manager: '{canonical}' (RMS label unavailable)")
        else:
            print("⚠️ No manager provided/detected; trainer model will not filter by manager.")
    except Exception:
        pass

    # Move to trainer utilization page (if needed)
    try:
        if "trainerutilization.aspx" not in driver.current_url.lower():
            driver.get(TRAINER_INDEX_URL)
    except Exception:
        pass

    meta = {
        "rms_display_name": rms_display_name,
        "provided_manager_name": manager_name,
        "score": score,
        "matched": matched,
        "trainer_kpi_url": TRAINER_KPI_URL,
        "source": "rms" if rms_display_name else ("provided" if manager_name else "unset"),
        "current_url": driver.current_url,
    }
    return driver, wait, canonical, meta

In [ ]:
def show_trainer_information(
    driver,
    wait,
    manager_name: str,
    *,
    force_rebuild: bool = False,
    preview_rows: int = 5,
    paginate: bool = True,
    manager_match_threshold: int = 85,   # <-- added back for compatibility
):
    """
    Build or load the normalized Trainer model using a DOM-level contains filter
    on the Manager column—replicating the 'Working' file behavior.

    Creates:
      RMS_Normalized_Trainer_Model.xlsx with sheets:
        - Managers:  ManagerID | ManagerName
        - Trainers:  TrainerID | TrainerName
        - Trainer_Manager_Map: TrainerID | ManagerID
    """
    from IPython.display import display as _ipy_display
    import pandas as pd
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
    from datetime import datetime

    # -------------------------------
    # Logger
    # -------------------------------
    def _log(level: str, msg: str):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] [{level}] {msg}")

    # -------------------------------
    # Helpers
    # -------------------------------
    def _normalize(s: str) -> str:
        return " ".join(str(s).lower().split())

    def _safe_display(df: pd.DataFrame, label: str):
        try:
            _ipy_display(df.head(preview_rows))
        except Exception:
            print(f"--- {label} (top {preview_rows}) ---")
            print(df.head(preview_rows).to_string(index=False))

    # -------------------------------
    # Files & constants
    # -------------------------------
    try:
        normalized_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
    except NameError:
        _log("ERROR", "MY_REPORTEES_FOLDER is not defined in this scope.")
        return None, None, None

    TRAINER_INDEX_URL = "https://rms.koenig-solutions.com/HR/trainerUtilization.aspx"

    expected_mgr_cols = ["ManagerID", "ManagerName"]
    expected_tr_cols  = ["TrainerID", "TrainerName"]
    expected_map_cols = ["TrainerID", "ManagerID"]

    # =========================================================
    # STEP 1: Load existing model (unless force_rebuild)
    # =========================================================
    if normalized_file.exists() and not force_rebuild:
        _log("INFO", "📋 Normalized Trainer Model (existing file loaded)")
        try:
            managers = pd.read_excel(normalized_file, sheet_name="Managers", engine="openpyxl")
            trainers = pd.read_excel(normalized_file, sheet_name="Trainers", engine="openpyxl")
            mapping  = pd.read_excel(normalized_file, sheet_name="Trainer_Manager_Map", engine="openpyxl")

            # Schema sanity check (order and names)
            if list(managers.columns) != expected_mgr_cols:
                _log("WARN", f"Managers sheet unexpected columns: {list(managers.columns)}")
            if list(trainers.columns) != expected_tr_cols:
                _log("WARN", f"Trainers sheet unexpected columns: {list(trainers.columns)}")
            if list(mapping.columns) != expected_map_cols:
                _log("WARN", f"Trainer_Manager_Map unexpected columns: {list(mapping.columns)}")

            # CLEAN JOINED VIEW FOR DISPLAY
            try:
                df_display = (
                    mapping
                    .merge(trainers, on="TrainerID", how="left")
                    .merge(managers, on="ManagerID", how="left")
                    [["ManagerName", "TrainerName"]]
                    .sort_values(["ManagerName", "TrainerName"])
                    .reset_index(drop=True)
                )
                print("📋 Manager → Trainer Structure\n")
                _safe_display(df_display, "Manager → Trainer")
            except Exception as e:
                _log("WARN", f"Failed to build display join: {e}")

            return managers, trainers, mapping

        except Exception as e:
            _log("WARN", f"Failed to load existing model; will attempt rebuild. Details: {e}")

    # =========================================================
    # STEP 2: Navigate to page
    # =========================================================
    print("ℹ️ Normalized file not found or rebuild requested.")
    print("🌐 Redirecting to Trainer Utilization page...\n")

    try:
        driver.get(TRAINER_INDEX_URL)
    except Exception as e:
        _log("ERROR", f"Navigation to Trainer Utilization failed: {e}")
        return None, None, None

    # =========================================================
    # STEP 3: Detect columns dynamically
    # =========================================================
    try:
        thead = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//thead[contains(@class,'thead-theme-primary') or self::thead]")
            )
        )
        headers = thead.find_elements(By.TAG_NAME, "th")

        manager_index = None
        name_index = None

        for idx, header in enumerate(headers):
            try:
                header_text = header.text.strip().lower()
            except StaleElementReferenceException:
                # Re-find if the header re-rendered
                headers = thead.find_elements(By.TAG_NAME, "th")
                header_text = headers[idx].text.strip().lower()

            if "manager" in header_text:
                manager_index = idx
            # Some UIs may show "trainer name" or "name"
            if header_text == "name" or "trainer" in header_text:
                name_index = idx

        if manager_index is None:
            raise RuntimeError("❌ Manager column not found in the table header.")
        if name_index is None:
            raise RuntimeError("❌ Name/Trainer column not found in the table header.")

        _log("INFO", f"Detected columns → manager_index={manager_index}, name_index={name_index}")

    except TimeoutException:
        _log("ERROR", "Timed out waiting for table header to load.")
        return None, None, None
    except Exception as e:
        _log("ERROR", f"Failed during header/column detection: {e}")
        return None, None, None

    # =========================================================
    # STEP 4: DOM-level filtered extraction (by Manager)
    # (faithful to 'Working' file: case-insensitive CONTAINS on manager cell)
    # =========================================================
    try:
        search_value = _normalize(manager_name)

        # NOTE:
        # We apply a case-insensitive substring match on the manager column cell:
        #  //tbody/tr[
        #    td[MGR_INDEX+1][contains(
        #        translate(normalize-space(.),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'),
        #        'aishwar c nigam'
        #    )]
        #  ]
        rows = driver.find_elements(
            By.XPATH,
            f"""
            //tbody/tr[
              td[{manager_index + 1}]
              [contains(
                 translate(normalize-space(.),
                   'ABCDEFGHIJKLMNOPQRSTUVWXYZ',
                   'abcdefghijklmnopqrstuvwxyz'
                 ),
                 '{search_value}'
              )]
            ]
            """
        )

        raw_records = []
        for row in rows:
            try:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) <= max(manager_index, name_index):
                    continue

                # Extract only first line of Trainer Name
                full_text = (cells[name_index].text or "").strip()
                trainer_name_cell = full_text.split("\n")[0].strip()

                manager_name_cell = (cells[manager_index].text or "").replace("\n", " ").strip()

                if trainer_name_cell:
                    raw_records.append({
                        "TrainerName": trainer_name_cell,
                        "ManagerName": manager_name_cell
                    })
            except StaleElementReferenceException:
                continue

        if not raw_records:
            raise RuntimeError(f"❌ No trainers found under Manager '{manager_name}'")

        df_raw = pd.DataFrame(raw_records)

    except Exception as e:
        _log("ERROR", f"Failed while scraping table rows: {e}")
        return None, None, None

    # =========================================================
    # STEP 5: Build Managers (PK)
    # =========================================================
    try:
        df_managers = (
            df_raw[["ManagerName"]]
            .drop_duplicates()
            .sort_values("ManagerName")
            .reset_index(drop=True)
        )
        df_managers["ManagerID"] = df_managers.index + 1
        df_managers = df_managers[["ManagerID", "ManagerName"]]
    except Exception as e:
        _log("ERROR", f"Failed to build Managers table: {e}")
        return None, None, None

    # =========================================================
    # STEP 6: Build Trainers (PK)
    # =========================================================
    try:
        df_trainers = (
            df_raw[["TrainerName"]]
            .drop_duplicates()
            .sort_values("TrainerName")
            .reset_index(drop=True)
        )
        df_trainers["TrainerID"] = df_trainers.index + 1
        df_trainers = df_trainers[["TrainerID", "TrainerName"]]
    except Exception as e:
        _log("ERROR", f"Failed to build Trainers table: {e}")
        return None, None, None

    # =========================================================
    # STEP 7: Build Mapping (FK–FK)
    # =========================================================
    try:
        df_map = (
            df_raw
            .merge(df_trainers, on="TrainerName")
            .merge(df_managers, on="ManagerName")
            [["TrainerID", "ManagerID"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
    except Exception as e:
        _log("ERROR", f"Failed to build Trainer_Manager_Map: {e}")
        return None, None, None

    # =========================================================
    # STEP 8: Save model
    # =========================================================
    try:
        with pd.ExcelWriter(normalized_file, engine="openpyxl") as writer:
            df_managers.to_excel(writer, sheet_name="Managers", index=False)
            df_trainers.to_excel(writer, sheet_name="Trainers", index=False)
            df_map.to_excel(writer, sheet_name="Trainer_Manager_Map", index=False)

        print(f"✅ Normalized trainer model created at:\n{normalized_file}")
        print(f"📈 Managers: {len(df_managers)}")
        print(f"📈 Trainers: {len(df_trainers)}")
        print(f"📈 Relationships: {len(df_map)}\n")

        _safe_display(df_managers, "Managers")
        _safe_display(df_trainers, "Trainers")
        _safe_display(df_map, "Trainer_Manager_Map")

        return df_managers, df_trainers, df_map

    except Exception as e:
        _log("ERROR", f"Failed to save normalized trainer model: {e}")
        return None, None, None

In [ ]:

# Replace ONLY the def line of your current show_trainer_information with this:
def show_trainer_information(
    driver,
    wait,
    manager_name: str,
    *,
    force_rebuild: bool = False,
    preview_rows: int = 5,
    paginate: bool = True,
    manager_match_threshold: int = 85,   # ← added for compatibility
):

    """
    Build or load the normalized Trainer model using a robust, ignore-case, permutation-safe
    DOM-level filter on the Manager column:

      Manager cell must contain BOTH FIRST and LAST tokens of the provided manager_name,
      in any order, with or without middle/initials, case-insensitive.

    Creates:
      RMS_Normalized_Trainer_Model.xlsx with sheets:
        - Managers:  ManagerID | ManagerName
        - Trainers:  TrainerID | TrainerName
        - Trainer_Manager_Map: TrainerID | ManagerID
    """
    # -------------------------------
    # Imports (scoped)
    # -------------------------------
    from IPython.display import display as _ipy_display
    import pandas as pd
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
    from datetime import datetime

    # -------------------------------
    # Logger
    # -------------------------------
    def _log(level: str, msg: str):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] [{level}] {msg}")

    # -------------------------------
    # Helpers
    # -------------------------------
    def _safe_display(df: pd.DataFrame, label: str):
        try:
            _ipy_display(df.head(preview_rows))
        except Exception:
            print(f"--- {label} (top {preview_rows}) ---")
            print(df.head(preview_rows).to_string(index=False))

    def _name_tokens(name: str):
        """Lowercased, accent-folded tokens for first/middles/last."""
        n = normalize_text(name or "")
        toks = [t for t in n.split() if t]
        if not toks:
            return "", [], ""
        first = toks[0]
        last  = toks[-1] if len(toks) >= 2 else ""
        middles = toks[1:-1] if len(toks) > 2 else []
        return first, middles, last

    # -------------------------------
    # Files & constants
    # -------------------------------
    try:
        normalized_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
    except NameError:
        _log("ERROR", "MY_REPORTEES_FOLDER is not defined.")
        return None, None, None

    TRAINER_INDEX_URL = "https://rms.koenig-solutions.com/HR/trainerUtilization.aspx"

    expected_mgr_cols = ["ManagerID", "ManagerName"]
    expected_tr_cols  = ["TrainerID", "TrainerName"]
    expected_map_cols = ["TrainerID", "ManagerID"]

    # =========================================================
    # STEP 1: Load existing model (unless force_rebuild)
    # =========================================================
    if normalized_file.exists() and not force_rebuild:
        _log("INFO", "📋 Normalized Trainer Model (existing file loaded)")
        try:
            managers = pd.read_excel(normalized_file, sheet_name="Managers", engine="openpyxl")
            trainers = pd.read_excel(normalized_file, sheet_name="Trainers", engine="openpyxl")
            mapping  = pd.read_excel(normalized_file, sheet_name="Trainer_Manager_Map", engine="openpyxl")

            # Schema sanity check (order and names)
            if list(managers.columns) != expected_mgr_cols:
                _log("WARN", f"Managers sheet unexpected columns: {list(managers.columns)}")
            if list(trainers.columns) != expected_tr_cols:
                _log("WARN", f"Trainers sheet unexpected columns: {list(trainers.columns)}")
            if list(mapping.columns) != expected_map_cols:
                _log("WARN", f"Trainer_Manager_Map unexpected columns: {list(mapping.columns)}")

            # Display join
            try:
                df_display = (
                    mapping
                    .merge(trainers, on="TrainerID", how="left")
                    .merge(managers, on="ManagerID", how="left")
                    [["ManagerName", "TrainerName"]]
                    .sort_values(["ManagerName", "TrainerName"])
                    .reset_index(drop=True)
                )
                print("📋 Manager → Trainer Structure\n")
                _safe_display(df_display, "Manager → Trainer")
            except Exception as e:
                _log("WARN", f"Failed to build display join: {e}")

            return managers, trainers, mapping

        except Exception as e:
            _log("WARN", f"Failed to load existing model; will attempt rebuild. Details: {e}")

    # =========================================================
    # STEP 2: Navigate to page
    # =========================================================
    print("ℹ️ Normalized file not found or rebuild requested.")
    print("🌐 Redirecting to Trainer Utilization page...\n")

    try:
        driver.get(TRAINER_INDEX_URL)
    except Exception as e:
        _log("ERROR", f"Navigation to Trainer Utilization failed: {e}")
        return None, None, None

    # =========================================================
    # STEP 3: Detect columns dynamically & anchor to that table
    # =========================================================
    def _detect_columns_and_table():
        try:
            thead = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//table//thead[contains(@class,'thead-theme-primary') or self::thead]")
                )
            )
        except TimeoutException:
            raise TimeoutException("Timed out waiting for table header to load.")

        # Anchor to the table that contains this THEAD
        try:
            table = thead.find_element(By.XPATH, "./ancestor::table[1]")
        except Exception:
            table = driver.find_element(By.XPATH, "(//table)[1]")

        headers = thead.find_elements(By.TAG_NAME, "th")
        if not headers:
            raise RuntimeError("No header cells (<th>) found in table header.")

        manager_index = None
        name_index    = None

        for idx in range(len(headers)):
            try:
                label = (headers[idx].text or "").strip().lower()
            except StaleElementReferenceException:
                headers = thead.find_elements(By.TAG_NAME, "th")
                label = (headers[idx].text or "").strip().lower()

            if manager_index is None and ("manager" in label or "reporting to" in label):
                manager_index = idx
            if name_index is None and (label == "name" or "trainer" in label):
                name_index = idx

        if manager_index is None:
            raise RuntimeError("❌ Manager column not found in the table header.")
        if name_index is None:
            raise RuntimeError("❌ Name/Trainer column not found in the table header.")

        _log("INFO", f"Detected columns → manager_index={manager_index}, name_index={name_index}")
        return table, manager_index, name_index

    try:
        table, manager_index, name_index = _detect_columns_and_table()
    except Exception as e:
        _log("ERROR", f"Failed during header/column detection: {e}")
        return None, None, None

    # =========================================================
    # STEP 4: DOM-level filtered extraction (ignore-case, permutations)
    # Requirement: manager cell must contain FIRST and LAST tokens (order agnostic).
    # =========================================================
    def _collect_rows_from(table_el, require_first: str, require_last: str) -> list[dict]:
        data: list[dict] = []

        # Build ignore-case translator
        T = "translate(normalize-space(.), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz')"

        # XPath predicate requiring both tokens in the manager column
        # NOTE: We anchor the search to the detected table to avoid wrong tbodies.
        pred = (
            f".//tbody/tr[td[{manager_index + 1}][contains({T}, '{require_first}') and contains({T}, '{require_last}')]]"
        )

        # 1) Try pure DOM-level filter first (fast)
        try:
            rows = table_el.find_elements(By.XPATH, pred)
        except Exception:
            rows = []

        # 2) If DOM filter returned nothing, fallback: take all data rows and filter in Python
        if not rows:
            try:
                rows = table_el.find_elements(By.XPATH, ".//tbody/tr[td]")
            except Exception:
                rows = []

        for row in rows:
            try:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) <= max(manager_index, name_index):
                    continue

                # Manager cell text (lowercased for token checks)
                mgr_text_raw = (cells[manager_index].text or "").replace("\n", " ").strip()
                mgr_text_n   = normalize_text(mgr_text_raw)

                # If we came through the fallback path, enforce token presence here
                if (require_first and require_last) and not (
                    (require_first in mgr_text_n) and (require_last in mgr_text_n)
                ):
                    continue

                # Trainer name: first line only
                full_text = (cells[name_index].text or "").strip()
                trainer_name = full_text.split("\n")[0].strip()

                if trainer_name:
                    data.append({
                        "TrainerName": trainer_name,
                        "ManagerName": mgr_text_raw
                    })
            except StaleElementReferenceException:
                continue

        return data

    # Parse tokens from the canonical manager_name
    first_tok, _middles, last_tok = _name_tokens(manager_name)

    # Guard: if we don't have both tokens, relax to no filter (collect everything)
    use_filter = bool(first_tok and last_tok)

    _log("INFO", f"Manager filter: {'ON' if use_filter else 'OFF'} | tokens=({first_tok}, {last_tok})")

    all_records = []

    if use_filter:
        # Collect with FIRST & LAST tokens required
        all_records = _collect_rows_from(table, first_tok, last_tok)

        # Optional pagination
        if paginate and not all_records:
            # Try to click 'Next' repeatedly (best-effort)
            candidates = [
                (By.XPATH, "//a[contains(translate(., 'NEXT', 'next'), 'next') and not(contains(@class,'disabled'))]"),
                (By.XPATH, "//li[contains(@class,'next') or contains(@class,'pagination-next')]/a[not(contains(@class,'disabled'))]"),
                (By.XPATH, "//a[normalize-space(.)='>']"),
                (By.XPATH, "//button[normalize-space(.)='Next' or contains(., 'Next')]"),
            ]
            try:
                pages = 0
                while pages < 50:
                    clicked = False
                    for by, sel in candidates:
                        try:
                            next_el = driver.find_element(by, sel)
                            if next_el.is_enabled():
                                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", next_el)
                                next_el.click()
                                # Re-detect table on each page
                                table, manager_index, name_index = _detect_columns_and_table()
                                batch = _collect_rows_from(table, first_tok, last_tok)
                                all_records.extend(batch)
                                pages += 1
                                clicked = True
                                break
                        except Exception:
                            continue
                    if not clicked:
                        break
            except Exception:
                pass

    # If still nothing (or tokens missing), collect EVERYTHING (no manager filter)
    if not all_records:
        _log("WARN", "No trainers matched with first+last tokens. Falling back to NO FILTER for this build…")
        try:
            all_rows = table.find_elements(By.XPATH, ".//tbody/tr[td]")
        except Exception:
            all_rows = []
        for row in all_rows:
            try:
                cells = row.find_elements(By.TAG_NAME, "td")
                if len(cells) <= max(manager_index, name_index):
                    continue
                full_text = (cells[name_index].text or "").strip()
                trainer_name = full_text.split("\n")[0].strip()
                mgr_text_raw = (cells[manager_index].text or "").replace("\n", " ").strip()
                if trainer_name:
                    all_records.append({"TrainerName": trainer_name, "ManagerName": mgr_text_raw})
            except StaleElementReferenceException:
                continue

    if not all_records:
        # Last-resort: save diagnostics to inspect DOM
        try:
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            png = MY_REPORTEES_FOLDER / f"trainer_utilization_debug_{ts}.png"
            html = MY_REPORTEES_FOLDER / f"trainer_utilization_debug_{ts}.html"
            driver.save_screenshot(str(png))
            with open(html, "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            _log("WARN", f"Saved diagnostics: {png.name}, {html.name}")
        except Exception:
            pass

        _log("ERROR", f"No trainers found under Manager '{manager_name}'.")
        return None, None, None

    df_raw = pd.DataFrame(all_records)

    # =========================================================
    # STEP 5: Build Managers (PK)
    # =========================================================
    try:
        df_managers = (
            df_raw[["ManagerName"]]
            .drop_duplicates()
            .sort_values("ManagerName")
            .reset_index(drop=True)
        )
        df_managers["ManagerID"] = df_managers.index + 1
        df_managers = df_managers[["ManagerID", "ManagerName"]]
    except Exception as e:
        _log("ERROR", f"Failed to build Managers table: {e}")
        return None, None, None

    # =========================================================
    # STEP 6: Build Trainers (PK)
    # =========================================================
    try:
        df_trainers = (
            df_raw[["TrainerName"]]
            .drop_duplicates()
            .sort_values("TrainerName")
            .reset_index(drop=True)
        )
        df_trainers["TrainerID"] = df_trainers.index + 1
        df_trainers = df_trainers[["TrainerID", "TrainerName"]]
    except Exception as e:
        _log("ERROR", f"Failed to build Trainers table: {e}")
        return None, None, None

    # =========================================================
    # STEP 7: Build Mapping (FK–FK)
    # =========================================================
    try:
        df_map = (
            df_raw
            .merge(df_trainers, on="TrainerName")
            .merge(df_managers, on="ManagerName")
            [["TrainerID", "ManagerID"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )
    except Exception as e:
        _log("ERROR", f"Failed to build Trainer_Manager_Map: {e}")
        return None, None, None

    # =========================================================
    # STEP 8: Save model
    # =========================================================
    try:
        with pd.ExcelWriter(normalized_file, engine="openpyxl") as writer:
            df_managers.to_excel(writer, sheet_name="Managers", index=False)
            df_trainers.to_excel(writer, sheet_name="Trainers", index=False)
            df_map.to_excel(writer, sheet_name="Trainer_Manager_Map", index=False)

        print(f"✅ Normalized trainer model created at:\n{normalized_file}")
        print(f"📈 Managers: {len(df_managers)}")
        print(f"📈 Trainers: {len(df_trainers)}")
        print(f"📈 Relationships: {len(df_map)}\n")

        _safe_display(df_managers, "Managers")
        _safe_display(df_trainers, "Trainers")
        _safe_display(df_map, "Trainer_Manager_Map")

        return df_managers, df_trainers, df_map

    except Exception as e:
        _log("ERROR", f"Failed to save normalized trainer model: {e}")
        return None, None, None

In [ ]:
"""
Cell 7 — Parse a 'Course Name' cell into structured fields.
"""

from __future__ import annotations
import re
from typing import Optional, Dict, Any

_MONTHS_RE = re.compile(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", re.I)
_PAREN_RE  = re.compile(r"\(([^)]{0,200})\)")
_ASSIGN_RE = re.compile(r"(?:total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*(\d*)", re.I)
_QUBIT_SCORE_RE = re.compile(r"current\s+qubits?\s*score\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?", re.I)
_MIN_SCORE_RE   = re.compile(r"min(?:imum)?\s*score(?:\s*required)?\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?", re.I)
_RATIO_RE       = re.compile(r"\b(\d{1,3})\s*/\s*(\d{1,3})\b")
_QCOUNT_RE      = re.compile(r"(?:total\s+count\s+of\s+qubits?\s+questions?|qubits?\s*questions?\s*count|qubit\s*question\s*count)\s*[:\-]?\s*(\d+)", re.I)
_QS_SUFFIX_RE   = re.compile(r"\b(\d+)\s*q(?:s|uestions?)\b", re.I)

def parse_course_cell(raw_text: str) -> Optional[Dict[str, Any]]:
    """
    Returns:
    {
      "CourseName": str,
      "IsFutureLive": bool,
      "AssignmentsDelivered": Optional[int],
      "QubitScore": Optional[int],
      "MinScoreRequired": Optional[int],
      "QubitQuestionCount": Optional[int],
    }
    """
    if not raw_text or not str(raw_text).strip():
        return None

    lines = [l.strip() for l in str(raw_text).splitlines() if str(l).strip()]
    if not lines:
        return None

    first_line = lines[0]
    rest_blob = " ".join(lines[1:]) if len(lines) > 1 else ""

    def _clean_course_name(value: str) -> str:
        text = str(value or "")
        text = re.sub(r"\((?:\s*total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*\d*\s*\)", "", text, flags=re.I)
        text = re.sub(r"\((?:current\s+)?qubits?.*?\)", "", text, flags=re.I)
        text = re.sub(r"\((?:min(?:imum)?\s+score|total\s+count\s+of\s+qubits?).*?\)", "", text, flags=re.I)
        text = re.sub(r"\s+", " ", text)
        return text.strip().strip("-??").strip()

    is_future_live = False
    assignments_delivered = None

    # Examine parentheses on the first line
    to_strip_chunks = []
    for chunk in _PAREN_RE.findall(first_line):
        chunk_l = chunk.strip().lower()
        if _MONTHS_RE.search(chunk_l) or "upcoming" in chunk_l or "future" in chunk_l:
            is_future_live = True
            to_strip_chunks.append(f"({chunk})")
        m_asg = _ASSIGN_RE.search(chunk)
        if m_asg:
            try:
                assignments_delivered = int(m_asg.group(1))
            except Exception:
                assignments_delivered = None
            to_strip_chunks.append(f"({chunk})")

    # Remove detected chunks from course name if auxiliary
    course_name = first_line
    for seg in to_strip_chunks:
        course_name = course_name.replace(seg, "")
    course_name = course_name.strip().strip("-–—").strip()

    # If no assignments in parentheses, scan the rest
    if assignments_delivered is None:
        m_asg2 = _ASSIGN_RE.search(rest_blob)
        if m_asg2:
            try:
                assignments_delivered = int(m_asg2.group(1))
            except Exception:
                assignments_delivered = None

    # Qubit metrics
    text_all = " ".join(lines)
    qubit_score = None
    min_score_required = None
    qubit_question_count = None

    m_qs = _QUBIT_SCORE_RE.search(text_all)
    if m_qs:
        try:
            qubit_score = int(round(float(m_qs.group(1))))
        except Exception:
            qubit_score = None

    if qubit_score is None:
        m_ratio = _RATIO_RE.search(text_all)
        if m_ratio:
            try:
                num = float(m_ratio.group(1)); den = float(m_ratio.group(2))
                if den > 0:
                    qubit_score = int(round((num / den) * 100))
            except Exception:
                qubit_score = None

    m_min = _MIN_SCORE_RE.search(text_all)
    if m_min:
        try:
            min_score_required = int(round(float(m_min.group(1))))
        except Exception:
            min_score_required = None

    m_qc = _QCOUNT_RE.search(text_all)
    if m_qc:
        try:
            qubit_question_count = int(m_qc.group(1))
        except Exception:
            qubit_question_count = None
    else:
        m_qsfx = _QS_SUFFIX_RE.search(text_all)
        if m_qsfx:
            try:
                qubit_question_count = int(m_qsfx.group(1))
            except Exception:
                qubit_question_count = None

    return {
        "CourseName": course_name,
        "IsFutureLive": is_future_live,
        "AssignmentsDelivered": assignments_delivered,
        "QubitScore": qubit_score,
        "MinScoreRequired": min_score_required,
        "QubitQuestionCount": qubit_question_count,
    }


In [ ]:
"""
Cell 6A - Vendor certification URL mapper for course and certification models.

Purpose:
- Keep RMS_Normalized_Course_Model.xlsx as the source of truth.
- For every current/future course, check whether a vendor certification/training URL can be mapped.
- Populate ExamURL plus tracking fields: CertificationVendor, CertificationName,
  CertificationStatus, CertificationCheckedOn, CertificationMatchReason.

Supported provider logic today:
- Microsoft: validated via Microsoft Learn URLs/API-style mappings.
- MongoDB: mapped to official MongoDB University certification pages.
- Alteryx: mapped to official Alteryx certification pages/resources.
- Unknown vendors: left blank with CertificationStatus = Review Needed or No Certification Match.
"""

from __future__ import annotations
from datetime import datetime
import re
from urllib.request import Request, urlopen
from urllib.error import HTTPError

CERT_TRACKING_COLUMNS = [
    "CertificationVendor",
    "CertificationName",
    "CertificationStatus",
    "CertificationCheckedOn",
    "CertificationMatchReason",
]

MSLEARN_COURSE_URLS = {
    "DP-900T00": "https://learn.microsoft.com/en-us/training/courses/dp-900t00",
    "PL-300T00": "https://learn.microsoft.com/en-us/training/courses/pl-300t00",
    "DP-700T00": "https://learn.microsoft.com/en-us/training/courses/dp-700t00",
    "AI-102T00": "https://learn.microsoft.com/en-us/training/courses/ai-102t00",
    "DP-080T00": "https://learn.microsoft.com/en-us/training/courses/dp-080t00",
    "DP-605T00": "https://learn.microsoft.com/en-us/training/courses/dp-605t00",
    "DP-100T01": "https://learn.microsoft.com/en-us/training/courses/dp-100t01",
    "AI-900T00": "https://learn.microsoft.com/en-us/training/courses/ai-900t00",
    "DP-800T00": "https://learn.microsoft.com/en-us/training/courses/dp-800t00",
    "DP-600T00": "https://learn.microsoft.com/en-us/training/courses/dp-600t00",
    "DP-604T00": "https://learn.microsoft.com/en-us/training/courses/dp-604t00",
}
MSLEARN_EXAM_URLS = {
    "DP-900": "https://learn.microsoft.com/en-us/credentials/certifications/exams/dp-900",
    "PL-300": "https://learn.microsoft.com/en-us/credentials/certifications/exams/pl-300",
    "AI-102": "https://learn.microsoft.com/en-us/credentials/certifications/exams/ai-102",
    "70-761": "https://learn.microsoft.com/en-us/credentials/certifications/exams/70-761",
    "98-381": "https://learn.microsoft.com/en-us/credentials/certifications/exams/98-381",
}
MSLEARN_TITLE_URLS = {
    "POWER BI DASHBOARD IN A DAY": "https://learn.microsoft.com/en-us/training/paths/dashboard-in-a-day",
    "GETTING STARTED WITH POWER BI": "https://learn.microsoft.com/en-us/training/modules/get-started-with-power-bi",
    "CREATE DASHBOARDS WITH POWER BI": "https://learn.microsoft.com/en-us/training/modules/create-dashboards-power-bi",
    "CREATING IMPACTFUL DASHBOARDS IN POWER BI": "https://learn.microsoft.com/en-us/training/modules/create-dashboards-power-bi",
    "INTRODUCTION TO POWER BI DAX": "https://learn.microsoft.com/en-us/training/paths/dax-power-bi",
    "EXPLORING MICROSOFT FABRIC IN A DAY": "https://learn.microsoft.com/en-us/training/paths/get-started-fabric",
}

MONGODB_CERT_URLS = [
    ("Associate Atlas Administrator", ["ATLAS", "ADMIN"], "https://learn.mongodb.com/pages/mongodb-associate-atlas-administrator-exam"),
    ("Associate Data Modeler", ["DATA", "MODEL"], "https://learn.mongodb.com/pages/mongodb-associate-data-modeler-exam"),
    ("Associate DBA", ["DBA"], "https://learn.mongodb.com/courses/database-administrator-certification-exam"),
    ("Associate DBA", ["DATABASE", "ADMIN"], "https://learn.mongodb.com/courses/database-administrator-certification-exam"),
    ("Associate Developer Node.js", ["NODE"], "https://learn.mongodb.com/courses/mongodb-associate-developer-exam-nodejs"),
    ("Associate Developer", ["DEVELOPER"], "https://learn.mongodb.com/pages/mongodb-associate-developer-exam"),
    ("MongoDB Certification Program", ["MONGODB"], "https://learn.mongodb.com/pages/mongodb-associate-developer-exam"),
]

ALTERYX_CERT_URLS = [
    ("Alteryx Designer Core", ["DESIGNER", "CORE"], "https://community.alteryx.com/t5/Certification-Resources/Designer-Core-Certification-Exam-Prep-Guide/ta-p/409403"),
    ("Alteryx Designer Advanced", ["DESIGNER", "ADVANCED"], "https://community.alteryx.com/t5/Certification-Resources/tkb-p/exam-prep"),
    ("Alteryx Designer Cloud Core", ["DESIGNER", "CLOUD"], "https://community.alteryx.com/t5/Certification-Resources/Alteryx-Designer-Cloud-Core-Exam-Prep-Guide/ta-p/1287545"),
    ("Alteryx Server Administration", ["SERVER", "ADMIN"], "https://community.alteryx.com/t5/Certification/bd-p/product-certification"),
    ("Alteryx Certification Program", ["ALTERYX"], "https://community.alteryx.com/t5/Certification/bd-p/product-certification"),
]


def _cert_norm(value) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9]+", " ", str(value or "").upper())).strip()


def validate_provider_url(url: str) -> tuple[bool, str]:
    if not url or not str(url).lower().startswith("http"):
        return False, ""
    trusted_domains = [
        "learn.microsoft.com",
        "learn.mongodb.com",
        "community.alteryx.com",
        "academy.alteryx.com",
    ]
    low = str(url).lower()
    try:
        with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=15) as response:
            return 200 <= response.status < 400, response.geturl().split("?")[0]
    except HTTPError as exc:
        if exc.code in {401, 403, 429} and any(domain in low for domain in trusted_domains):
            return True, str(url).split("?")[0]
        return 200 <= exc.code < 400, exc.geturl().split("?")[0]
    except Exception:
        if any(domain in low for domain in trusted_domains):
            return True, str(url).split("?")[0]
        return False, ""


def extract_mslearn_training_code(course_name: str) -> str:
    m = re.search(r"\b(AZ|AI|DP|PL|SC|MS|MD|MB)[- ]?(\d{3,4})\s*T(\d{2})(?:[- ]?[A-Z])?\b", str(course_name or "").upper())
    return f"{m.group(1)}-{m.group(2)}T{m.group(3)}" if m else ""


def extract_mslearn_exam_code(course_name: str) -> str:
    text = str(course_name or "").upper()
    m = re.search(r"\b(AZ|AI|DP|PL|SC|MS|MD|MB)[- ]?(\d{3,4})\s*T\d{2}(?:[- ]?[A-Z])?\b", text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    m = re.search(r"\b(AZ|AI|DP|PL|SC|MS|MD|MB)[- ]?(\d{3,4})\b", text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    m = re.search(r"\b(70|98)[- ]?(\d{3})\b", text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"
    return ""


def _match_keyword_rule(name_norm: str, rules: list[tuple[str, list[str], str]], vendor: str):
    for cert_name, keywords, url in rules:
        if all(keyword in name_norm for keyword in keywords):
            ok, final = validate_provider_url(url)
            return {
                "exists": ok,
                "url": final if ok else None,
                "vendor": vendor,
                "certification_name": cert_name,
                "status": "Mapped" if ok else "Review Needed",
                "reason": f"{vendor} keyword match: {', '.join(keywords)}" if ok else f"{vendor} candidate URL failed validation",
            }
    return None


def detect_certification_info(course_name: str) -> dict:
    name = _cert_norm(course_name)
    training_code = extract_mslearn_training_code(course_name)
    exam_code = extract_mslearn_exam_code(course_name)

    candidate = None
    cert_name = None
    reason = None

    if "EXAM PREP" in name and exam_code in MSLEARN_EXAM_URLS:
        candidate = MSLEARN_EXAM_URLS[exam_code]
        cert_name = f"Microsoft exam {exam_code}"
        reason = f"Microsoft exam code {exam_code} from Exam Prep title"
    elif training_code in MSLEARN_COURSE_URLS:
        candidate = MSLEARN_COURSE_URLS[training_code]
        cert_name = f"Microsoft course {training_code}"
        reason = f"Microsoft training course code {training_code}"
    elif name.startswith("20761C") or "QUERYING DATA WITH TRANSACT SQL 2016" in name:
        candidate = MSLEARN_EXAM_URLS["70-761"]
        cert_name = "Microsoft exam 70-761"
        reason = "legacy Microsoft exam mapping 70-761"
    elif name == "INTRODUCTION TO PROGRAMMING USING PYTHON":
        candidate = MSLEARN_EXAM_URLS["98-381"]
        cert_name = "Microsoft exam 98-381"
        reason = "exact Microsoft exam title match"
    elif name in MSLEARN_TITLE_URLS:
        candidate = MSLEARN_TITLE_URLS[name]
        cert_name = "Microsoft Learn training path/module"
        reason = f"approved Microsoft Learn title mapping: {name}"

    if candidate:
        ok, final = validate_provider_url(candidate)
        return {
            "exists": ok,
            "url": final if ok else None,
            "vendor": "Microsoft",
            "certification_name": cert_name,
            "status": "Mapped" if ok else "Review Needed",
            "reason": reason if ok else f"Microsoft candidate URL failed validation: {candidate}",
        }

    if "MONGODB" in name or "MONGO DB" in name:
        match = _match_keyword_rule(name, MONGODB_CERT_URLS, "MongoDB")
        if match:
            return match
        return {"exists": False, "url": None, "vendor": "MongoDB", "certification_name": None, "status": "Review Needed", "reason": "MongoDB course detected; no specific certification keyword matched"}

    if "ALTERYX" in name:
        match = _match_keyword_rule(name, ALTERYX_CERT_URLS, "Alteryx")
        if match:
            return match
        return {"exists": False, "url": None, "vendor": "Alteryx", "certification_name": None, "status": "Review Needed", "reason": "Alteryx course detected; no specific certification keyword matched"}

    return {"exists": False, "url": None, "vendor": None, "certification_name": None, "status": "No Certification Match", "reason": "No supported vendor certification/training match found"}


def detect_exam_info(course_name: str) -> tuple[bool, str | None]:
    """Backward-compatible wrapper used by older cells."""
    info = detect_certification_info(course_name)
    return bool(info.get("exists")), info.get("url")


def update_course_exam_urls_from_mslearn(df_courses):
    """Populate ExamURL and tracking columns for all current/future courses."""
    if df_courses is None or df_courses.empty:
        return df_courses
    df_courses = df_courses.copy()
    if "ExamURL" not in df_courses.columns:
        df_courses["ExamURL"] = None
    for col in CERT_TRACKING_COLUMNS:
        if col not in df_courses.columns:
            df_courses[col] = None

    for idx, row in df_courses.iterrows():
        info = detect_certification_info(row.get("CourseName", ""))
        df_courses.at[idx, "ExamURL"] = info.get("url") if info.get("exists") else None
        df_courses.at[idx, "CertificationVendor"] = info.get("vendor")
        df_courses.at[idx, "CertificationName"] = info.get("certification_name")
        df_courses.at[idx, "CertificationStatus"] = info.get("status")
        df_courses.at[idx, "CertificationCheckedOn"] = datetime.now().strftime("%Y-%m-%d")
        df_courses.at[idx, "CertificationMatchReason"] = info.get("reason")
    return df_courses

In [ ]:
def extract_trainer_course_mapping(driver, wait) -> None:
    """
    Step 2 — Build/extend the normalized Course model and Trainer→Course mapping
    using the Trainer Course Grade page. Idempotent, no dupes.

    Creates/updates:
      RMS_Normalized_Course_Model.xlsx
        • Courses:            CourseID | CourseName | ExamURL
        • Trainer_Course_Map: TrainerID | CourseID | IsFutureLive | AssignmentsDelivered
                              | QubitScore | MinScoreRequired | QubitQuestionCount
    """
    from datetime import datetime
    import time
    import pandas as pd
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.support import expected_conditions as EC

    def _log(level: str, msg: str):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] [{level}] {msg}")

    _log("INFO", "🔹 Building Course Model...")

    # ---------- Files & schema ----------
    trainer_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
    course_file  = MY_REPORTEES_FOLDER / "RMS_Normalized_Course_Model.xlsx"

    expected_course_cols = ["CourseID", "CourseName", "ExamURL"] + CERT_TRACKING_COLUMNS
    expected_map_cols = [
        "TrainerID", "CourseID", "IsFutureLive",
        "AssignmentsDelivered", "QubitScore",
        "MinScoreRequired", "QubitQuestionCount"
    ]

    # ---------- Load Trainer dimension ----------
    try:
        if not trainer_file.exists():
            _log("ERROR", "❌ Normalized trainer model missing.")
            return
        df_trainers = pd.read_excel(trainer_file, sheet_name="Trainers", engine="openpyxl")
        if not {"TrainerID", "TrainerName"}.issubset(df_trainers.columns):
            _log("ERROR", "Trainer file missing required columns: TrainerID, TrainerName")
            return
        trainer_list = df_trainers["TrainerName"].dropna().astype(str).str.strip().tolist()
        _log("INFO", f"📄 Trainers loaded: {len(trainer_list)}")
    except Exception as e:
        _log("ERROR", f"Failed to load trainer model: {e}")
        return

    # ---------- Load / init Course + Map ----------
    try:
        if course_file.exists():
            _log("INFO", "Course model exists; loading...")
            df_courses = pd.read_excel(course_file, sheet_name="Courses", engine="openpyxl")
            df_map     = pd.read_excel(course_file, sheet_name="Trainer_Course_Map", engine="openpyxl")
            for col in expected_course_cols:
                if col not in df_courses.columns:
                    df_courses[col] = None
            df_courses = df_courses[expected_course_cols]
            if list(df_map.columns) != expected_map_cols:
                _log("WARN", "Trainer_Course_Map schema mismatch detected. Rebuilding map only.")
                df_map = pd.DataFrame(columns=expected_map_cols)
        else:
            _log("INFO", "Course model not found; creating fresh frames…")
            df_courses = pd.DataFrame(columns=expected_course_cols)
            df_map     = pd.DataFrame(columns=expected_map_cols)
    except Exception as e:
        _log("ERROR", f"Failed to load or create course model: {e}")
        return

    def _clean_course_master_name(value: str) -> str:
        text = str(value or "")
        text = re.sub(r"\((?:\s*total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*\d*\s*\)", "", text, flags=re.I)
        text = re.sub(r"\((?:current\s+)?qubits?.*?\)", "", text, flags=re.I)
        text = re.sub(r"\((?:min(?:imum)?\s+score|total\s+count\s+of\s+qubits?).*?\)", "", text, flags=re.I)
        text = re.sub(r"\s+", " ", text)
        return text.strip().strip("-–—").strip()

    def _norm_course(s: str) -> str:
        return normalize_text(_clean_course_master_name(s))

    def _dedupe_course_master(df_courses_in: pd.DataFrame, df_map_in: pd.DataFrame | None = None):
        if df_courses_in.empty or "CourseName" not in df_courses_in.columns:
            return df_courses_in, df_map_in
        df = df_courses_in.copy()
        df["CourseName"] = df["CourseName"].map(_clean_course_master_name)
        df["_CourseKey"] = df["CourseName"].map(_norm_course)
        df["_CourseID_Num"] = pd.to_numeric(df["CourseID"], errors="coerce")
        df = df.sort_values(["_CourseKey", "_CourseID_Num"], na_position="last").reset_index(drop=True)

        id_redirect = {}
        keep_indices = []
        for key, group in df.groupby("_CourseKey", dropna=False):
            if not key:
                keep_indices.extend(group.index.tolist())
                continue
            keep_idx = group.index[0]
            keep_id = df.loc[keep_idx, "CourseID"]
            keep_indices.append(keep_idx)
            for _, dup in group.iloc[1:].iterrows():
                id_redirect[dup["CourseID"]] = keep_id

        df = df.loc[sorted(set(keep_indices))].drop(columns=["_CourseKey", "_CourseID_Num"]).reset_index(drop=True)
        if df_map_in is not None and not df_map_in.empty and id_redirect:
            mapped = df_map_in.copy()
            mapped["CourseID"] = mapped["CourseID"].map(lambda x: id_redirect.get(x, x))
            return df, mapped
        return df, df_map_in

    # ---------- CourseID counter ----------
    try:
        if not df_courses.empty and "CourseID" in df_courses.columns:
            max_id = pd.to_numeric(df_courses["CourseID"], errors="coerce").max()
            course_id_counter = int(max_id) + 1 if pd.notna(max_id) else 1
        else:
            course_id_counter = 1
        _log("INFO", f"Starting CourseID at: {course_id_counter}")
    except Exception as e:
        _log("ERROR", f"Failed to compute CourseID counter: {e}")
        return

    TRAINER_COURSE_URL = "https://rms.koenig-solutions.com/Trainer/frmTrainerCourseGrade.aspx"

    # ---------- Iterate trainers ----------
    for trainer in trainer_list:
        try:
            tr_row = df_trainers[df_trainers["TrainerName"] == trainer]
            if tr_row.empty:
                _log("WARN", f"Trainer not found in df_trainers: {trainer}")
                continue

            trainer_id = int(tr_row["TrainerID"].iloc[0])
            _log("INFO", f"🔄 Processing TrainerID={trainer_id} | {trainer}")

            def _select_trainer_with_retry(trainer_name: str, attempts: int = 3) -> bool:
                first_token = trainer_name.split()[0]
                target_norm = normalize_text(trainer_name)
                last_error = None
                for attempt in range(1, attempts + 1):
                    try:
                        driver.get(TRAINER_COURSE_URL)
                        time.sleep(2)
                        inp = wait.until(EC.element_to_be_clickable((By.ID, "cphMainContent_mainContent_txtTrainer")))
                        inp.clear()
                        time.sleep(0.2)

                        for ch in first_token:
                            inp.send_keys(ch)
                            time.sleep(0.08)

                        time.sleep(1.5)
                        for _ in range(60):
                            inp = wait.until(EC.element_to_be_clickable((By.ID, "cphMainContent_mainContent_txtTrainer")))
                            inp.send_keys(Keys.ARROW_DOWN)
                            time.sleep(0.12)
                            current = (inp.get_attribute("value") or "").strip()
                            if normalize_text(current) == target_norm:
                                inp.send_keys(Keys.ENTER)
                                time.sleep(1.5)
                                return True
                    except Exception as exc:
                        last_error = exc
                        _log("WARN", f"Trainer selection retry {attempt}/{attempts} failed for {trainer_name}: {exc}")
                        time.sleep(1)
                if last_error:
                    _log("WARN", f"Trainer selection failed for {trainer_name} after {attempts} attempts: {last_error}")
                else:
                    _log("WARN", f"Could not match trainer from dropdown: {trainer_name}")
                return False

            if not _select_trainer_with_retry(trainer):
                continue

            # Locate table
            try:
                header = wait.until(
                    EC.presence_of_element_located((
                        By.XPATH,
                        "//tr[.//th[normalize-space()='Trainer Name' or contains(.,'Trainer')]]"
                        "[.//th[normalize-space()='Course Name' or contains(.,'Course')]]"
                    ))
                )
                table = header.find_element(By.XPATH, "./ancestor::table")
                rows  = table.find_elements(By.XPATH, ".//tr[td]")
            except Exception as e:
                _log("WARN", f"Failed to locate course table for {trainer}: {e}")
                continue

            # Iterate course rows
            for r in rows:
                try:
                    tds = r.find_elements(By.TAG_NAME, "td")
                    if len(tds) < 2:
                        continue

                    raw_course_text = (tds[1].text or "").strip()
                    if not raw_course_text:
                        continue

                    # Parse cell to extract fields
                    try:
                        parsed = parse_course_cell(raw_course_text)
                        if not parsed or "CourseName" not in parsed or not parsed["CourseName"]:
                            _log("DEBUG", f"parse_course_cell returned no CourseName | text={raw_course_text}")
                            continue
                    except Exception as e:
                        _log("WARN", f"parse_course_cell failed: {e} | text={raw_course_text}")
                        continue

                    course_name = _clean_course_master_name(parsed["CourseName"])
                    key = _norm_course(course_name)

                    # Lookup existing course by normalized key
                    course_id = None
                    if not df_courses.empty:
                        try:
                            mask = df_courses["CourseName"].map(_norm_course) == key
                            if mask.any():
                                course_id = int(df_courses.loc[mask, "CourseID"].iloc[0])
                        except Exception:
                            course_id = None

                    # Create course if new
                    if course_id is None:
                        _log("INFO", f"➕ New course detected: {course_name}")
                        try:
                            exists, exam_url = detect_exam_info(course_name)
                        except Exception:
                            exists, exam_url = False, None

                        try:
                            cert_info = detect_certification_info(course_name)
                            new_row = {
                                "CourseID": course_id_counter,
                                "CourseName": course_name,
                                "ExamURL": cert_info.get("url") if cert_info.get("exists") else None,
                                "CertificationVendor": cert_info.get("vendor"),
                                "CertificationName": cert_info.get("certification_name"),
                                "CertificationStatus": cert_info.get("status"),
                                "CertificationCheckedOn": datetime.now().strftime("%Y-%m-%d"),
                                "CertificationMatchReason": cert_info.get("reason"),
                            }
                            df_courses = pd.concat([df_courses, pd.DataFrame([new_row])], ignore_index=True)
                            course_id = course_id_counter
                            course_id_counter += 1
                        except Exception as e:
                            _log("ERROR", f"Failed to append new course row: {e}")
                            continue

                    # Upsert Trainer → Course relation
                    try:
                        if df_map.empty:
                            exists_rel = False
                        else:
                            exists_rel = bool(
                                df_map[
                                    (df_map["TrainerID"] == trainer_id) &
                                    (df_map["CourseID"] == course_id)
                                ].shape[0]
                            )

                        if not exists_rel:
                            rel_row = {
                                "TrainerID": trainer_id,
                                "CourseID": course_id,
                                "IsFutureLive": parsed.get("IsFutureLive"),
                                "AssignmentsDelivered": parsed.get("AssignmentsDelivered"),
                                "QubitScore": parsed.get("QubitScore"),
                                "MinScoreRequired": parsed.get("MinScoreRequired"),
                                "QubitQuestionCount": parsed.get("QubitQuestionCount"),
                            }
                            df_map = pd.concat([df_map, pd.DataFrame([rel_row])], ignore_index=True)
                    except Exception as e:
                        _log("WARN", f"Failed to upsert relation for TrainerID={trainer_id}, CourseID={course_id}: {e}")
                        continue

                except Exception as e_row:
                    _log("WARN", f"Row processing failed: {e_row}")
                    continue

        except Exception as e_trainer:
            _log("WARN", f"RMS instability — skipping trainer {trainer}: {e_trainer}")
            continue

    # ---------- Save (dedupe & persist) ----------
    try:
        if not df_courses.empty:
            df_courses, df_map = _dedupe_course_master(df_courses, df_map)
            df_courses = update_course_exam_urls_from_mslearn(df_courses)
            df_courses, df_map = _dedupe_course_master(df_courses, df_map)
            df_courses = (
                df_courses
                .sort_values(["CourseID", "CourseName"])
                .drop_duplicates(subset=["CourseID"], keep="first")
                .reset_index(drop=True)
            )
        if not df_map.empty:
            df_map = (
                df_map
                .drop_duplicates(subset=["TrainerID", "CourseID"], keep="last")
                .reset_index(drop=True)
            )

        with pd.ExcelWriter(course_file, engine="openpyxl") as writer:
            df_courses.to_excel(writer, sheet_name="Courses", index=False)
            df_map.to_excel(writer, sheet_name="Trainer_Course_Map", index=False)

        _log("INFO", "💾 Course model saved successfully.")
        print("\n🎯 Trainer → Course normalized mapping completed.")
        print(f"📈 Courses: {len(df_courses)}")
        print(f"📈 Trainer-Course Relations: {len(df_map)}")

    except Exception as e:
        _log("ERROR", f"Failed to save course model: {e}")
        return


In [ ]:
"""
Cell 7 — Parse a 'Course Name' cell into structured fields.
"""

from __future__ import annotations
import re
from typing import Optional, Dict, Any

_MONTHS_RE = re.compile(r"\b(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b", re.I)
_PAREN_RE  = re.compile(r"\(([^)]{0,200})\)")
_ASSIGN_RE = re.compile(r"(?:total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*(\d*)", re.I)
_QUBIT_SCORE_RE = re.compile(r"current\s+qubits?\s*score\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?", re.I)
_MIN_SCORE_RE   = re.compile(r"min(?:imum)?\s*score(?:\s*required)?\s*[:\-]?\s*(\d+(?:\.\d+)?)\s*%?", re.I)
_RATIO_RE       = re.compile(r"\b(\d{1,3})\s*/\s*(\d{1,3})\b")
_QCOUNT_RE      = re.compile(r"(?:total\s+count\s+of\s+qubits?\s+questions?|qubits?\s*questions?\s*count|qubit\s*question\s*count)\s*[:\-]?\s*(\d+)", re.I)
_QS_SUFFIX_RE   = re.compile(r"\b(\d+)\s*q(?:s|uestions?)\b", re.I)

def parse_course_cell(raw_text: str) -> Optional[Dict[str, Any]]:
    """
    Returns:
    {
      "CourseName": str,
      "IsFutureLive": bool,
      "AssignmentsDelivered": Optional[int],
      "QubitScore": Optional[int],
      "MinScoreRequired": Optional[int],
      "QubitQuestionCount": Optional[int],
    }
    """
    if not raw_text or not str(raw_text).strip():
        return None

    lines = [l.strip() for l in str(raw_text).splitlines() if str(l).strip()]
    if not lines:
        return None

    first_line = lines[0]
    rest_blob = " ".join(lines[1:]) if len(lines) > 1 else ""

    def _clean_course_name(value: str) -> str:
        text = str(value or "")
        text = re.sub(r"\((?:\s*total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*\d*\s*\)", "", text, flags=re.I)
        text = re.sub(r"\((?:current\s+)?qubits?.*?\)", "", text, flags=re.I)
        text = re.sub(r"\((?:min(?:imum)?\s+score|total\s+count\s+of\s+qubits?).*?\)", "", text, flags=re.I)
        text = re.sub(r"\s+", " ", text)
        return text.strip().strip("-??").strip()

    is_future_live = False
    assignments_delivered = None

    # Examine parentheses on the first line
    to_strip_chunks = []
    for chunk in _PAREN_RE.findall(first_line):
        chunk_l = chunk.strip().lower()
        if _MONTHS_RE.search(chunk_l) or "upcoming" in chunk_l or "future" in chunk_l:
            is_future_live = True
            to_strip_chunks.append(f"({chunk})")
        m_asg = _ASSIGN_RE.search(chunk)
        if m_asg:
            try:
                assignments_delivered = int(m_asg.group(1))
            except Exception:
                assignments_delivered = None
            to_strip_chunks.append(f"({chunk})")

    # Remove detected chunks from course name if auxiliary
    course_name = first_line
    for seg in to_strip_chunks:
        course_name = course_name.replace(seg, "")
    course_name = course_name.strip().strip("-–—").strip()

    # If no assignments in parentheses, scan the rest
    if assignments_delivered is None:
        m_asg2 = _ASSIGN_RE.search(rest_blob)
        if m_asg2:
            try:
                assignments_delivered = int(m_asg2.group(1))
            except Exception:
                assignments_delivered = None

    # Qubit metrics
    text_all = " ".join(lines)
    qubit_score = None
    min_score_required = None
    qubit_question_count = None

    m_qs = _QUBIT_SCORE_RE.search(text_all)
    if m_qs:
        try:
            qubit_score = int(round(float(m_qs.group(1))))
        except Exception:
            qubit_score = None

    if qubit_score is None:
        m_ratio = _RATIO_RE.search(text_all)
        if m_ratio:
            try:
                num = float(m_ratio.group(1)); den = float(m_ratio.group(2))
                if den > 0:
                    qubit_score = int(round((num / den) * 100))
            except Exception:
                qubit_score = None

    m_min = _MIN_SCORE_RE.search(text_all)
    if m_min:
        try:
            min_score_required = int(round(float(m_min.group(1))))
        except Exception:
            min_score_required = None

    m_qc = _QCOUNT_RE.search(text_all)
    if m_qc:
        try:
            qubit_question_count = int(m_qc.group(1))
        except Exception:
            qubit_question_count = None
    else:
        m_qsfx = _QS_SUFFIX_RE.search(text_all)
        if m_qsfx:
            try:
                qubit_question_count = int(m_qsfx.group(1))
            except Exception:
                qubit_question_count = None

    return {
        "CourseName": course_name,
        "IsFutureLive": is_future_live,
        "AssignmentsDelivered": assignments_delivered,
        "QubitScore": qubit_score,
        "MinScoreRequired": min_score_required,
        "QubitQuestionCount": qubit_question_count,
    }


In [ ]:
"""
Cell 9 - Normalize Trainer -> Certification mapping from RMS Certification page.

This version is resilient to RMS header changes:
  - detects headers from th or td rows
  - only requires Trainer plus at least one Course/Exam column
  - treats Courseware/Exam as optional
  - returns True/False so the menu reports partial failures honestly
"""

from __future__ import annotations
from datetime import datetime

def extract_trainer_certification_results(
    driver,
    wait,
    *,
    fuzzy_threshold_trainer: int = 90,
    fuzzy_threshold_course: int = 90,
    allow_fuzzy_trainer: bool = True,
    allow_fuzzy_course: bool = True,
):
    """
    Produces: RMS_Normalized_Certification_Model.xlsx / Trainer_Certification_Map
    Columns:
      TrainerID | CourseID | ExamName | ExamCode | Result | ApprovalStatus | Vendor | ExamURL
    """
    trainer_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
    course_file  = MY_REPORTEES_FOLDER / "RMS_Normalized_Course_Model.xlsx"
    cert_file    = MY_REPORTEES_FOLDER / "RMS_Normalized_Certification_Model.xlsx"

    if not trainer_file.exists() or not course_file.exists():
        print("[ERROR] Required models missing. Build Trainer and Course models first.")
        return False

    df_trainers = pd.read_excel(trainer_file, sheet_name="Trainers", engine="openpyxl")
    df_courses  = pd.read_excel(course_file,  sheet_name="Courses",  engine="openpyxl")

    if not {"TrainerID", "TrainerName"}.issubset(df_trainers.columns):
        print("[ERROR] Trainer model invalid (needs TrainerID, TrainerName).")
        return False
    if not {"CourseID", "CourseName", "ExamURL"}.issubset(df_courses.columns):
        print("[ERROR] Course model invalid (needs CourseID, CourseName, ExamURL).")
        return False

    df_trainers = df_trainers.copy()
    df_courses  = df_courses.copy()
    df_trainers["TrainerName_N"] = df_trainers["TrainerName"].map(normalize_text)
    df_courses["CourseName_N"]   = df_courses["CourseName"].map(normalize_text)

    columns = ["TrainerID", "CourseID", "ExamName", "ExamCode", "Result", "ApprovalStatus", "Vendor", "ExamURL"]
    if cert_file.exists():
        try:
            df_map = pd.read_excel(cert_file, sheet_name="Trainer_Certification_Map", engine="openpyxl")
            for col_name in columns:
                if col_name not in df_map.columns:
                    df_map[col_name] = None
            df_map = df_map[columns]
        except Exception:
            df_map = pd.DataFrame(columns=columns)
    else:
        df_map = pd.DataFrame(columns=columns)

    GRID_ID = "cphMainContent_mainContent_GridView1"
    driver.get("https://rms.koenig-solutions.com/Trainer/frmCertification.aspx")
    try:
        wait.until(EC.presence_of_element_located((By.ID, GRID_ID)))
    except TimeoutException:
        print("[ERROR] Certification grid did not load.")
        return False

    table = driver.find_element(By.ID, GRID_ID)

    def _cell_text(cell) -> str:
        return (cell.text or "").replace("\n", " ").strip()

    def _find_header_cells():
        header_cells = table.find_elements(By.XPATH, ".//thead//th")
        if header_cells:
            return header_cells
        header_cells = table.find_elements(By.XPATH, ".//tr[th][1]/th")
        if header_cells:
            return header_cells
        # Some ASP.NET grids render the header as td cells in the first row.
        first_row_cells = table.find_elements(By.XPATH, ".//tr[1]/*")
        if first_row_cells:
            return first_row_cells
        return []

    def _find_col_indices():
        header_cells = _find_header_cells()
        if not header_cells:
            raise RuntimeError("No table header cells found")

        labels = [normalize_text(_cell_text(cell)) for cell in header_cells]
        print("Detected certification grid headers:")
        print(" - " + " | ".join([label or "(blank)" for label in labels]))

        colmap = {
            "trainer": None,
            "exam_name": None,
            "course_name": None,
            "courseware_or_exam": None,
            "result": None,
            "vendor": None,
            "approval": None,
        }

        def has(label: str, *terms: str) -> bool:
            return any(term in label for term in terms)

        for idx, label in enumerate(labels):
            compact = label.replace(" ", "").replace("/", "")
            if colmap["trainer"] is None and has(label, "trainer", "faculty", "instructor"):
                colmap["trainer"] = idx
            elif colmap["courseware_or_exam"] is None and (
                "courseware" in label
                or "coursewareorexam" in compact
                or compact in {"examcourseware", "type", "itemtype"}
            ):
                colmap["courseware_or_exam"] = idx
            elif colmap["course_name"] is None and (
                "course name" in label
                or "coursename" in compact
                or compact in {"course", "training", "courseexam", "examcourse"}
            ):
                colmap["course_name"] = idx
            elif colmap["exam_name"] is None and (
                "exam name" in label
                or "certification" in label
                or "exam title" in label
                or compact in {"exam", "examname", "examtitle", "certificationname"}
            ):
                colmap["exam_name"] = idx
            elif colmap["result"] is None and has(label, "result", "pass", "fail", "status"):
                colmap["result"] = idx
            elif colmap["vendor"] is None and has(label, "vendor", "provider", "oem"):
                colmap["vendor"] = idx
            elif colmap["approval"] is None and has(label, "approval", "approved", "status", "remarks"):
                colmap["approval"] = idx

        # If RMS gives one generic Course/Exam column, use it for both names.
        if colmap["exam_name"] is None and colmap["course_name"] is not None:
            colmap["exam_name"] = colmap["course_name"]
        if colmap["course_name"] is None and colmap["exam_name"] is not None:
            colmap["course_name"] = colmap["exam_name"]

        required_missing = []
        if colmap["trainer"] is None:
            required_missing.append("trainer")
        if colmap["exam_name"] is None and colmap["course_name"] is None:
            required_missing.append("exam_or_course_name")
        if required_missing:
            raise RuntimeError(f"Required columns not found in header: {required_missing}")
        return colmap

    try:
        col = _find_col_indices()
    except Exception as e:
        print(f"[ERROR] Column detection failed: {e}")
        return False

    rows = table.find_elements(By.XPATH, ".//tr[td]")
    added = 0
    updated = 0
    skipped = 0

    EXAM_CODE_RE = re.compile(r"\b([A-Z]{1,8})[- ]?(\d{2,5})\b")

    def _extract_exam_code(*texts) -> str | None:
        for text in texts:
            if not text:
                continue
            match = EXAM_CODE_RE.search(str(text).upper())
            if match:
                return f"{match.group(1)}-{match.group(2)}"
        return None

    def _td_value(tds, idx) -> str:
        if idx is None or idx >= len(tds):
            return ""
        return (tds[idx].text or "").strip()

    for row in rows:
        try:
            tds = row.find_elements(By.TAG_NAME, "td")
            if not tds:
                continue

            max_needed = max([v for v in col.values() if v is not None] or [0])
            if len(tds) <= max_needed:
                skipped += 1
                continue

            # If RMS exposes Courseware/Exam, keep Exam rows. If it does not, do not block extraction.
            courseware_exam = _td_value(tds, col["courseware_or_exam"])
            if col["courseware_or_exam"] is not None and normalize_text(courseware_exam) not in {"exam", "certification"}:
                skipped += 1
                continue

            trainer_name = (_td_value(tds, col["trainer"]) or "").split("\n")[0].strip()
            exam_name    = _td_value(tds, col["exam_name"])
            course_name  = _td_value(tds, col["course_name"])
            if not exam_name and course_name:
                exam_name = course_name
            if not course_name and exam_name:
                course_name = exam_name
            vendor       = _td_value(tds, col["vendor"]) if col["vendor"] is not None else None

            approval = None
            try:
                approval_el = row.find_element(By.XPATH, ".//span[contains(@id,'lnkStatus') or contains(@id,'lblStatus')]")
                approval = approval_el.text.strip()
            except Exception:
                approval = _td_value(tds, col["approval"]) if col["approval"] is not None else ""

            result = "Pending"
            if col["result"] is not None:
                try:
                    result_btn = tds[col["result"]].find_element(By.XPATH, ".//input[contains(@id,'btnResult')]")
                    raw_result = (result_btn.get_attribute("value") or "").strip().lower()
                    if raw_result == "pass":
                        result = "Pass"
                    elif raw_result == "fail":
                        result = "Fail"
                except Exception:
                    result_text = (_td_value(tds, col["result"]) or "").lower()
                    if "pass" in result_text:
                        result = "Pass"
                    elif "fail" in result_text:
                        result = "Fail"
                    elif result_text:
                        result = result_text.title()

            tn = normalize_text(trainer_name)
            trow = df_trainers[df_trainers["TrainerName_N"] == tn]
            if trow.empty and allow_fuzzy_trainer and tn:
                best = None
                best_score = -1
                for _, trainer_row in df_trainers.iterrows():
                    score = fuzzy_score(tn, trainer_row["TrainerName_N"])
                    if score > best_score:
                        best_score, best = score, trainer_row
                if best is not None and best_score >= fuzzy_threshold_trainer:
                    trow = pd.DataFrame([best])
            if trow.empty:
                skipped += 1
                continue
            trainer_id = int(trow["TrainerID"].iloc[0])

            cn = normalize_text(course_name)
            crow = df_courses[df_courses["CourseName_N"] == cn] if cn else pd.DataFrame()

            exam_code = _extract_exam_code(exam_name, course_name)
            if crow.empty and exam_code:
                mask = df_courses["ExamURL"].astype(str).str.contains(exam_code, case=False, na=False)
                crow = df_courses[mask]
                if crow.empty and "CourseName" in df_courses.columns:
                    mask = df_courses["CourseName"].astype(str).str.contains(exam_code, case=False, na=False)
                    crow = df_courses[mask]

            if crow.empty and allow_fuzzy_course and cn:
                best = None
                best_score = -1
                for _, course_row in df_courses.iterrows():
                    score = fuzzy_score(cn, course_row["CourseName_N"])
                    if score > best_score:
                        best_score, best = score, course_row
                if best is not None and best_score >= fuzzy_threshold_course:
                    crow = pd.DataFrame([best])

            if crow.empty:
                skipped += 1
                continue

            course_id = int(crow["CourseID"].iloc[0])
            exam_url  = crow["ExamURL"].iloc[0] if "ExamURL" in crow.columns else None

            key_mask = (df_map["TrainerID"] == trainer_id) & (df_map["CourseID"] == course_id)
            if key_mask.any():
                df_map.loc[key_mask, ["Result", "ApprovalStatus", "ExamName", "ExamCode", "Vendor", "ExamURL"]] = [
                    result, approval, exam_name, exam_code, vendor, exam_url
                ]
                updated += 1
            else:
                df_map.loc[len(df_map)] = [
                    trainer_id, course_id, exam_name, exam_code, result, approval, vendor, exam_url
                ]
                added += 1

        except StaleElementReferenceException:
            skipped += 1
            continue
        except Exception:
            skipped += 1
            continue

    try:
        if not df_map.empty:
            df_map = (
                df_map
                .drop_duplicates(subset=["TrainerID", "CourseID"], keep="last")
                .reset_index(drop=True)
            )
        with pd.ExcelWriter(cert_file, engine="openpyxl") as writer:
            df_map.to_excel(writer, sheet_name="Trainer_Certification_Map", index=False)
    except Exception as e:
        print(f"[ERROR] Failed to save certification model: {e}")
        return False

    print("\nCertification mapping completed.")
    print(f"Trainer-Certification rows: {len(df_map)} (+{added} new, ~{updated} updated, {skipped} skipped)")
    return True


In [ ]:
# =========================================================
# Certification Assignments Dashboard — MS Exams only (ExamURL present)
# Table columns (right): TrainerName | ExamName | CourseName | MSLearnLink | Result
#   • No selection  -> ALL trainers
#   • On selection  -> filtered to that trainer + header chips (Assigned, Cleared, Pending)
# Search works via typing, Enter, and 'Filter' button.
# =========================================================

from __future__ import annotations
from typing import List, Optional
from datetime import datetime
import pandas as pd
try:
    from IPython.display import display, HTML
except Exception:
    def display(x):
        print(x)
    class HTML(str):
        pass

# ---- safety fallbacks (if Cell 3 wasn't executed) ----
try:
    normalize_text  # type: ignore[name-defined]
except NameError:
    import re, unicodedata
    _NON_ALPHA_SPACE = re.compile(r"[^a-z\s]+")
    def _fold(s: str) -> str:
        nfkd = unicodedata.normalize("NFKD", s)
        return nfkd.encode("ascii", "ignore").decode("ascii")
    def normalize_text(s: str|None) -> str:  # type: ignore[no-redef]
        if not s: return ""
        s2 = _fold(s).lower().replace("\r"," ").replace("\n"," ").replace("\t"," ")
        s2 = _NON_ALPHA_SPACE.sub(" ", s2)
        return " ".join(s2.split())

# ---------- load & shape ----------
def _load_assignment_view_ms_only() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build the assignment view restricted to mapped Microsoft Learn courses only.

    A course is included only when:
      1) Courses.ExamURL contains a valid learn.microsoft.com URL, OR
      2) CourseName contains a recognizable Microsoft exam code and we can map it
         to a Microsoft Learn exam URL.

    Returns:
      assignment_view: TrainerName | ExamName | CourseName | MSLearnLink | Result
      per_trainer    : Trainer | Assigned | Cleared | Pending
    """
    trainer_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
    course_file  = MY_REPORTEES_FOLDER / "RMS_Normalized_Course_Model.xlsx"
    cert_file    = MY_REPORTEES_FOLDER / "RMS_Normalized_Certification_Model.xlsx"

    empty_view = pd.DataFrame(columns=["TrainerName", "ExamName", "CourseName", "MSLearnLink", "Result"])
    empty_sum  = pd.DataFrame(columns=["Trainer", "Assigned", "Cleared", "Pending"])

    if not trainer_file.exists():
        raise FileNotFoundError("Trainer model not found.")
    trainers = pd.read_excel(trainer_file, sheet_name="Trainers", engine="openpyxl")[["TrainerID", "TrainerName"]]

    if not course_file.exists():
        return empty_view, empty_sum

    courses_all = pd.read_excel(course_file, sheet_name="Courses", engine="openpyxl")
    tcm = pd.read_excel(course_file, sheet_name="Trainer_Course_Map", engine="openpyxl")[["TrainerID", "CourseID"]]

    if "ExamURL" not in courses_all.columns:
        courses_all["ExamURL"] = ""

    def _valid_ms_learn_url(value) -> str:
        url = str(value or "").strip()
        if not url or url.lower() in {"nan", "none", "null"}:
            return ""
        low = url.lower()
        if low.startswith("http") and "learn.microsoft.com" in low:
            return url
        return ""

    def _extract_ms_exam_code(*values) -> str:
        text = " ".join(str(v or "") for v in values).upper()
        # Handles DP-700T00-A, DP-700 T00, PL-300T00, AZ-104, AI-102, SC-200, etc.
        m = re.search(r"\b(AZ|AI|DP|PL|SC|MS|MD|MB)[-\s]?(\d{3,4})\b", text)
        if not m:
            return ""
        return f"{m.group(1)}-{m.group(2)}"

    def _mapped_ms_learn_url(row) -> str:
        existing = _valid_ms_learn_url(row.get("ExamURL", ""))
        if existing:
            return existing
        code = _extract_ms_exam_code(row.get("CourseName", ""), row.get("ExamURL", ""))
        if not code:
            return ""
        return f"https://learn.microsoft.com/en-us/credentials/certifications/exams/{code.lower()}"

    courses_all = courses_all.copy()
    courses_all["MSLearnLink"] = courses_all.apply(_mapped_ms_learn_url, axis=1)
    courses_all["MSExamCode"] = courses_all.apply(lambda r: _extract_ms_exam_code(r.get("CourseName", ""), r.get("ExamURL", "")), axis=1)

    # Keep only Excel courses that can be tied to Microsoft Learn.
    courses_ms = courses_all[
        courses_all["MSLearnLink"].astype(str).str.contains("learn.microsoft.com", case=False, na=False)
    ][["CourseID", "CourseName", "MSLearnLink", "MSExamCode"]].drop_duplicates()

    if courses_ms.empty:
        print("No courses with valid Microsoft Learn links were found or mapped from the Course model.")
        return empty_view, empty_sum

    assigned = (
        tcm.drop_duplicates()
           .merge(courses_ms, on="CourseID", how="inner")
           .merge(trainers, on="TrainerID", how="left")
           .fillna({"TrainerName": "", "CourseName": "", "MSLearnLink": "", "MSExamCode": ""})
    )

    if cert_file.exists():
        cert_xl = pd.ExcelFile(cert_file, engine="openpyxl")
        sheet = "Trainer_Certification_Model" if "Trainer_Certification_Model" in cert_xl.sheet_names else "Trainer_Certification_Map"
        cert = pd.read_excel(cert_file, sheet_name=sheet, engine="openpyxl")
    else:
        cert = pd.DataFrame(columns=["TrainerID", "CourseID", "ExamName", "ExamCode", "Result", "ApprovalStatus", "ExamURL"])

    for col in ["TrainerID", "CourseID", "ExamName", "ExamCode", "Result", "ApprovalStatus", "ExamURL"]:
        if col not in cert.columns:
            cert[col] = ""

    cert = cert.copy()
    cert["MSLearnLink"] = cert["ExamURL"].map(_valid_ms_learn_url)
    cert["CertExamCode"] = cert.apply(lambda r: _extract_ms_exam_code(r.get("ExamCode", ""), r.get("ExamName", ""), r.get("ExamURL", "")), axis=1)
    cert["_status_text"] = (cert["Result"].astype(str) + " " + cert["ApprovalStatus"].astype(str)).str.lower()
    cert["_cleared"] = cert["_status_text"].str.contains("pass|passed|clear|cleared|approved|success", regex=True, na=False)

    cert_pass = cert[cert["_cleared"]][["TrainerID", "CourseID", "ExamName", "CertExamCode"]].copy()
    cert_pass = cert_pass.dropna(subset=["TrainerID"]).drop_duplicates(subset=["TrainerID", "CourseID", "CertExamCode"], keep="last")

    by_course = assigned.merge(
        cert_pass[["TrainerID", "CourseID", "ExamName"]],
        on=["TrainerID", "CourseID"],
        how="left",
    )

    # If CourseID mapping is missing in cert file, fall back to TrainerID + exam code.
    cert_by_code = cert_pass[cert_pass["CertExamCode"].astype(str).str.strip().ne("")][["TrainerID", "CertExamCode", "ExamName"]]
    view = by_course.merge(
        cert_by_code,
        left_on=["TrainerID", "MSExamCode"],
        right_on=["TrainerID", "CertExamCode"],
        how="left",
        suffixes=("", "_ByCode"),
    )

    view["ExamName"] = view["ExamName"].fillna(view["ExamName_ByCode"]).fillna(view["MSExamCode"])
    view["Result"] = view.apply(
        lambda r: "Pass" if str(r.get("ExamName", "")).strip() and (
            pd.notna(r.get("ExamName")) and (pd.notna(r.get("CourseID")) or str(r.get("CertExamCode", "")).strip())
        ) else "Pending",
        axis=1,
    )

    view = view[["TrainerName", "ExamName", "CourseName", "MSLearnLink", "Result"]].copy()
    view = view.sort_values(["TrainerName", "Result", "CourseName"]).reset_index(drop=True)

    if view.empty:
        return empty_view, empty_sum

    g = view.groupby("TrainerName")["Result"].value_counts().unstack(fill_value=0).reset_index()
    for c in ["Pass", "Pending"]:
        if c not in g.columns:
            g[c] = 0
    g["Assigned"] = g["Pass"] + g["Pending"]
    per_trainer = g.rename(columns={"TrainerName": "Trainer", "Pass": "Cleared"})[["Trainer", "Assigned", "Cleared", "Pending"]]
    per_trainer = per_trainer.sort_values(["Cleared", "Assigned"], ascending=[False, False]).reset_index(drop=True)

    return view, per_trainer

def _chips_header(trainer: str, df_sel: pd.DataFrame) -> HTML:
    assigned = len(df_sel)
    cleared  = int((df_sel["Result"] == "Pass").sum())
    pending  = int((df_sel["Result"] == "Pending").sum())
    return HTML(
        f"""
        <div style="display:flex;align-items:center;gap:12px;margin:6px 0 10px 0;">
          <div style="font-weight:700;font-size:18px">{trainer}</div>
          <span style="background:#605e5c;color:#fff;padding:2px 8px;border-radius:10px;font-size:12px;">Assigned {assigned}</span>
          <span style="background:#107c10;color:#fff;padding:2px 8px;border-radius:10px;font-size:12px;">Cleared {cleared}</span>
          <span style="background:#d13438;color:#fff;padding:2px 8px;border-radius:10px;font-size:12px;">Pending {pending}</span>
        </div>
        """
    )

# ---------- MAIN ----------
def certification_compliance_dashboard(export: bool = False, use_widgets: bool = True):
    """
    MS Exam Assignments Dashboard:
      • Only courses where ExamURL is present (MS exam courses)
      • Right table ALWAYS: TrainerName | ExamName | CourseName | MSLearnLink | Result
         - No selection -> ALL trainers
         - Selection    -> filtered to that trainer + header chips
      • Top: Search + Filter + Clear
    """
    assignment_view, per_trainer = _load_assignment_view_ms_only()

    # Console fallback
    if use_widgets:
        try:
            import ipywidgets as widgets
        except Exception:
            use_widgets = False

    if not use_widgets:
        print("MS EXAM ASSIGNMENTS (Console)".center(96, "="))
        print("\nSUMMARY:")
        print(per_trainer.to_string(index=False))
        print("\nDETAILS (ALL):")
        cols = ["TrainerName", "ExamName", "CourseName", "MSLearnLink", "Result"]
        print(assignment_view[cols].sort_values(["TrainerName","Result","CourseName"]).to_string(index=False))
        if export:
            ts = datetime.now().strftime("%Y%m%d_%H%M")
            per_trainer.to_csv(MY_REPORTEES_FOLDER / f"MSExam_Summary_{ts}.csv", index=False)
            assignment_view.to_csv(MY_REPORTEES_FOLDER / f"MSExam_Details_{ts}.csv", index=False)
            print(f"\nExports: MSExam_Summary_{ts}.csv, MSExam_Details_{ts}.csv")
        return {"summary": per_trainer, "details": assignment_view}

    # ---------------- widgets UI ----------------
    import ipywidgets as widgets

    # Controls
    search     = widgets.Text(value="", placeholder="Search trainers…", description="", layout=widgets.Layout(width="60%"))
    btn_filter = widgets.Button(description="Filter", button_style="primary", icon="search", layout=widgets.Layout(width="120px"))
    btn_clear  = widgets.Button(description="Clear",  button_style="",        icon="times",  layout=widgets.Layout(width="100px"))

    # Panes
    out_left  = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", min_height="240px"))
    out_right = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="6px", min_height="300px"))

    # List (no label or padding)
    all_trainers = ["All Trainers"] + sorted(assignment_view["TrainerName"].dropna().unique().tolist(), key=lambda s: normalize_text(s))
    trainer_select = widgets.Select(
        options=all_trainers,
        value="All Trainers",
        description="",
        rows=12,
        layout=widgets.Layout(width="100%", height="240px"),
        style={"description_width": "0px"}
    )

    # Filtering
    def _apply_search(df: pd.DataFrame) -> pd.DataFrame:
        q = normalize_text(search.value)
        if not q: return df
        return df[df["TrainerName"].map(normalize_text).str.contains(q, na=False)]

    def _trainer_candidates() -> List[str]:
        df = _apply_search(assignment_view)
        return ["All Trainers"] + sorted(df["TrainerName"].dropna().unique().tolist(), key=lambda s: normalize_text(s))

    # Painters
    def paint_left():
        with out_left:
            out_left.clear_output(wait=True)
            df = _apply_search(assignment_view)
            # Recompute summary on filtered set
            g = df.groupby("TrainerName")["Result"].value_counts().unstack(fill_value=0).reset_index()
            for c in ["Pass","Pending"]:
                if c not in g.columns: g[c] = 0
            g["Assigned"] = g["Pass"] + g["Pending"]
            s = g.rename(columns={"TrainerName":"Trainer","Pass":"Cleared"})[["Trainer","Assigned","Cleared","Pending"]]
            s = s.sort_values(["Cleared","Assigned"], ascending=[False, False]).reset_index(drop=True)
            display(s)

    def paint_right():
        with out_right:
            out_right.clear_output(wait=True)
            df = _apply_search(assignment_view)
            sel = trainer_select.value
            cols = ["TrainerName", "ExamName", "CourseName", "MSLearnLink", "Result"]

            if not sel or sel == "All Trainers":
                if df.empty:
                    display(HTML("<i>No data to display.</i>"))
                    return
                display(df[cols].sort_values(["TrainerName","Result","CourseName"]).reset_index(drop=True))
            else:
                df_sel = df[df["TrainerName"] == sel]
                display(_chips_header(sel, df_sel))
                display(df_sel[cols].reset_index(drop=True))

    # Triggers (robust: typing, Enter, and button)
    def _refresh(_=None):
        candidates = _trainer_candidates()
        trainer_select.options = candidates
        if trainer_select.value not in candidates:
            trainer_select.value = "All Trainers"
        paint_left()
        paint_right()

    search.observe(_refresh, names="value")
    search.on_submit(lambda _: _refresh())
    btn_filter.on_click(lambda _: _refresh())

    def _clear(_btn):
        search.value = ""
        trainer_select.value = None
        _refresh()
    btn_clear.on_click(_clear)

    trainer_select.observe(lambda ch: paint_right() if ch["name"] == "value" else None, names="value")

    # Layout
    top_row = widgets.HBox([search, btn_filter, btn_clear], layout=widgets.Layout(gap="8px", align_items="center"))
    left_panel  = widgets.VBox([widgets.HTML("<b>Trainers</b>"), trainer_select, out_left], layout=widgets.Layout(width="35%"))
    right_panel = widgets.VBox([widgets.HTML("<b>Details</b>"), out_right], layout=widgets.Layout(width="65%"))
    panels = widgets.HBox([left_panel, right_panel], layout=widgets.Layout(gap="10px"))

    # Initial render
    _refresh()
    display(top_row, panels)

    return {
        "summary": per_trainer,
        "details": assignment_view,
        "widgets": {
            "search": search,
            "btn_filter": btn_filter,
            "btn_clear": btn_clear,
            "trainer_select": trainer_select,
            "out_left": out_left,
            "out_right": out_right,
        },
    }

In [ ]:
"""
Cell 11A - Trainer Intelligence: current skills, exam status, and next-course planning.

Uses the normalized RMS models already created by this notebook:
  - RMS_Normalized_Trainer_Model.xlsx
  - RMS_Normalized_Course_Model.xlsx
  - RMS_Normalized_Certification_Model.xlsx

Menu option "Trainer Intelligence" calls trainer_intelligence_console().
"""

from __future__ import annotations
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Iterable, List, Tuple
import re
import math
import numpy as np
import pandas as pd
try:
    from IPython.display import display, Markdown
except Exception:
    def display(x):
        print(x)
    class Markdown(str):
        pass

TRAINER_INTELLIGENCE_WEIGHTS = {
    "skill": 0.30,
    "cert_cleared": 0.25,
    "cert_pending": 0.08,
    "demand": 0.22,
    "experience": 0.10,
    "strategic": 0.05,
}

COURSE_STATUS_NOTES = {
    "AI-102": "AI-102 / Azure AI Engineer retires on 2026-06-30. Use only for near-term demand; move trainer toward AI agents and generative AI.",
    "AI-900": "AI-900 / Azure AI Fundamentals retires on 2026-06-30. Use only for short-term fundamentals demand.",
}

# Editable market-prior catalog. RMS popular-course data can be added later as a demand override.
TRAINER_INTELLIGENCE_CATALOG = pd.DataFrame([
    {"CourseCode":"DP-700", "CourseName":"Microsoft Fabric Data Engineer", "Category":"Data & Fabric", "DemandScore":0.96, "StrategicBoost":0.96, "RequiredCerts":["DP-700", "DP-600"], "Skills":["Microsoft Fabric", "Lakehouse", "Warehouse", "Data Factory", "Pipelines", "PySpark", "SQL", "KQL", "Medallion Architecture"]},
    {"CourseCode":"DP-600", "CourseName":"Fabric Analytics Engineer Associate", "Category":"Data & Fabric", "DemandScore":0.94, "StrategicBoost":0.93, "RequiredCerts":["DP-600"], "Skills":["Microsoft Fabric", "Power BI", "Semantic Models", "DAX", "Data Modeling", "Lakehouse", "SQL", "KQL"]},
    {"CourseCode":"PL-300", "CourseName":"Power BI Data Analyst", "Category":"Data & Fabric", "DemandScore":0.88, "StrategicBoost":0.78, "RequiredCerts":["PL-300"], "Skills":["Power BI", "DAX", "Visualization", "Power Query", "Data Modeling"]},
    {"CourseCode":"DP-203", "CourseName":"Azure Data Engineer", "Category":"Data Engineering", "DemandScore":0.82, "StrategicBoost":0.70, "RequiredCerts":["DP-203"], "Skills":["ADF", "ETL", "Azure Synapse", "Data Lake", "Databricks", "SQL", "Data Engineering"]},
    {"CourseCode":"AI-3025", "CourseName":"Work with AI agents", "Category":"Agentic AI", "DemandScore":0.95, "StrategicBoost":0.99, "RequiredCerts":[], "Skills":["AI Agents", "Azure AI Foundry", "Prompt Engineering", "Tool Calling", "RAG", "Evaluation", "Responsible AI"]},
    {"CourseCode":"AI-3016", "CourseName":"Develop generative AI apps in Azure", "Category":"Generative AI", "DemandScore":0.93, "StrategicBoost":0.97, "RequiredCerts":[], "Skills":["Azure OpenAI", "Prompt Engineering", "RAG", "Vector Search", "Python", "Responsible AI"]},
    {"CourseCode":"GH-300", "CourseName":"GitHub Copilot for Developers", "Category":"Developer Productivity", "DemandScore":0.90, "StrategicBoost":0.91, "RequiredCerts":[], "Skills":["GitHub Copilot", "Prompt Engineering", "Code Review", "Developer Productivity", "GitHub"]},
    {"CourseCode":"AZ-104", "CourseName":"Azure Administrator", "Category":"Cloud", "DemandScore":0.87, "StrategicBoost":0.76, "RequiredCerts":["AZ-104"], "Skills":["Azure", "Virtual Machines", "Networking", "Identity", "Storage", "Monitoring"]},
    {"CourseCode":"AZ-305", "CourseName":"Azure Solutions Architect", "Category":"Cloud", "DemandScore":0.84, "StrategicBoost":0.76, "RequiredCerts":["AZ-305"], "Skills":["Azure Architecture", "Security", "Networking", "Governance", "Data Platform"]},
    {"CourseCode":"SC-200", "CourseName":"Security Operations Analyst", "Category":"Security", "DemandScore":0.82, "StrategicBoost":0.80, "RequiredCerts":["SC-200"], "Skills":["Microsoft Sentinel", "Defender", "KQL", "Security Operations", "Incident Response"]},
    {"CourseCode":"SC-900", "CourseName":"Security Compliance Identity Fundamentals", "Category":"Security", "DemandScore":0.80, "StrategicBoost":0.70, "RequiredCerts":["SC-900"], "Skills":["Security", "Compliance", "Identity", "Microsoft Entra", "Governance"]},
    {"CourseCode":"AI-900", "CourseName":"Azure AI Fundamentals", "Category":"AI", "DemandScore":0.84, "StrategicBoost":0.55, "RequiredCerts":["AI-900"], "Skills":["Azure AI", "Generative AI", "Responsible AI", "Computer Vision", "NLP"]},
    {"CourseCode":"AI-102", "CourseName":"Azure AI Engineer", "Category":"AI", "DemandScore":0.76, "StrategicBoost":0.42, "RequiredCerts":["AI-102"], "Skills":["Azure AI Services", "Search", "APIs", "NLP", "Computer Vision", "Responsible AI"]},
])
TRAINER_INTELLIGENCE_CATALOG["StatusNote"] = TRAINER_INTELLIGENCE_CATALOG["CourseCode"].map(COURSE_STATUS_NOTES).fillna("")


def _ti_model_paths() -> Dict[str, Path]:
    try:
        base = MY_REPORTEES_FOLDER
    except NameError:
        base = Path(".")
    return {
        "trainer": Path(base) / "RMS_Normalized_Trainer_Model.xlsx",
        "course": Path(base) / "RMS_Normalized_Course_Model.xlsx",
        "cert": Path(base) / "RMS_Normalized_Certification_Model.xlsx",
    }


def _ti_norm(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).lower()
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    text = re.sub(r"[^a-z0-9+#.\s-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _ti_course_code(value: Any) -> str:
    text = str(value or "").upper()
    match = re.search(r"\b([A-Z]{1,5}-\d{2,4})\b", text)
    return match.group(1) if match else _ti_norm(value).upper()


def _ti_is_cleared(*values: Any) -> bool:
    text = " ".join(_ti_norm(v) for v in values)
    return any(word in text for word in ["pass", "passed", "clear", "cleared", "approved", "success"])


def _ti_status(result: Any, approval: Any = "") -> str:
    if _ti_is_cleared(result, approval):
        return "Cleared"
    text = " ".join([_ti_norm(result), _ti_norm(approval)])
    if any(word in text for word in ["pending", "assigned", "scheduled", "progress", "not attempted", "fail", "failed"]):
        return "Pending"
    return "Assigned"


def _ti_read_excel(path: Path, sheet_name: str) -> pd.DataFrame:
    if path.exists():
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    return pd.DataFrame()




def _ti_split(value: Any) -> List[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    return [p.strip() for p in re.split(r"[,;|\n]+", str(value)) if p.strip()]


def _ti_future_reportee_template_path() -> Path:
    try:
        base = Path(MY_REPORTEES_FOLDER)
    except NameError:
        base = Path(".")
    return base / "Future_Reportees_Input_Template.csv"


def ensure_future_reportee_template() -> Path:
    preferred = _ti_future_reportee_template_path()
    template = pd.DataFrame([{
        "TrainerName": "Example Future Reportee",
        "Skills": "Power BI; SQL; Microsoft Fabric",
        "ClearedCertifications": "DP-600",
        "PendingCertifications": "DP-700",
        "PastCourses": "DP-600",
        "Feedback": 4.2,
        "Availability": 0.7,
        "Include": "No",
    }])

    preferred.parent.mkdir(parents=True, exist_ok=True)
    if not preferred.exists():
        template.to_csv(preferred, index=False)
    return preferred


def load_future_reportee_profiles() -> pd.DataFrame:
    path = ensure_future_reportee_template()
    try:
        df = pd.read_csv(path)
    except Exception:
        return pd.DataFrame()
    if "Include" in df.columns:
        df = df[df["Include"].astype(str).str.lower().isin(["yes", "y", "true", "1"])]
    rows = []
    for i, row in df.iterrows():
        skills = _ti_split(row.get("Skills"))
        cleared = [_ti_course_code(x) for x in _ti_split(row.get("ClearedCertifications"))]
        pending = [_ti_course_code(x) for x in _ti_split(row.get("PendingCertifications"))]
        past = _ti_split(row.get("PastCourses"))
        rows.append({
            "TrainerID": f"future-{i+1}",
            "TrainerName": row.get("TrainerName", f"Future Reportee {i+1}"),
            "CurrentSkills": sorted(set(skills + cleared + pending)),
            "PastCourses": sorted(set(past)),
            "ClearedCertifications": sorted(set(c for c in cleared if c)),
            "PendingCertifications": sorted(set(c for c in pending if c)),
            "AssignedCertifications": sorted(set(c for c in cleared + pending if c)),
            "Feedback": float(row.get("Feedback", 4.0)) if not pd.isna(row.get("Feedback", 4.0)) else 4.0,
            "Availability": float(row.get("Availability", 0.6)) if not pd.isna(row.get("Availability", 0.6)) else 0.6,
        })
    return pd.DataFrame(rows)


def _ti_clean_course_name(value: Any) -> str:
    text = str(value or "")
    text = re.sub(r"\((?:\s*total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*\d*\s*\)", "", text, flags=re.I)
    text = re.sub(r"\((?:current\s+)?qubits?.*?\)", "", text, flags=re.I)
    text = re.sub(r"\((?:min(?:imum)?\s+score|total\s+count\s+of\s+qubits?).*?\)", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip("-–—").strip()


def _ti_dedupe_courses_and_map(courses: pd.DataFrame, tcm: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if courses.empty or "CourseName" not in courses.columns:
        return courses, tcm
    df = courses.copy()
    df["CourseName"] = df["CourseName"].map(_ti_clean_course_name)
    df["_CourseKey"] = df["CourseName"].map(_ti_norm)
    df["_CourseID_Num"] = pd.to_numeric(df["CourseID"], errors="coerce") if "CourseID" in df.columns else np.nan
    df = df.sort_values(["_CourseKey", "_CourseID_Num"], na_position="last").reset_index(drop=True)
    redirect = {}
    keep = []
    for key, group in df.groupby("_CourseKey", dropna=False):
        if not key:
            keep.extend(group.index.tolist())
            continue
        keep_idx = group.index[0]
        keep_id = df.loc[keep_idx, "CourseID"]
        keep.append(keep_idx)
        for _, dup in group.iloc[1:].iterrows():
            redirect[dup["CourseID"]] = keep_id
    df = df.loc[sorted(set(keep))].drop(columns=["_CourseKey", "_CourseID_Num"]).reset_index(drop=True)
    if not tcm.empty and redirect and "CourseID" in tcm.columns:
        tcm = tcm.copy()
        tcm["CourseID"] = tcm["CourseID"].map(lambda x: redirect.get(x, x))
        if {"TrainerID", "CourseID"}.issubset(tcm.columns):
            tcm = tcm.drop_duplicates(subset=["TrainerID", "CourseID"], keep="last").reset_index(drop=True)
    return df, tcm


def build_current_trainer_intelligence() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    paths = _ti_model_paths()
    if not paths["trainer"].exists():
        raise FileNotFoundError("Trainer model not found. Run option 1 first to refresh governance data.")

    trainers = _ti_read_excel(paths["trainer"], "Trainers")
    courses = _ti_read_excel(paths["course"], "Courses")
    tcm = _ti_read_excel(paths["course"], "Trainer_Course_Map")
    cert_map = _ti_read_excel(paths["cert"], "Trainer_Certification_Map")
    courses, tcm = _ti_dedupe_courses_and_map(courses, tcm)

    course_lookup = {}
    if not courses.empty and {"CourseID", "CourseName"}.issubset(courses.columns):
        course_lookup = courses.set_index("CourseID")["CourseName"].to_dict()

    exam_rows = []
    profile_rows = []

    for _, trainer in trainers.iterrows():
        trainer_id = trainer.get("TrainerID")
        trainer_name = trainer.get("TrainerName", f"Trainer {trainer_id}")

        trainer_course_rows = tcm[tcm["TrainerID"] == trainer_id] if not tcm.empty and "TrainerID" in tcm.columns else pd.DataFrame()
        past_courses = []
        for cid in trainer_course_rows.get("CourseID", pd.Series(dtype=object)).dropna().tolist():
            past_courses.append(str(course_lookup.get(cid, cid)))

        trainer_cert_rows = cert_map[cert_map["TrainerID"] == trainer_id] if not cert_map.empty and "TrainerID" in cert_map.columns else pd.DataFrame()
        cleared_certs, pending_certs, all_certs = [], [], []

        for _, cert in trainer_cert_rows.iterrows():
            exam_name = cert.get("ExamName", "")
            exam_code = _ti_course_code(cert.get("ExamCode") or exam_name or cert.get("CourseID"))
            course_name = course_lookup.get(cert.get("CourseID"), cert.get("CourseID", ""))
            result = cert.get("Result", "")
            approval = cert.get("ApprovalStatus", "")
            status = _ti_status(result, approval)

            if exam_code:
                all_certs.append(exam_code)
            if status == "Cleared":
                cleared_certs.append(exam_code)
            elif exam_code:
                pending_certs.append(exam_code)

            exam_rows.append({
                "TrainerName": trainer_name,
                "ExamCode": exam_code,
                "ExamName": exam_name,
                "CourseName": course_name,
                "Result": result,
                "ApprovalStatus": approval,
                "ExamStatus": status,
            })

        derived_skills = set()
        for value in list(past_courses) + list(all_certs):
            code = _ti_course_code(value)
            if code:
                derived_skills.add(code)
            for _, row in TRAINER_INTELLIGENCE_CATALOG.iterrows():
                if code == row["CourseCode"] or code in str(row["CourseName"]).upper():
                    derived_skills.update(row["Skills"])

        profile_rows.append({
            "TrainerID": trainer_id,
            "TrainerName": trainer_name,
            "CurrentSkills": sorted(derived_skills),
            "PastCourses": sorted(set(past_courses)),
            "ClearedCertifications": sorted(set(c for c in cleared_certs if c)),
            "PendingCertifications": sorted(set(c for c in pending_certs if c)),
            "AssignedCertifications": sorted(set(c for c in all_certs if c)),
            "Feedback": float(trainer.get("Feedback", 4.2)) if not pd.isna(trainer.get("Feedback", 4.2)) else 4.2,
            "Availability": float(trainer.get("Availability", 0.65)) if not pd.isna(trainer.get("Availability", 0.65)) else 0.65,
        })

    profiles = pd.DataFrame(profile_rows)
    future_profiles = load_future_reportee_profiles()
    if not future_profiles.empty:
        profiles = pd.concat([profiles, future_profiles], ignore_index=True)
    exam_status = pd.DataFrame(exam_rows)
    return profiles, exam_status, TRAINER_INTELLIGENCE_CATALOG.copy()


def _ti_skill_gap(current_skills: Iterable[str], target_skills: Iterable[str]) -> Tuple[List[str], float]:
    current = {_ti_norm(x) for x in current_skills if _ti_norm(x)}
    missing = [skill for skill in target_skills if _ti_norm(skill) not in current]
    score = 1 - len(missing) / max(len(list(target_skills)), 1)
    return missing, round(score, 3)


def _ti_cert_scores(cleared: Iterable[str], pending: Iterable[str], required: Iterable[str], course_code: str) -> Tuple[float, float]:
    cleared_set = {_ti_course_code(x) for x in cleared}
    pending_set = {_ti_course_code(x) for x in pending}
    required_set = {_ti_course_code(x) for x in required if x}
    course_code = _ti_course_code(course_code)
    if course_code in cleared_set:
        return 1.0, 0.0
    if course_code in pending_set:
        return 0.0, 1.0
    if not required_set:
        return 0.55, 0.0
    return len(cleared_set & required_set) / len(required_set), len(pending_set & required_set) / len(required_set)


def build_next_course_recommendations(profiles: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:
    rows = []
    w = TRAINER_INTELLIGENCE_WEIGHTS
    for _, trainer in profiles.iterrows():
        for _, course in catalog.iterrows():
            missing, skill_score = _ti_skill_gap(trainer["CurrentSkills"], course["Skills"])
            cert_cleared, cert_pending = _ti_cert_scores(
                trainer["ClearedCertifications"],
                trainer["PendingCertifications"],
                course["RequiredCerts"],
                course["CourseCode"],
            )
            past_codes = {_ti_course_code(x) for x in trainer["PastCourses"]}
            experience = 1.0 if course["CourseCode"] in past_codes else (0.35 if trainer["PastCourses"] else 0.15)
            score = (
                skill_score * w["skill"] +
                cert_cleared * w["cert_cleared"] +
                cert_pending * w["cert_pending"] +
                course["DemandScore"] * w["demand"] +
                experience * w["experience"] +
                course["StrategicBoost"] * w["strategic"]
            )
            if course["StatusNote"]:
                score -= 0.08
            score = max(0, min(1, score))

            rows.append({
                "TrainerName": trainer["TrainerName"],
                "RecommendedCourse": course["CourseCode"],
                "CourseName": course["CourseName"],
                "Category": course["Category"],
                "AssignmentProbability": round(score, 3),
                "DemandScore": course["DemandScore"],
                "SkillMatch": skill_score,
                "CertClearedMatch": cert_cleared,
                "CertPendingMatch": cert_pending,
                "MissingSkills": missing,
                "CurrentClearedCerts": trainer["ClearedCertifications"],
                "CurrentPendingCerts": trainer["PendingCertifications"],
                "StatusNote": course["StatusNote"],
            })

    recs = pd.DataFrame(rows).sort_values(["TrainerName", "AssignmentProbability"], ascending=[True, False])
    recs["Priority"] = pd.cut(
        recs["AssignmentProbability"],
        bins=[-0.01, 0.45, 0.65, 0.80, 1.01],
        labels=["Watch", "Build", "Strong", "Ready Now"],
    )
    return recs


def build_weekly_next_actions(recommendations: pd.DataFrame, top_n: int = 1) -> pd.DataFrame:
    rows = []
    top = recommendations.groupby("TrainerName", group_keys=False).head(top_n)
    for _, row in top.iterrows():
        actions = []
        missing = row["MissingSkills"] or []
        if missing:
            actions.append("Close skill gaps: " + ", ".join(missing[:5]))
        if row["CertClearedMatch"] < 1 and row["CertPendingMatch"] > 0:
            actions.append("Move pending certification to cleared proof in RMS")
        elif row["CertClearedMatch"] < 1 and row["RecommendedCourse"] not in ["AI-3025", "AI-3016", "GH-300"]:
            actions.append(f"Plan certification readiness for {row['RecommendedCourse']}")
        if row["StatusNote"]:
            actions.append(row["StatusNote"])
        actions.append("Update trainer profile evidence: skills, cert proof, labs, delivery notes")
        rows.append({
            "TrainerName": row["TrainerName"],
            "NextBestCourse": row["RecommendedCourse"],
            "CourseName": row["CourseName"],
            "Priority": row["Priority"],
            "AssignmentProbability": row["AssignmentProbability"],
            "NextActions": " | ".join(actions),
        })
    return pd.DataFrame(rows)


def trainer_intelligence_console(export: bool = True, top_n: int = 5, trainer_name: str | None = None) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    profiles, exam_status, catalog = build_current_trainer_intelligence()

    selected_trainer = (trainer_name or "").strip()
    if selected_trainer and selected_trainer.lower() not in {"all", "all trainers", "*"}:
        trainer_mask = profiles["TrainerName"].map(_ti_norm) == _ti_norm(selected_trainer)
        if not trainer_mask.any():
            print(f"Trainer not found: {selected_trainer}. Showing all trainers instead.")
            selected_trainer = ""
        else:
            profiles = profiles[trainer_mask].reset_index(drop=True)
            if not exam_status.empty and "TrainerName" in exam_status.columns:
                exam_status = exam_status[exam_status["TrainerName"].map(_ti_norm) == _ti_norm(selected_trainer)].reset_index(drop=True)

    recommendations = build_next_course_recommendations(profiles, catalog)
    top_recommendations = recommendations.groupby("TrainerName", group_keys=False).head(top_n) if not recommendations.empty else recommendations
    weekly_actions = build_weekly_next_actions(recommendations) if not recommendations.empty else pd.DataFrame(columns=["TrainerName", "NextBestCourse", "CourseName", "Priority", "AssignmentProbability", "NextActions"])

    future_template = ensure_future_reportee_template()
    print(f"Future reportee template: {future_template}")
    if selected_trainer:
        print(f"Selected trainer: {selected_trainer}")
    else:
        print("Selected trainer: All trainers")

    display(Markdown("## Trainer Intelligence: Current Readiness"))
    profile_view = profiles.copy()
    for col in ["CurrentSkills", "PastCourses", "ClearedCertifications", "PendingCertifications", "AssignedCertifications"]:
        profile_view[col] = profile_view[col].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    display(profile_view)

    display(Markdown("## Exam / Certification Status"))
    display(exam_status.sort_values(["TrainerName", "ExamStatus", "ExamCode"]) if not exam_status.empty else exam_status)

    display(Markdown("## Next Course Recommendations"))
    rec_view = top_recommendations.copy()
    for col in ["MissingSkills", "CurrentClearedCerts", "CurrentPendingCerts"]:
        rec_view[col] = rec_view[col].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)
    display(rec_view)

    display(Markdown("## Weekly Action Plan"))
    display(weekly_actions)

    if export:
        preferred_output = Path(MY_REPORTEES_FOLDER) / "Trainer_Intelligence_Output_Latest.xlsx"
        try:
            with pd.ExcelWriter(preferred_output, engine="openpyxl") as writer:
                profile_view.to_excel(writer, sheet_name="CurrentReadiness", index=False)
                exam_status.to_excel(writer, sheet_name="ExamStatus", index=False)
                rec_view.to_excel(writer, sheet_name="TopRecommendations", index=False)
                weekly_actions.to_excel(writer, sheet_name="WeeklyActions", index=False)
                catalog.to_excel(writer, sheet_name="DemandCatalog", index=False)
            print(f"\nExported Trainer Intelligence workbook: {preferred_output}")
        except Exception as exc:
            print(f"Export failed. Close the Excel file if it is open, then rerun: {preferred_output} ({exc})")

    return profiles, exam_status, top_recommendations, weekly_actions


def _ti_json_ready_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert dashboard data to JSON-safe values.
    Cast categorical columns before blank filling so pandas does not reject new blank categories.
    """
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].apply(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)
        if str(out[col].dtype) == "category":
            out[col] = out[col].astype(object)
    return out.where(pd.notna(out), "")


def trainer_intelligence_html_dashboard(export: bool = True, top_n: int = 5) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    HTML + JS trainer intelligence dashboard.
    Mirrors menu option 2 behavior: trainer dropdown, search/filtering, KPI blocks,
    and segregated sections for readiness, exams, recommendations, and weekly actions.
    """
    profiles, exam_status, catalog = build_current_trainer_intelligence()
    recommendations = build_next_course_recommendations(profiles, catalog)
    top_recommendations = recommendations.groupby("TrainerName", group_keys=False).head(top_n) if not recommendations.empty else recommendations
    weekly_actions = build_weekly_next_actions(recommendations) if not recommendations.empty else pd.DataFrame(
        columns=["TrainerName", "NextBestCourse", "CourseName", "Priority", "AssignmentProbability", "NextActions"]
    )

    future_template = ensure_future_reportee_template()
    profile_view = _ti_json_ready_df(profiles)
    exam_view = _ti_json_ready_df(exam_status.sort_values(["TrainerName", "ExamStatus", "ExamCode"]) if not exam_status.empty else exam_status)
    rec_view = _ti_json_ready_df(top_recommendations)
    action_view = _ti_json_ready_df(weekly_actions)

    trainers = sorted(profile_view.get("TrainerName", pd.Series(dtype=str)).dropna().astype(str).unique().tolist())
    payload = {
        "profiles": profile_view.to_dict("records"),
        "exams": exam_view.to_dict("records"),
        "recommendations": rec_view.to_dict("records"),
        "actions": action_view.to_dict("records"),
        "trainers": trainers,
        "futureTemplate": str(future_template),
    }
    payload_json = json.dumps(payload, ensure_ascii=False, default=str).replace("</", "<\\/")

    html = f"""
    <div id="trainerIntelligenceDashboard" style="font-family:Segoe UI, Arial, sans-serif; color:#1f2937;">
      <style>
        #trainerIntelligenceDashboard .toolbar {{display:flex; gap:10px; flex-wrap:wrap; align-items:end; margin:12px 0;}}
        #trainerIntelligenceDashboard label {{font-size:12px; color:#4b5563; display:block; margin-bottom:4px;}}
        #trainerIntelligenceDashboard select, #trainerIntelligenceDashboard input {{
          border:1px solid #cbd5e1; border-radius:6px; padding:7px 8px; min-width:210px; background:white;
        }}
        #trainerIntelligenceDashboard .kpis {{display:grid; grid-template-columns:repeat(5,minmax(120px,1fr)); gap:8px; margin:10px 0 14px;}}
        #trainerIntelligenceDashboard .kpi {{border:1px solid #d1d5db; border-radius:8px; padding:10px; background:#f8fafc;}}
        #trainerIntelligenceDashboard .kpi b {{display:block; font-size:20px; color:#111827;}}
        #trainerIntelligenceDashboard .kpi span {{font-size:12px; color:#64748b;}}
        #trainerIntelligenceDashboard .tabs {{display:flex; gap:6px; flex-wrap:wrap; margin:8px 0 10px;}}
        #trainerIntelligenceDashboard .tab {{
          border:1px solid #cbd5e1; background:white; border-radius:6px; padding:7px 10px; cursor:pointer; font-size:12px;
        }}
        #trainerIntelligenceDashboard .tab.active {{background:#0f172a; color:white; border-color:#0f172a;}}
        #trainerIntelligenceDashboard h3 {{margin:4px 0 2px;}}
        #trainerIntelligenceDashboard h4 {{margin:12px 0 8px; font-size:15px; color:#111827;}}
        #trainerIntelligenceDashboard table {{border-collapse:collapse; width:100%; font-size:12px;}}
        #trainerIntelligenceDashboard th {{position:sticky; top:0; background:#0f172a; color:white; text-align:left; padding:8px;}}
        #trainerIntelligenceDashboard td {{border-bottom:1px solid #e5e7eb; padding:7px 8px; vertical-align:top;}}
        #trainerIntelligenceDashboard tr:hover td {{background:#f1f5f9;}}
        #trainerIntelligenceDashboard .tableWrap {{max-height:520px; overflow:auto; border:1px solid #d1d5db; border-radius:8px;}}
        #trainerIntelligenceDashboard .pill {{display:inline-block; border:1px solid #cbd5e1; border-radius:999px; padding:2px 8px; background:#fff;}}
        #trainerIntelligenceDashboard .muted {{font-size:12px; color:#64748b;}}
        @media (max-width: 860px) {{
          #trainerIntelligenceDashboard .kpis {{grid-template-columns:repeat(2,minmax(120px,1fr));}}
          #trainerIntelligenceDashboard select, #trainerIntelligenceDashboard input {{min-width:100%;}}
        }}
      </style>
      <h3>Trainer Intelligence Dashboard</h3>
      <div class="muted">Select trainer and review readiness, exams, next recommendations, and weekly actions in one place.</div>
      <div class="toolbar">
        <div><label>Trainer</label><select id="tiTrainer"></select></div>
        <div><label>Search</label><input id="tiSearch" placeholder="Search skills, exams, courses, actions"></div>
      </div>
      <div class="kpis">
        <div class="kpi"><b id="tiKpiTrainers">0</b><span>Trainers</span></div>
        <div class="kpi"><b id="tiKpiCleared">0</b><span>Cleared certs</span></div>
        <div class="kpi"><b id="tiKpiPending">0</b><span>Pending certs</span></div>
        <div class="kpi"><b id="tiKpiCourses">0</b><span>Past courses</span></div>
        <div class="kpi"><b id="tiKpiTop">0</b><span>Top recommendations</span></div>
      </div>
      <div class="tabs">
        <button class="tab active" data-view="readiness">Current Readiness</button>
        <button class="tab" data-view="exams">Exam Status</button>
        <button class="tab" data-view="recommendations">Next Recommendations</button>
        <button class="tab" data-view="actions">Weekly Actions</button>
      </div>
      <div class="muted">Future reportee input template: <span id="tiTemplate"></span></div>
      <div id="tiContent"></div>
    </div>
    <script>
    (function() {{
      const data = {payload_json};
      const root = document.getElementById('trainerIntelligenceDashboard');
      const byId = id => root.querySelector('#' + id);
      const trainerSel = byId('tiTrainer'), search = byId('tiSearch'), content = byId('tiContent');
      let activeView = 'readiness';
      function text(v) {{ return (v === undefined || v === null) ? '' : String(v); }}
      function esc(v) {{ return text(v).replace(/[&<>"']/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[ch])); }}
      function addOptions() {{
        trainerSel.innerHTML = '<option value="">All trainers</option>' + data.trainers.map(t => `<option value="${{esc(t)}}">${{esc(t)}}</option>`).join('');
        byId('tiTemplate').textContent = data.futureTemplate || '';
      }}
      function rowMatch(row) {{
        if (trainerSel.value && text(row.TrainerName) !== trainerSel.value) return false;
        const q = search.value.toLowerCase().trim();
        if (!q) return true;
        return Object.values(row).map(text).join(' ').toLowerCase().indexOf(q) >= 0;
      }}
      function filtered(rows) {{ return rows.filter(rowMatch); }}
      function countCsv(value) {{
        const s = text(value).trim();
        if (!s) return 0;
        return s.split(',').map(x => x.trim()).filter(Boolean).length;
      }}
      function updateKpis() {{
        const profiles = filtered(data.profiles);
        const recs = filtered(data.recommendations);
        byId('tiKpiTrainers').textContent = new Set(profiles.map(r => text(r.TrainerName)).filter(Boolean)).size;
        byId('tiKpiCleared').textContent = profiles.reduce((n,r) => n + countCsv(r.ClearedCertifications), 0);
        byId('tiKpiPending').textContent = profiles.reduce((n,r) => n + countCsv(r.PendingCertifications), 0);
        byId('tiKpiCourses').textContent = profiles.reduce((n,r) => n + countCsv(r.PastCourses), 0);
        byId('tiKpiTop').textContent = recs.length;
      }}
      function table(headers, rows, renderRow) {{
        return `<div class="tableWrap"><table><thead><tr>${{headers.map(h=>`<th>${{esc(h)}}</th>`).join('')}}</tr></thead><tbody>${{rows.map(renderRow).join('') || `<tr><td colspan="${{headers.length}}">No rows for this filter.</td></tr>`}}</tbody></table></div>`;
      }}
      function renderReadiness() {{
        const rows = filtered(data.profiles);
        content.innerHTML = '<h4>Current Readiness</h4>' + table(['Trainer','Current Skills','Past Courses','Cleared','Pending','Feedback','Availability'], rows, r => `
          <tr><td><b>${{esc(r.TrainerName)}}</b></td><td>${{esc(r.CurrentSkills)}}</td><td>${{esc(r.PastCourses)}}</td><td>${{esc(r.ClearedCertifications)}}</td><td>${{esc(r.PendingCertifications)}}</td><td>${{esc(r.Feedback)}}</td><td>${{esc(r.Availability)}}</td></tr>`);
      }}
      function renderExams() {{
        const rows = filtered(data.exams);
        content.innerHTML = '<h4>Exam / Certification Status</h4>' + table(['Trainer','Exam Code','Exam Name','Course','Result','Approval','Status'], rows, r => `
          <tr><td>${{esc(r.TrainerName)}}</td><td><span class="pill">${{esc(r.ExamCode)}}</span></td><td>${{esc(r.ExamName)}}</td><td>${{esc(r.CourseName)}}</td><td>${{esc(r.Result)}}</td><td>${{esc(r.ApprovalStatus)}}</td><td><span class="pill">${{esc(r.ExamStatus)}}</span></td></tr>`);
      }}
      function renderRecommendations() {{
        const rows = filtered(data.recommendations);
        content.innerHTML = '<h4>Next Course Recommendations</h4>' + table(['Trainer','Course','Category','Priority','Probability','Skill Match','Missing Skills','Cert Readiness'], rows, r => `
          <tr><td>${{esc(r.TrainerName)}}</td><td><b>${{esc(r.RecommendedCourse)}}</b><br><span class="muted">${{esc(r.CourseName)}}</span></td><td>${{esc(r.Category)}}</td><td><span class="pill">${{esc(r.Priority)}}</span></td><td>${{esc(r.AssignmentProbability)}}</td><td>${{esc(r.SkillMatchScore)}}</td><td>${{esc(r.MissingSkills)}}</td><td>Cleared: ${{esc(r.CertClearedMatch)}}<br>Pending: ${{esc(r.CertPendingMatch)}}</td></tr>`);
      }}
      function renderActions() {{
        const rows = filtered(data.actions);
        content.innerHTML = '<h4>Weekly Action Plan</h4>' + table(['Trainer','Next Best Course','Priority','Probability','Actions'], rows, r => `
          <tr><td><b>${{esc(r.TrainerName)}}</b></td><td>${{esc(r.NextBestCourse)}}<br><span class="muted">${{esc(r.CourseName)}}</span></td><td><span class="pill">${{esc(r.Priority)}}</span></td><td>${{esc(r.AssignmentProbability)}}</td><td>${{esc(r.NextActions).replaceAll(' | ', '<br>')}}</td></tr>`);
      }}
      function render() {{
        updateKpis();
        if (activeView === 'readiness') renderReadiness();
        else if (activeView === 'exams') renderExams();
        else if (activeView === 'recommendations') renderRecommendations();
        else renderActions();
      }}
      root.querySelectorAll('.tab').forEach(btn => btn.addEventListener('click', () => {{
        root.querySelectorAll('.tab').forEach(b => b.classList.remove('active'));
        btn.classList.add('active');
        activeView = btn.getAttribute('data-view');
        render();
      }}));
      trainerSel.addEventListener('input', render);
      search.addEventListener('input', render);
      addOptions();
      render();
    }})();
    </script>
    """

    try:
        display(HTML(html))
    except Exception:
        print("HTML display is unavailable. Showing recommendation table instead.")
        display(rec_view.head(20))

    if export:
        preferred_output = Path(MY_REPORTEES_FOLDER) / "Trainer_Intelligence_Output_Latest.xlsx"
        try:
            with pd.ExcelWriter(preferred_output, engine="openpyxl") as writer:
                profile_view.to_excel(writer, sheet_name="CurrentReadiness", index=False)
                exam_view.to_excel(writer, sheet_name="ExamStatus", index=False)
                rec_view.to_excel(writer, sheet_name="TopRecommendations", index=False)
                action_view.to_excel(writer, sheet_name="WeeklyActions", index=False)
                catalog.to_excel(writer, sheet_name="DemandCatalog", index=False)
            print(f"Exported Trainer Intelligence workbook: {preferred_output}")
        except Exception as exc:
            print(f"Export failed. Close the Excel file if it is open, then rerun: {preferred_output} ({exc})")

    return profiles, exam_status, top_recommendations, weekly_actions


In [ ]:
"""
Cell 11B - Major makeover: trainer-course HTML dashboard, RMS popularity intelligence,
and AutoTall allocation intelligence.

This cell is intentionally self-contained. It uses the normalized trainer/course/cert
models already maintained by the notebook, then adds browser-backed RMS analysis when
you run it inside an authenticated RMS session.
"""

from __future__ import annotations
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, Iterable, List, Tuple
import json
import re
import math
import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(value):
        print(value)


POPULAR_COURSES_URL = "https://rms.koenig-solutions.com/Manager/frmPopularCourses.aspx"
AUTOTALL_DASHBOARD_URL = "https://rms.koenig-solutions.com/AutoTallDashboard.aspx?UId=zAPSAzVXZzfrj/i1WxohaQ=="


def _mk_model_folder() -> Path:
    try:
        return Path(MY_REPORTEES_FOLDER)
    except NameError:
        return Path(".")


def _mk_paths() -> Dict[str, Path]:
    base = _mk_model_folder()
    return {
        "trainer": base / "RMS_Normalized_Trainer_Model.xlsx",
        "course": base / "RMS_Normalized_Course_Model.xlsx",
        "cert": base / "RMS_Normalized_Certification_Model.xlsx",
        "output": base,
    }


def _mk_read_excel(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")


def _mk_norm(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value).lower()
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text)
    text = re.sub(r"[^a-z0-9+#.\s/-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _mk_course_code(value: Any) -> str:
    text = str(value or "").upper()
    match = re.search(r"\b([A-Z]{1,8}-\d{2,5})\b", text)
    return match.group(1) if match else ""


def _mk_token_set(value: Any) -> set[str]:
    return set(_mk_norm(value).split())


def _mk_first_existing(columns: Iterable[str], candidates: Iterable[str]) -> str | None:
    normalized = {_mk_norm(c).replace(" ", ""): c for c in columns}
    for candidate in candidates:
        key = _mk_norm(candidate).replace(" ", "")
        if key in normalized:
            return normalized[key]
    for col in columns:
        col_key = _mk_norm(col)
        for candidate in candidates:
            cand_key = _mk_norm(candidate)
            if cand_key and cand_key in col_key:
                return col
    return None


def _mk_safe_number(value: Any, default: float = 0.0) -> float:
    try:
        if value is None or pd.isna(value):
            return default
        text = str(value).replace("%", "").replace(",", "").strip()
        if not text:
            return default
        return float(text)
    except Exception:
        return default


def _mk_clean_course_name_for_view(value: Any) -> str:
    text = str(value or "")
    text = re.sub(r"\((?:\s*total\s+)?assign(?:ment)?s?\s*(?:delivered|done|completed)?\s*[:\-]?\s*\d*\s*\)", "", text, flags=re.I)
    text = re.sub(r"\((?:current\s+)?qubits?.*?\)", "", text, flags=re.I)
    text = re.sub(r"\((?:min(?:imum)?\s+score|total\s+count\s+of\s+qubits?).*?\)", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip("-–—").strip()


def _mk_dedupe_courses_and_map_for_view(courses: pd.DataFrame, tcm: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if courses.empty or "CourseName" not in courses.columns:
        return courses, tcm
    df = courses.copy()
    df["CourseName"] = df["CourseName"].map(_mk_clean_course_name_for_view)
    df["_CourseKey"] = df["CourseName"].map(_mk_norm)
    df["_CourseID_Num"] = pd.to_numeric(df["CourseID"], errors="coerce") if "CourseID" in df.columns else np.nan
    df = df.sort_values(["_CourseKey", "_CourseID_Num"], na_position="last").reset_index(drop=True)
    redirect = {}
    keep = []
    for key, group in df.groupby("_CourseKey", dropna=False):
        if not key:
            keep.extend(group.index.tolist())
            continue
        keep_idx = group.index[0]
        keep_id = df.loc[keep_idx, "CourseID"]
        keep.append(keep_idx)
        for _, dup in group.iloc[1:].iterrows():
            redirect[dup["CourseID"]] = keep_id
    df = df.loc[sorted(set(keep))].drop(columns=["_CourseKey", "_CourseID_Num"]).reset_index(drop=True)
    if not tcm.empty and redirect and "CourseID" in tcm.columns:
        tcm = tcm.copy()
        tcm["CourseID"] = tcm["CourseID"].map(lambda x: redirect.get(x, x))
        if {"TrainerID", "CourseID"}.issubset(tcm.columns):
            tcm = tcm.drop_duplicates(subset=["TrainerID", "CourseID"], keep="last").reset_index(drop=True)
    return df, tcm


def load_trainer_course_solved_view() -> pd.DataFrame:
    """
    Build the complete trainer-course solved view from the normalized Excel models.
    Includes only real model data; no static trainer is forced.
    """
    paths = _mk_paths()
    trainers = _mk_read_excel(paths["trainer"], "Trainers")
    courses = _mk_read_excel(paths["course"], "Courses")
    tcm = _mk_read_excel(paths["course"], "Trainer_Course_Map")
    certs = _mk_read_excel(paths["cert"], "Trainer_Certification_Map")
    courses, tcm = _mk_dedupe_courses_and_map_for_view(courses, tcm)

    if trainers.empty or courses.empty or tcm.empty:
        print("Trainer/course models are not ready. Run menu option 1 to refresh RMS governance data first.")
        return pd.DataFrame()

    df = tcm.copy()
    df = df.merge(trainers, on="TrainerID", how="left", suffixes=("", "_Trainer"))
    df = df.merge(courses, on="CourseID", how="left", suffixes=("", "_Course"))

    if not certs.empty:
        cert_cols = [c for c in ["TrainerID", "CourseID", "ExamName", "ExamCode", "ExamURL", "Result", "ApprovalStatus", "IssueDate"] if c in certs.columns]
        cert_view = certs[cert_cols].copy()
        df = df.merge(cert_view, on=["TrainerID", "CourseID"], how="left", suffixes=("", "_TrainerCert"))
    else:
        df["Result"] = ""
        df["ApprovalStatus"] = ""

    for col in ["TrainerName", "CourseName", "ExamURL", "CertificationVendor", "CertificationName", "CertificationStatus", "Result", "ApprovalStatus"]:
        if col not in df.columns:
            df[col] = ""

    def cert_state(row: pd.Series) -> str:
        result_text = _mk_norm(str(row.get("Result", "")) + " " + str(row.get("ApprovalStatus", "")))
        if any(x in result_text for x in ["pass", "passed", "clear", "cleared", "approved", "success"]):
            return "Cleared"
        if result_text.strip():
            return "Pending/Assigned"
        if str(row.get("ExamURL", "")).strip():
            return "Mapped, Not Cleared"
        return "No Certification Match"

    df["TrainerCertificationState"] = df.apply(cert_state, axis=1)
    df["HasValidCertificationLink"] = df["ExamURL"].fillna("").astype(str).str.startswith(("http://", "https://"))

    ordered = [
        "TrainerID", "TrainerName", "CourseID", "CourseName",
        "AssignmentsDelivered", "QubitScore", "MinScoreRequired", "QubitQuestionCount", "IsFutureLive",
        "CertificationVendor", "CertificationName", "CertificationStatus", "ExamURL",
        "Result", "ApprovalStatus", "TrainerCertificationState", "HasValidCertificationLink",
    ]
    ordered = [c for c in ordered if c in df.columns]
    return df[ordered].sort_values(["TrainerName", "CourseName"]).reset_index(drop=True)


def trainer_course_html_dashboard(export: bool = True) -> pd.DataFrame:
    """
    HTML + JS dashboard for 'which courses are solved by trainers'.
    Provides trainer/status/result dropdowns plus search without needing ipywidgets.
    """
    df = load_trainer_course_solved_view()
    if df.empty:
        return df

    def _dashboard_sort_rank(row: pd.Series) -> int:
        state = str(row.get("TrainerCertificationState", "") or "").strip().lower()
        has_link = bool(row.get("HasValidCertificationLink", False)) or str(row.get("ExamURL", "") or "").startswith(("http://", "https://"))
        if state == "cleared":
            return 0
        if has_link:
            return 1
        return 2

    df = df.copy()
    df["_SortRank"] = df.apply(_dashboard_sort_rank, axis=1)
    df = df.sort_values(["_SortRank", "TrainerName", "CourseName"], ascending=[True, True, True]).drop(columns=["_SortRank"]).reset_index(drop=True)

    rows = df.fillna("").astype(str).to_dict("records")
    trainers = sorted(df["TrainerName"].fillna("").astype(str).unique().tolist())
    statuses = sorted(df["CertificationStatus"].fillna("").astype(str).replace("", "Blank").unique().tolist())
    cert_states = sorted(df["TrainerCertificationState"].fillna("").astype(str).replace("", "Blank").unique().tolist())

    payload = {
        "rows": rows,
        "trainers": trainers,
        "statuses": statuses,
        "certStates": cert_states,
    }
    payload_json = json.dumps(payload, ensure_ascii=False).replace("</", "<\\/")

    html = f"""
    <div id="trainerSolvedDashboard" style="font-family:Segoe UI, Arial, sans-serif; color:#1f2937;">
      <style>
        #trainerSolvedDashboard .bar {{display:flex; gap:10px; flex-wrap:wrap; align-items:end; margin:12px 0;}}
        #trainerSolvedDashboard label {{font-size:12px; color:#4b5563; display:block; margin-bottom:4px;}}
        #trainerSolvedDashboard select, #trainerSolvedDashboard input {{
          border:1px solid #cbd5e1; border-radius:6px; padding:7px 8px; min-width:190px; background:white;
        }}
        #trainerSolvedDashboard .kpis {{display:grid; grid-template-columns:repeat(4,minmax(120px,1fr)); gap:8px; margin:10px 0 14px;}}
        #trainerSolvedDashboard .kpi {{border:1px solid #d1d5db; border-radius:8px; padding:10px; background:#f8fafc;}}
        #trainerSolvedDashboard .kpi b {{display:block; font-size:20px; color:#111827;}}
        #trainerSolvedDashboard .kpi span {{font-size:12px; color:#64748b;}}
        #trainerSolvedDashboard table {{border-collapse:collapse; width:100%; table-layout:fixed; font-size:12px;}}
        #trainerSolvedDashboard th {{position:sticky; top:0; background:#0f172a; color:white; text-align:left; padding:7px 8px;}}
        #trainerSolvedDashboard td {{border-bottom:1px solid #e5e7eb; padding:7px 8px; vertical-align:middle;}}
        #trainerSolvedDashboard tr:hover td {{background:#f1f5f9;}}
        #trainerSolvedDashboard a {{color:#0f766e; font-weight:700; text-decoration:none;}}
        #trainerSolvedDashboard .tableWrap {{max-height:620px; overflow:auto; border:1px solid #d1d5db; border-radius:8px;}}
        #trainerSolvedDashboard .trainerCol {{width:14%;}}
        #trainerSolvedDashboard .courseCol {{width:33%;}}
        #trainerSolvedDashboard .deliveredCol {{width:7%; text-align:center;}}
        #trainerSolvedDashboard .qubitCol {{width:10%; text-align:center;}}
        #trainerSolvedDashboard .certCol {{width:24%;}}
        #trainerSolvedDashboard .stateCol {{width:12%; text-align:center;}}
        #trainerSolvedDashboard .courseName {{font-weight:700; line-height:1.25;}}
        #trainerSolvedDashboard .courseMeta {{color:#64748b; font-size:11px; margin-top:2px;}}
        #trainerSolvedDashboard .certBox {{border-radius:7px; padding:6px 8px; line-height:1.25; border:1px solid;}}
        #trainerSolvedDashboard .certMapped {{background:#ecfdf5; color:#065f46; border-color:#99f6e4;}}
        #trainerSolvedDashboard .certMissing {{background:#fffbeb; color:#92400e; border-color:#fde68a;}}
        #trainerSolvedDashboard .certPending {{background:#eff6ff; color:#1d4ed8; border-color:#bfdbfe;}}
        #trainerSolvedDashboard .certTitle {{font-weight:700;}}
        #trainerSolvedDashboard .linkChip {{float:right; border:1px solid currentColor; border-radius:999px; padding:1px 7px; font-size:11px; margin-left:6px;}}
        #trainerSolvedDashboard .pill {{display:inline-block; border:1px solid #cbd5e1; border-radius:999px; padding:3px 9px; background:#fff; white-space:nowrap;}}
        #trainerSolvedDashboard .stateCleared {{background:#dcfce7; color:#166534; border-color:#86efac;}}
        #trainerSolvedDashboard .statePending {{background:#fef3c7; color:#92400e; border-color:#fcd34d;}}
        #trainerSolvedDashboard .stateMissing {{background:#fee2e2; color:#991b1b; border-color:#fecaca;}}
        @media (max-width: 760px) {{
          #trainerSolvedDashboard .kpis {{grid-template-columns:repeat(2,minmax(120px,1fr));}}
          #trainerSolvedDashboard select, #trainerSolvedDashboard input {{min-width:100%;}}
        }}
      </style>
      <h3 style="margin:4px 0 2px;">Trainer Solved Courses Dashboard</h3>
      <div style="font-size:12px;color:#64748b;">Filter by trainer, mapped certification status, trainer exam result, and course text.</div>
      <div class="bar">
        <div><label>Trainer</label><select id="tsTrainer"></select></div>
        <div><label>Certification Mapping</label><select id="tsStatus"></select></div>
        <div><label>Trainer Exam State</label><select id="tsCertState"></select></div>
        <div><label>Search Course</label><input id="tsSearch" placeholder="Type course/code/vendor"></div>
      </div>
      <div class="kpis">
        <div class="kpi"><b id="tsKpiRows">0</b><span>Visible solved rows</span></div>
        <div class="kpi"><b id="tsKpiTrainers">0</b><span>Visible trainers</span></div>
        <div class="kpi"><b id="tsKpiMapped">0</b><span>Rows with valid cert link</span></div>
        <div class="kpi"><b id="tsKpiCleared">0</b><span>Trainer certs cleared</span></div>
      </div>
      <div class="tableWrap"><table>
        <thead><tr>
          <th class="trainerCol">Trainer</th>
          <th class="courseCol">Course</th>
          <th class="deliveredCol">Done</th>
          <th class="qubitCol">Qubit</th>
          <th class="certCol">Certification</th>
          <th class="stateCol">Trainer State</th>
        </tr></thead>
        <tbody id="tsBody"></tbody>
      </table></div>
    </div>
    <script>
    (function() {{
      const data = {payload_json};
      const root = document.getElementById('trainerSolvedDashboard');
      const byId = id => root.querySelector('#' + id);
      const trainerSel = byId('tsTrainer'), statusSel = byId('tsStatus'), certSel = byId('tsCertState'), search = byId('tsSearch'), body = byId('tsBody');
      function addOptions(sel, values, label) {{
        sel.innerHTML = '';
        const opt = document.createElement('option'); opt.value = ''; opt.textContent = label; sel.appendChild(opt);
        values.filter(Boolean).forEach(v => {{ const o=document.createElement('option'); o.value=v; o.textContent=v; sel.appendChild(o); }});
      }}
      function text(v) {{ return (v === undefined || v === null) ? '' : String(v); }}
      function sortRank(r) {{
        const state = text(r.TrainerCertificationState).trim().toLowerCase();
        const hasLink = text(r.HasValidCertificationLink).toLowerCase() === 'true' || text(r.ExamURL).startsWith('http');
        if (state === 'cleared') return 0;
        if (hasLink) return 1;
        return 2;
      }}
      function render() {{
        const q = search.value.toLowerCase().trim();
        const filtered = data.rows.filter(r => {{
          if (trainerSel.value && text(r.TrainerName) !== trainerSel.value) return false;
          const status = text(r.CertificationStatus) || 'Blank';
          if (statusSel.value && status !== statusSel.value) return false;
          const state = text(r.TrainerCertificationState) || 'Blank';
          if (certSel.value && state !== certSel.value) return false;
          const hay = [r.CourseName, r.CourseID, r.CertificationVendor, r.CertificationName, r.ExamURL].map(text).join(' ').toLowerCase();
          return !q || hay.indexOf(q) >= 0;
        }}).sort((a, b) => {{
          const rankDiff = sortRank(a) - sortRank(b);
          if (rankDiff !== 0) return rankDiff;
          const trainerDiff = text(a.TrainerName).localeCompare(text(b.TrainerName));
          if (trainerDiff !== 0) return trainerDiff;
          return text(a.CourseName).localeCompare(text(b.CourseName));
        }});
        byId('tsKpiRows').textContent = filtered.length;
        byId('tsKpiTrainers').textContent = new Set(filtered.map(r => text(r.TrainerName)).filter(Boolean)).size;
        byId('tsKpiMapped').textContent = filtered.filter(r => text(r.HasValidCertificationLink).toLowerCase() === 'true').length;
        byId('tsKpiCleared').textContent = filtered.filter(r => text(r.TrainerCertificationState) === 'Cleared').length;
        body.innerHTML = filtered.map(r => {{
          const url = text(r.ExamURL);
          const hasLink = url.startsWith('http');
          const status = text(r.CertificationStatus);
          const state = text(r.TrainerCertificationState);
          const certName = [r.CertificationVendor, r.CertificationName].map(text).filter(Boolean).join(' / ');
          const certClass = hasLink ? (state === 'Cleared' ? 'certMapped' : 'certPending') : 'certMissing';
          const stateClass = state === 'Cleared' ? 'stateCleared' : (state === 'No Certification Match' ? 'stateMissing' : 'statePending');
          const certHtml = hasLink
            ? `<div class="certBox ${{certClass}}"><a class="linkChip" href="${{url}}" target="_blank" title="Open exam/certification page">Link</a><div class="certTitle">${{certName || 'Mapped certification'}}</div><div>${{status || 'Mapped'}}</div></div>`
            : `<div class="certBox certMissing"><div class="certTitle">No valid certification mapped</div><div>Needs review</div></div>`;
          const qScore = text(r.QubitScore);
          const qMin = text(r.MinScoreRequired);
          return `<tr>
            <td class="trainerCol">${{text(r.TrainerName)}}</td>
            <td class="courseCol"><div class="courseName">${{text(r.CourseName)}}</div><div class="courseMeta">Course ID: ${{text(r.CourseID)}}</div></td>
            <td class="deliveredCol">${{text(r.AssignmentsDelivered)}}</td>
            <td class="qubitCol">${{qScore || '-'}}<br><span class="courseMeta">min ${{qMin || '-'}}</span></td>
            <td class="certCol">${{certHtml}}</td>
            <td class="stateCol"><span class="pill ${{stateClass}}">${{state || 'Unknown'}}</span></td>
          </tr>`;
        }}).join('');
      }}
      addOptions(trainerSel, data.trainers, 'All trainers');
      addOptions(statusSel, data.statuses, 'All mapping statuses');
      addOptions(certSel, data.certStates, 'All trainer states');
      [trainerSel, statusSel, certSel, search].forEach(el => el.addEventListener('input', render));
      render();
    }})();
    </script>
    """

    if HTML is not None:
        display(HTML(html))
    else:
        print(df.head(20).to_string(index=False))

    if export:
        out = _mk_paths()["output"] / "Trainer_Solved_Courses_Dashboard_Latest.xlsx"
        try:
            df.to_excel(out, index=False)
            print(f"Exported trainer solved courses view: {out}")
        except Exception as exc:
            print(f"Export failed. Close the Excel file if it is open, then rerun: {out} ({exc})")
    return df


def _mk_read_html_tables_from_driver(driver, url: str, wait_seconds: int = 3) -> List[pd.DataFrame]:
    if driver is None:
        print("No live RMS browser session is available; saved model data will be used where possible.")
        return []
    try:
        driver.get(url)
        try:
            import time
            time.sleep(wait_seconds)
        except Exception:
            pass
        html = driver.page_source
        tables = _mk_tables_from_html(html)
        return [table for table in tables if not table.empty]
    except Exception as exc:
        print(f"Could not read RMS page tables from {url}: {exc}")
        return []


def fetch_rms_popular_courses_with_driver(driver=None) -> pd.DataFrame:
    """
    Parse RMS Popular Courses after login. This does not use mock popularity.
    """
    tables = _mk_read_html_tables_from_driver(driver, POPULAR_COURSES_URL)
    if not tables:
        return pd.DataFrame()
    best = max(tables, key=lambda t: (t.shape[0], t.shape[1])).copy()
    best.columns = [str(c).strip() for c in best.columns]
    course_col = _mk_first_existing(best.columns, ["Course", "Course Name", "CourseName", "Title", "Name"])
    popularity_col = _mk_first_existing(best.columns, ["Popularity", "Popular", "Count", "Bookings", "Enquiries", "Demand", "Total"])
    category_col = _mk_first_existing(best.columns, ["Category", "Technology", "Vendor", "Segment"])

    if course_col is None:
        course_col = best.columns[0]
    out = pd.DataFrame()
    out["CourseName"] = best[course_col].astype(str).str.strip()
    out["CourseCode"] = out["CourseName"].apply(_mk_course_code)
    out["RMSPopularityRaw"] = best[popularity_col].apply(_mk_safe_number) if popularity_col else 1.0
    out["RMSCategory"] = best[category_col].astype(str) if category_col else ""
    out = out[out["CourseName"].str.len() > 0].copy()
    total = max(out["RMSPopularityRaw"].sum(), 1.0)
    out["PopularityRatio"] = out["RMSPopularityRaw"] / total
    out["DemandScore"] = (out["RMSPopularityRaw"] / max(out["RMSPopularityRaw"].max(), 1.0)).clip(0, 1)
    return out.sort_values(["DemandScore", "CourseName"], ascending=[False, True]).reset_index(drop=True)


def _mk_catalog_from_models() -> pd.DataFrame:
    paths = _mk_paths()
    courses = _mk_read_excel(paths["course"], "Courses")
    if courses.empty:
        return pd.DataFrame()
    catalog = courses.copy()
    if "CourseName" not in catalog.columns:
        return pd.DataFrame()
    catalog["CourseCode"] = catalog["CourseName"].apply(_mk_course_code)
    catalog["ModelCourseNameNorm"] = catalog["CourseName"].apply(_mk_norm)
    return catalog


def _mk_profile_skill_map() -> pd.DataFrame:
    try:
        profiles, exam_df, courses = build_current_trainer_intelligence()
        return profiles
    except Exception as exc:
        print(f"Could not build trainer skill profiles: {exc}")
        return pd.DataFrame()


def _mk_skill_match(profile_skills: Iterable[str], course_name: Any, course_code: Any = "") -> Tuple[float, List[str]]:
    profile_tokens = set()
    for skill in profile_skills or []:
        profile_tokens |= _mk_token_set(skill)
    course_tokens = _mk_token_set(str(course_name) + " " + str(course_code))
    useful = {t for t in course_tokens if len(t) >= 3 and t not in {"course", "training", "microsoft", "certified", "associate", "fundamentals"}}
    if not useful:
        return 0.0, []
    matched = useful & profile_tokens
    missing = sorted(useful - profile_tokens)[:8]
    return round(len(matched) / len(useful), 3), missing


def build_rms_popularity_opportunity_analysis(popular_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Combine RMS course popularity, current trainer course coverage, saved skills,
    certification links, and cleared exams into an opportunity score.
    """
    paths = _mk_paths()
    trainers = _mk_read_excel(paths["trainer"], "Trainers")
    courses = _mk_read_excel(paths["course"], "Courses")
    tcm = _mk_read_excel(paths["course"], "Trainer_Course_Map")
    certs = _mk_read_excel(paths["cert"], "Trainer_Certification_Map")
    courses, tcm = _mk_dedupe_courses_and_map_for_view(courses, tcm)
    profiles = _mk_profile_skill_map()

    if trainers.empty or courses.empty:
        print("Trainer/course models are missing. Run menu option 1 first.")
        return pd.DataFrame()

    if popular_df is None or popular_df.empty:
        print("RMS popular-course page was not available. Using internal delivered-course frequency as demand proxy.")
        demand = tcm.groupby("CourseID").size().reset_index(name="RMSPopularityRaw") if not tcm.empty else pd.DataFrame(columns=["CourseID", "RMSPopularityRaw"])
        popular_df = courses.merge(demand, on="CourseID", how="left")
        popular_df["RMSPopularityRaw"] = popular_df["RMSPopularityRaw"].fillna(1.0)
        popular_df["CourseCode"] = popular_df["CourseName"].apply(_mk_course_code)
        total = max(popular_df["RMSPopularityRaw"].sum(), 1.0)
        popular_df["PopularityRatio"] = popular_df["RMSPopularityRaw"] / total
        popular_df["DemandScore"] = (popular_df["RMSPopularityRaw"] / max(popular_df["RMSPopularityRaw"].max(), 1.0)).clip(0, 1)

    model_courses = courses.copy()
    model_courses["CourseCode"] = model_courses["CourseName"].apply(_mk_course_code)
    model_courses["CourseNameNorm"] = model_courses["CourseName"].apply(_mk_norm)

    pop = popular_df.copy()
    pop["CourseCode"] = pop.get("CourseCode", pop["CourseName"].apply(_mk_course_code)).fillna("").astype(str)
    pop["CourseNameNorm"] = pop["CourseName"].apply(_mk_norm)

    # Avoid many-to-many matches where CourseCode is blank. Exact code wins, exact normalized name is fallback.
    code_lookup = model_courses[model_courses["CourseCode"].astype(str).str.len() > 0].drop_duplicates("CourseCode").set_index("CourseCode")
    name_lookup = model_courses.drop_duplicates("CourseNameNorm").set_index("CourseNameNorm")
    merged_rows = []
    for _, pop_row in pop.iterrows():
        row = pop_row.to_dict()
        match = None
        code = str(pop_row.get("CourseCode", "") or "")
        name_key = pop_row.get("CourseNameNorm", "")
        if code and code in code_lookup.index:
            match = code_lookup.loc[code]
        elif name_key in name_lookup.index:
            match = name_lookup.loc[name_key]
        if match is not None:
            for col, value in match.to_dict().items():
                if col in row:
                    row[f"{col}_Model"] = value
                else:
                    row[col] = value
        merged_rows.append(row)
    merged = pd.DataFrame(merged_rows)

    if "CourseID" not in merged.columns:
        merged["CourseID"] = np.nan

    coverage = tcm.groupby("CourseID")["TrainerID"].nunique().reset_index(name="MappedTrainerCount") if not tcm.empty else pd.DataFrame(columns=["CourseID", "MappedTrainerCount"])
    merged = merged.merge(coverage, on="CourseID", how="left")
    total_trainers = max(len(trainers), 1)
    merged["MappedTrainerCount"] = merged["MappedTrainerCount"].fillna(0)
    merged["TrainerCourseRatio"] = merged["MappedTrainerCount"] / total_trainers

    rows = []
    for _, profile in profiles.iterrows():
        trainer_id = profile.get("TrainerID")
        trainer_name = profile.get("TrainerName")
        current_skills = profile.get("CurrentSkills", [])
        cleared = set(profile.get("ClearedCertifications", []) or [])
        pending = set(profile.get("PendingCertifications", []) or [])

        for _, course in merged.iterrows():
            course_id = course.get("CourseID", "")
            course_name = course.get("CourseName_Model") or course.get("CourseName") or ""
            course_code = course.get("CourseCode", "")
            exam_url = course.get("ExamURL", "")
            cert_vendor = course.get("CertificationVendor", "")
            cert_status = course.get("CertificationStatus", "")
            skill_score, missing_skills = _mk_skill_match(current_skills, course_name, course_code)
            cert_bonus = 1.0 if course_code and course_code in cleared else (0.45 if course_code and course_code in pending else 0.0)
            demand_score = _mk_safe_number(course.get("DemandScore"), 0.5)
            coverage_gap = 1.0 - min(_mk_safe_number(course.get("TrainerCourseRatio"), 0.0), 1.0)
            mapped_cert = 1.0 if str(exam_url).startswith(("http://", "https://")) else 0.0
            opportunity = (
                demand_score * 0.34
                + skill_score * 0.28
                + coverage_gap * 0.18
                + mapped_cert * 0.12
                + cert_bonus * 0.08
            )
            next_action = "Complete certification" if mapped_cert and cert_bonus < 1 else "Build missing skills"
            if cert_bonus >= 1:
                next_action = "Use as assignment strength"
            rows.append({
                "TrainerID": trainer_id,
                "TrainerName": trainer_name,
                "CourseID": course_id,
                "CourseName": course_name,
                "CourseCode": course_code,
                "DemandScore": round(demand_score, 3),
                "PopularityRatio": round(_mk_safe_number(course.get("PopularityRatio"), 0.0), 4),
                "MappedTrainerCount": int(_mk_safe_number(course.get("MappedTrainerCount"), 0)),
                "TrainerCourseRatio": round(_mk_safe_number(course.get("TrainerCourseRatio"), 0.0), 3),
                "SkillMatchScore": skill_score,
                "MissingSkills": ", ".join(missing_skills),
                "CertificationVendor": cert_vendor,
                "CertificationStatus": cert_status,
                "ExamURL": exam_url,
                "CertificationReadiness": "Cleared" if cert_bonus >= 1 else ("Pending" if cert_bonus > 0 else "Not Started"),
                "OpportunityScore": round(opportunity, 3),
                "RecommendedNextAction": next_action,
            })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["OpportunityScore", "DemandScore", "TrainerName"], ascending=[False, False, True]).reset_index(drop=True)


def rms_popularity_opportunity_console(driver=None, export: bool = True, top_n: int = 25) -> pd.DataFrame:
    popular = fetch_rms_popular_courses_with_driver(driver) if driver is not None else pd.DataFrame()
    analysis = build_rms_popularity_opportunity_analysis(popular)
    if analysis.empty:
        return analysis
    print("RMS Popularity + Trainer Skill Opportunity Analysis")
    print("This ranks courses by RMS demand, trainer coverage ratio, certification mapping, and each trainer's saved skills.")
    display_cols = [
        "TrainerName", "CourseName", "CourseCode", "DemandScore", "TrainerCourseRatio",
        "SkillMatchScore", "CertificationReadiness", "OpportunityScore", "RecommendedNextAction", "MissingSkills", "ExamURL"
    ]
    display(analysis[display_cols].head(top_n))
    if export:
        out = _mk_paths()["output"] / "RMS_Opportunity_Latest.xlsx"
        try:
            analysis.to_excel(out, index=False)
            print(f"Exported RMS opportunity analysis: {out}")
        except Exception as exc:
            print(f"Export failed. Close the Excel file if it is open, then rerun: {out} ({exc})")
    return analysis


def fetch_autotall_dashboard_with_driver(driver=None) -> List[pd.DataFrame]:
    """
    Parse AutoTall allocation dashboard tables after RMS login.
    """
    return _mk_read_html_tables_from_driver(driver, AUTOTALL_DASHBOARD_URL)


def analyze_autotall_allocations(tables: List[pd.DataFrame]) -> pd.DataFrame:
    """
    Generic allocation analyzer. It detects trainer/course/segment-like columns from
    the AutoTall page so the notebook keeps working even if RMS changes table headers.
    """
    if not tables:
        print("AutoTall tables are not available. Open this from an authenticated RMS browser session.")
        return pd.DataFrame()

    candidates = []
    for table in tables:
        t = table.copy()
        t.columns = [str(c).strip() for c in t.columns]
        trainer_col = _mk_first_existing(t.columns, ["Trainer", "Faculty", "Instructor", "Trainer Name"])
        course_col = _mk_first_existing(t.columns, ["Course", "Course Name", "Title", "Technology"])
        segment_col = _mk_first_existing(t.columns, ["Segment", "Criteria", "Bucket", "Category", "Status", "Allocation"])
        if trainer_col or course_col:
            candidates.append((t, trainer_col, course_col, segment_col))

    if not candidates:
        print("AutoTall page was read, but no trainer/course allocation table could be identified.")
        return pd.DataFrame()

    frames = []
    for t, trainer_col, course_col, segment_col in candidates:
        f = pd.DataFrame()
        f["TrainerName"] = t[trainer_col].astype(str) if trainer_col else ""
        f["CourseName"] = t[course_col].astype(str) if course_col else ""
        f["SegmentOrCriteria"] = t[segment_col].astype(str) if segment_col else ""
        f["SourceRows"] = 1
        frames.append(f)
    raw = pd.concat(frames, ignore_index=True)
    raw = raw[(raw["TrainerName"].str.strip() != "") | (raw["CourseName"].str.strip() != "")]

    paths = _mk_paths()
    trainers = _mk_read_excel(paths["trainer"], "Trainers")
    my_names = set(trainers.get("TrainerName", pd.Series(dtype=str)).dropna().astype(str).map(_mk_norm).tolist())
    raw["IsMyReportee"] = raw["TrainerName"].map(lambda x: _mk_norm(x) in my_names)

    summary = raw.groupby(["CourseName", "SegmentOrCriteria"], dropna=False).agg(
        TotalAllocations=("SourceRows", "sum"),
        UniqueTrainers=("TrainerName", "nunique"),
        MyReporteeAllocations=("IsMyReportee", "sum"),
    ).reset_index()
    summary["MyReporteeShare"] = (summary["MyReporteeAllocations"] / summary["TotalAllocations"].replace(0, np.nan)).fillna(0).round(3)
    summary["OpportunitySignal"] = np.where(
        summary["MyReporteeShare"] == 0,
        "Open opportunity: no current reportee allocation",
        np.where(summary["MyReporteeShare"] < 0.5, "Improve share", "Current strength")
    )
    return summary.sort_values(["MyReporteeShare", "TotalAllocations"], ascending=[True, False]).reset_index(drop=True)


def autotall_allocation_console(driver=None, export: bool = True, top_n: int = 30) -> pd.DataFrame:
    tables = fetch_autotall_dashboard_with_driver(driver)
    analysis = analyze_autotall_allocations(tables)
    if analysis.empty:
        return analysis
    print("AutoTall Allocation Intelligence")
    print("This shows where your reportees already appear versus courses/segments where opportunity is open.")
    display(analysis.head(top_n))
    if export:
        out = _mk_paths()["output"] / "AutoTall_Allocation_Latest.xlsx"
        try:
            analysis.to_excel(out, index=False)
            print(f"Exported AutoTall allocation analysis: {out}")
        except Exception as exc:
            print(f"Export failed. Close the Excel file if it is open, then rerun: {out} ({exc})")
    return analysis


def _mk_tables_from_html(html: str) -> List[pd.DataFrame]:
    """
    Parse HTML tables without requiring lxml/html5lib. Keeps RMS scraping working in plain Anaconda.
    """
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception:
        return []
    tables = []
    for table in soup.find_all("table"):
        rows = []
        for tr in table.find_all("tr"):
            cells = tr.find_all(["th", "td"])
            values = [cell.get_text(" ", strip=True) for cell in cells]
            if values:
                rows.append(values)
        if not rows:
            continue
        width = max(len(r) for r in rows)
        rows = [r + [""] * (width - len(r)) for r in rows]
        header = rows[0]
        body = rows[1:] if len(rows) > 1 else []
        if not body:
            continue
        if len(set(header)) != len(header) or all(str(h).strip() == "" for h in header):
            header = [f"Column{i+1}" for i in range(width)]
            body = rows
        df = pd.DataFrame(body, columns=[str(h).strip() or f"Column{i+1}" for i, h in enumerate(header)])
        if not df.empty:
            tables.append(df)
    return tables


def _mk_display_html(html_text: str) -> None:
    try:
        display(HTML(html_text))
    except Exception:
        print(re.sub("<[^>]+>", " ", html_text))


def _mk_short(value: Any, max_len: int = 110) -> str:
    text = "" if value is None or (isinstance(value, float) and pd.isna(value)) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) <= max_len else text[: max_len - 3] + "..."



def _mk_profile_text(profile: pd.Series) -> str:
    parts = []
    for col in ["CurrentSkills", "PastCourses", "ClearedCertifications", "PendingCertifications"]:
        value = profile.get(col, [])
        if isinstance(value, list):
            parts.extend(map(str, value))
        else:
            parts.append(str(value or ""))
    return " ".join(parts).upper()


def _mk_cleared_codes(profile: pd.Series) -> set[str]:
    cleared = profile.get("ClearedCertifications", []) or []
    if not isinstance(cleared, list):
        cleared = [x.strip() for x in str(cleared).split(",") if x.strip()]
    return {str(x).upper().strip() for x in cleared}


def _mk_next_exam_target(profile: pd.Series, upgrade_code: Any, upgrade_name: Any) -> Dict[str, str]:
    """
    Convert a future-skill recommendation into a concrete 45-day cert/proof target.
    Rules are intentionally manager-friendly and vendor-aware.
    """
    text = _mk_profile_text(profile) + " " + str(upgrade_code or "").upper() + " " + str(upgrade_name or "").upper()
    cleared = _mk_cleared_codes(profile)

    # AI-102 / AI-900 are retiring on 2026-06-30, so they should not be the default future-ready target.
    if any(term in text for term in ["AI-3025", "AI AGENT", "AZURE AI FOUNDRY", "RAG", "PROMPT"]):
        if "DP-700" not in cleared and any(term in text for term in ["FABRIC", "DP-700", "LAKEHOUSE", "DATA ENGINEER", "SQL"]):
            return {
                "TargetType": "Certification + proof",
                "ExamTarget": "DP-700",
                "ExamTargetName": "Microsoft Certified: Fabric Data Engineer Associate",
                "Exam45DayTarget": "Prepare and schedule DP-700 within 45 days",
                "ProofPack": "AI agent demo using Fabric/Lakehouse data, RAG, tool calling, evaluation notes",
                "CertificationMove": "Use DP-700 as formal cert target; use AI-3025 as proof/lab specialization.",
                "ManagerReviewCadence": "Week 1 skill plan, Week 2 lab checkpoint, Week 4 mock/practice score, Week 6 exam/proof upload.",
            }
        return {
            "TargetType": "Proof pack",
            "ExamTarget": "AI-3025 proof pack",
            "ExamTargetName": "Work with AI agents proof portfolio",
            "Exam45DayTarget": "Upload AI agent proof pack within 45 days",
            "ProofPack": "Azure AI Foundry agent, prompt patterns, RAG workflow, tool calling, evaluation screenshot, delivery notes",
            "CertificationMove": "No stable formal AI-3025 exam target; build proof and attach RMS evidence.",
            "ManagerReviewCadence": "Week 1 design, Week 2 working demo, Week 4 evaluation, Week 6 RMS proof upload.",
        }

    if "DP-700" not in cleared and any(term in text for term in ["FABRIC", "DP-700", "LAKEHOUSE", "DATA ENGINEER"]):
        return {
            "TargetType": "Certification",
            "ExamTarget": "DP-700",
            "ExamTargetName": "Microsoft Certified: Fabric Data Engineer Associate",
            "Exam45DayTarget": "Prepare and schedule DP-700 within 45 days",
            "ProofPack": "Fabric pipeline/lakehouse lab, PySpark or SQL transformation, monitoring notes",
            "CertificationMove": "Formal certification target: DP-700.",
            "ManagerReviewCadence": "Weekly DP-700 topic checkpoint and practice assessment before scheduling.",
        }

    if "DP-600" not in cleared and any(term in text for term in ["DP-600", "FABRIC ANALYTICS", "SEMANTIC", "POWER BI", "DAX"]):
        return {
            "TargetType": "Certification",
            "ExamTarget": "DP-600",
            "ExamTargetName": "Microsoft Certified: Fabric Analytics Engineer Associate",
            "Exam45DayTarget": "Prepare and schedule DP-600 within 45 days",
            "ProofPack": "Semantic model, DAX measure set, Fabric/Power BI governance notes",
            "CertificationMove": "Formal certification target: DP-600.",
            "ManagerReviewCadence": "Weekly DP-600 topic checkpoint and model/demo review.",
        }

    if "PL-300" not in cleared and any(term in text for term in ["PL-300", "POWER BI", "DAX", "DASHBOARD"]):
        return {
            "TargetType": "Certification",
            "ExamTarget": "PL-300",
            "ExamTargetName": "Microsoft Certified: Power BI Data Analyst Associate",
            "Exam45DayTarget": "Prepare and schedule PL-300 within 45 days",
            "ProofPack": "Power BI dashboard, data model, DAX measures, refresh/demo notes",
            "CertificationMove": "Formal certification target: PL-300.",
            "ManagerReviewCadence": "Weekly PL-300 practice and dashboard evidence review.",
        }

    if "DP-900" not in cleared and any(term in text for term in ["DP-900", "AZURE DATA", "DATA FUNDAMENTALS"]):
        return {
            "TargetType": "Certification",
            "ExamTarget": "DP-900",
            "ExamTargetName": "Microsoft Certified: Azure Data Fundamentals",
            "Exam45DayTarget": "Prepare and schedule DP-900 within 45 days",
            "ProofPack": "Azure data concepts notes and short RMS evidence update",
            "CertificationMove": "Formal foundation certification target: DP-900.",
            "ManagerReviewCadence": "Two-week DP-900 prep, then schedule exam.",
        }

    return {
        "TargetType": "Proof pack",
        "ExamTarget": str(upgrade_code or upgrade_name or "Proof pack"),
        "ExamTargetName": str(upgrade_name or upgrade_code or "Role proof portfolio"),
        "Exam45DayTarget": "Upload role proof pack within 45 days",
        "ProofPack": "Lab/demo, screenshots, outline, delivery notes, and RMS skill evidence",
        "CertificationMove": "No trusted formal exam target found; use proof pack and vendor link review.",
        "ManagerReviewCadence": "Weekly proof checkpoint until RMS evidence is complete.",
    }




def build_manager_action_plan(
    profiles: pd.DataFrame,
    recommendations: pd.DataFrame,
    weekly_actions: pd.DataFrame,
    opportunity: pd.DataFrame | None = None,
    allocation: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Convert detailed intelligence into manager-friendly decisions.
    """
    rows = []
    opportunity = opportunity if opportunity is not None else pd.DataFrame()
    allocation = allocation if allocation is not None else pd.DataFrame()

    for _, profile in profiles.iterrows():
        trainer = profile.get("TrainerName", "")
        rec_rows = recommendations[recommendations["TrainerName"].astype(str) == str(trainer)] if not recommendations.empty and "TrainerName" in recommendations.columns else pd.DataFrame()
        opp_rows = opportunity[opportunity["TrainerName"].astype(str) == str(trainer)] if not opportunity.empty and "TrainerName" in opportunity.columns else pd.DataFrame()
        action_rows = weekly_actions[weekly_actions["TrainerName"].astype(str) == str(trainer)] if not weekly_actions.empty and "TrainerName" in weekly_actions.columns else pd.DataFrame()

        # Upgrade decision should come from the next-skill roadmap, not from courses already delivered.
        # RMS popularity and AutoTall are used later to shape the allotment play.
        if not rec_rows.empty:
            top = rec_rows.sort_values("AssignmentProbability", ascending=False).iloc[0]
            upgrade_code = top.get("RecommendedCourse", "")
            upgrade_name = top.get("CourseName", "")
            skill_match_label = top.get("SkillMatchScore", top.get("MatchScore", top.get("SkillMatch", "")))
            why = (
                f"Best next upgrade from current skills. Assignment readiness {top.get('AssignmentProbability', '')}; "
                f"skill match {skill_match_label}."
            )
            missing = ", ".join(top.get("MissingSkills", [])) if isinstance(top.get("MissingSkills"), list) else top.get("MissingSkills", "")
            if top.get("RecommendedCourse") in ["AI-3025", "AI-3016", "GH-300"]:
                cert_step = "Build lab/demo proof and update RMS skills; certification may not be mandatory"
            else:
                cert_step = "Close certification/skill evidence gap"
        elif not opp_rows.empty:
            top = opp_rows.sort_values(["OpportunityScore", "DemandScore"], ascending=[False, False]).iloc[0]
            upgrade_code = top.get("CourseCode") or top.get("CourseName", "")
            upgrade_name = top.get("CourseName", "")
            why = (
                f"Demand {top.get('DemandScore', '')}, skill match {top.get('SkillMatchScore', '')}, "
                f"trainer coverage ratio {top.get('TrainerCourseRatio', '')}."
            )
            missing = top.get("MissingSkills", "")
            cert_step = top.get("RecommendedNextAction", "")
        else:
            upgrade_code = ""
            upgrade_name = "No recommendation available"
            why = "Refresh data first."
            missing = ""
            cert_step = "Refresh RMS data"

        current_skills = profile.get("CurrentSkills", [])
        past_courses = profile.get("PastCourses", [])
        cleared = profile.get("ClearedCertifications", [])
        pending = profile.get("PendingCertifications", [])
        skill_count = len(current_skills) if isinstance(current_skills, list) else len(_mk_short(current_skills).split(","))
        course_count = len(past_courses) if isinstance(past_courses, list) else len(_mk_short(past_courses).split(","))
        cleared_count = len(cleared) if isinstance(cleared, list) else (0 if not cleared else len(str(cleared).split(",")))
        pending_count = len(pending) if isinstance(pending, list) else (0 if not pending else len(str(pending).split(",")))

        if cleared_count > 0:
            readiness = "Ready to position now"
        elif pending_count > 0:
            readiness = "Push pending exam to cleared proof"
        else:
            readiness = "Needs certification proof"

        exam_plan = _mk_next_exam_target(profile, upgrade_code, upgrade_name)
        exam_target = exam_plan["Exam45DayTarget"]
        cert_step = exam_plan["CertificationMove"]
        proof_pack = exam_plan["ProofPack"]
        manager_cadence = exam_plan["ManagerReviewCadence"]

        manager_action = action_rows.iloc[0].get("NextActions", "") if not action_rows.empty else ""
        if not manager_action:
            manager_action = f"Assign learning plan for {_mk_short(upgrade_code or upgrade_name, 40)} and update RMS evidence."
        manager_action = f"{exam_target} | Build proof pack: {proof_pack} | {manager_cadence}"

        allotment_play = "After proof upload, position trainer for future-ready AI + data delivery."
        if str(upgrade_code).upper() in ["AI-3025", "AI-3016"] or "AI" in str(upgrade_name).upper():
            allotment_play = (
                f"After {exam_plan['ExamTarget']} / proof pack, position for AI agents + Fabric/Power BI modernization deliveries; "
                "do not use legacy SQL-only courses as the main upgrade target."
            )
        elif not allocation.empty and {"CourseName", "OpportunitySignal"}.issubset(allocation.columns):
            open_alloc = allocation[allocation["OpportunitySignal"].astype(str).str.contains("Open|Improve", case=False, na=False)]
            if not open_alloc.empty:
                allotment_play = "Check AutoTall gaps: " + _mk_short(open_alloc.iloc[0].get("CourseName", ""), 70)
        elif not opp_rows.empty:
            low_coverage = opp_rows.sort_values(["TrainerCourseRatio", "DemandScore"], ascending=[True, False]).head(1)
            if not low_coverage.empty:
                allotment_play = "Target low-coverage demand course: " + _mk_short(low_coverage.iloc[0].get("CourseName", ""), 70)

        rows.append({
            "TrainerName": trainer,
            "ManagerDecision": readiness,
            "UpgradeNext": _mk_short(str(upgrade_code).strip() or upgrade_name, 80),
            "UpgradeCourseName": _mk_short(upgrade_name, 120),
            "WhyThisMatters": _mk_short(why, 180),
            "SkillGapToClose": _mk_short(missing or "No major skill gap detected; update evidence and certification proof.", 180),
            "CertificationMove": _mk_short(cert_step, 180),
            "Exam45DayTarget": _mk_short(exam_target, 140),
            "ConcreteExamTarget": _mk_short(exam_plan["ExamTarget"], 80),
            "ExamTargetName": _mk_short(exam_plan["ExamTargetName"], 140),
            "ProofPack": _mk_short(proof_pack, 220),
            "ManagerReviewCadence": _mk_short(manager_cadence, 180),
            "AllotmentStrategy": _mk_short(allotment_play, 220),
            "ThisWeekManagerAction": _mk_short(manager_action, 320),
            "SkillCount": skill_count,
            "PastCourseCount": course_count,
            "ClearedCertCount": cleared_count,
            "PendingCertCount": pending_count,
        })
    return pd.DataFrame(rows)





def _mk_is_formal_exam_course(course_code: Any, course_name: Any, cert_step: Any = "") -> bool:
    text = f"{course_code} {course_name} {cert_step}".upper()
    if any(vendor in text for vendor in ["MICROSOFT", "MONGODB", "ALTERYX", "DBT", "AWS", "GOOGLE", "CISCO"]):
        return True
    return bool(re.search(r"\b[A-Z]{1,8}-\d{2,5}\b", text))


def build_team_strategy_summary(profiles: pd.DataFrame, action_plan: pd.DataFrame) -> pd.DataFrame:
    """
    Manager view: common course/skill clusters across reportees.
    Helps decide what can be pushed as a team learning sprint.
    """
    cluster_rows = []
    for _, profile in profiles.iterrows():
        trainer = profile.get("TrainerName", "")
        for course in profile.get("PastCourses", []) or []:
            cluster_rows.append({"Cluster": _mk_short(course, 90), "TrainerName": trainer, "Signal": "Common delivered area"})

    if cluster_rows:
        clusters = pd.DataFrame(cluster_rows)
        common = (
            clusters.groupby("Cluster")
            .agg(TrainerCount=("TrainerName", "nunique"), Trainers=("TrainerName", lambda s: ", ".join(sorted(set(map(str, s))))))
            .reset_index()
            .sort_values(["TrainerCount", "Cluster"], ascending=[False, True])
        )
        common = common[common["TrainerCount"] > 1].head(8)
    else:
        common = pd.DataFrame(columns=["Cluster", "TrainerCount", "Trainers"])

    upgrade = pd.DataFrame()
    if not action_plan.empty and "UpgradeNext" in action_plan.columns:
        upgrade = (
            action_plan.groupby(["UpgradeNext", "UpgradeCourseName"], dropna=False)
            .agg(TrainerCount=("TrainerName", "nunique"), Trainers=("TrainerName", lambda s: ", ".join(sorted(set(map(str, s))))))
            .reset_index()
            .sort_values(["TrainerCount", "UpgradeNext"], ascending=[False, True])
            .head(8)
            .rename(columns={"UpgradeNext": "Cluster", "UpgradeCourseName": "CourseName"})
        )
        upgrade["Signal"] = "Team upgrade target"

    rows = []
    for _, row in upgrade.iterrows():
        rows.append({
            "Priority": "Team upgrade",
            "Focus": row.get("Cluster", ""),
            "CourseName": row.get("CourseName", ""),
            "TrainerCount": row.get("TrainerCount", 0),
            "Trainers": row.get("Trainers", ""),
            "ManagerMove": "Run as one cohort: assign 45-day proof/exam target and update RMS skills together.",
        })
    for _, row in common.iterrows():
        rows.append({
            "Priority": "Common strength",
            "Focus": row.get("Cluster", ""),
            "CourseName": row.get("Cluster", ""),
            "TrainerCount": row.get("TrainerCount", 0),
            "Trainers": row.get("Trainers", ""),
            "ManagerMove": "Use this as shared delivery strength; identify next-level cert/course for the same cluster.",
        })
    return pd.DataFrame(rows)


def display_team_strategy_board(team_strategy: pd.DataFrame) -> pd.DataFrame:
    if team_strategy.empty:
        print("No common team strategy signal found yet.")
        return team_strategy
    rows = team_strategy.fillna("").astype(str).to_dict("records")
    payload_json = json.dumps({"rows": rows}, ensure_ascii=False).replace("</", "<\\/")
    html = f"""
    <div id="teamStrategyBoard" style="font-family:Segoe UI, Arial, sans-serif; color:#1f2937; margin-top:16px;">
      <style>
        #teamStrategyBoard .item {{border:1px solid #d1d5db; border-radius:8px; padding:10px; margin-bottom:8px; background:#fff; max-width:1100px;}}
        #teamStrategyBoard .tag {{display:inline-block; border:1px solid #bfdbfe; background:#eff6ff; color:#1d4ed8; border-radius:999px; padding:2px 8px; font-size:12px; margin-bottom:5px;}}
        #teamStrategyBoard .title {{font-weight:700; font-size:15px;}}
        #teamStrategyBoard .muted {{font-size:12px; color:#64748b;}}
      </style>
      <h3 style="margin:4px 0 2px;">Team Strategy Board</h3>
      <div class="muted">Use this for common courses and cohort-based upgrades when reportees increase.</div>
      <div id="teamStrategyItems"></div>
    </div>
    <script>
    (function() {{
      const data = {payload_json};
      const root = document.getElementById('teamStrategyBoard');
      const box = root.querySelector('#teamStrategyItems');
      function esc(v) {{ return String(v || '').replace(/[&<>"']/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[ch])); }}
      box.innerHTML = data.rows.map(r => `<div class="item">
        <div class="tag">${{esc(r.Priority)}} · ${{esc(r.TrainerCount)}} trainer(s)</div>
        <div class="title">${{esc(r.Focus)}}</div>
        <div>${{esc(r.CourseName)}}</div>
        <div class="muted">Trainers: ${{esc(r.Trainers)}}</div>
        <div style="margin-top:5px;">${{esc(r.ManagerMove)}}</div>
      </div>`).join('');
    }})();
    </script>
    """
    _mk_display_html(html)
    return team_strategy








def _v2_paths() -> Dict[str, Path]:
    base = _mk_paths()["output"]
    return {
        "history": base / "Trainer_Intelligence_History.xlsx",
        "opportunity_matrix": base / "Course_Opportunity_Matrix_Latest.xlsx",
    }


def append_refresh_snapshot(
    opportunity: pd.DataFrame | None = None,
    allocation: pd.DataFrame | None = None,
    note: str = "option1_refresh",
) -> Path:
    """
    Store a lightweight historical snapshot on every option 1 refresh.
    This powers trend/prediction without needing live RMS during option 3.
    """
    paths = _v2_paths()
    history_file = paths["history"]
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        courses = _mk_read_excel(_mk_paths()["course"], "Courses")
        tcm = _mk_read_excel(_mk_paths()["course"], "Trainer_Course_Map")
    except Exception:
        courses, tcm = pd.DataFrame(), pd.DataFrame()

    if opportunity is None or opportunity.empty:
        try:
            opportunity = build_rms_popularity_opportunity_analysis(pd.DataFrame())
        except Exception:
            opportunity = pd.DataFrame()
    allocation = allocation if allocation is not None else pd.DataFrame()

    course_snapshot = pd.DataFrame()
    if not courses.empty:
        demand = tcm.groupby("CourseID").size().reset_index(name="TrainerCoverageCount") if not tcm.empty and "CourseID" in tcm.columns else pd.DataFrame(columns=["CourseID", "TrainerCoverageCount"])
        course_snapshot = courses.merge(demand, on="CourseID", how="left")
        course_snapshot["TrainerCoverageCount"] = course_snapshot["TrainerCoverageCount"].fillna(0)
        course_snapshot["SnapshotTime"] = stamp
        course_snapshot["SourceNote"] = note
        keep = [c for c in ["SnapshotTime", "SourceNote", "CourseID", "CourseName", "ExamURL", "CertificationVendor", "CertificationStatus", "TrainerCoverageCount"] if c in course_snapshot.columns]
        course_snapshot = course_snapshot[keep]

    opportunity_snapshot = pd.DataFrame()
    if opportunity is not None and not opportunity.empty:
        opportunity_snapshot = opportunity.copy()
        opportunity_snapshot["SnapshotTime"] = stamp
        opportunity_snapshot["SourceNote"] = note
        keep = [c for c in ["SnapshotTime", "SourceNote", "TrainerName", "CourseName", "CourseCode", "DemandScore", "TrainerCourseRatio", "SkillMatchScore", "OpportunityScore", "CertificationReadiness", "RecommendedNextAction"] if c in opportunity_snapshot.columns]
        opportunity_snapshot = opportunity_snapshot[keep]

    allocation_snapshot = pd.DataFrame()
    if allocation is not None and not allocation.empty:
        allocation_snapshot = allocation.copy()
        allocation_snapshot["SnapshotTime"] = stamp
        allocation_snapshot["SourceNote"] = note

    old_courses = pd.DataFrame()
    old_opp = pd.DataFrame()
    old_alloc = pd.DataFrame()
    if history_file.exists():
        try:
            old_courses = pd.read_excel(history_file, sheet_name="CourseSnapshots", engine="openpyxl")
        except Exception:
            pass
        try:
            old_opp = pd.read_excel(history_file, sheet_name="OpportunitySnapshots", engine="openpyxl")
        except Exception:
            pass
        try:
            old_alloc = pd.read_excel(history_file, sheet_name="AllocationSnapshots", engine="openpyxl")
        except Exception:
            pass

    course_out = pd.concat([old_courses, course_snapshot], ignore_index=True) if not course_snapshot.empty else old_courses
    opp_out = pd.concat([old_opp, opportunity_snapshot], ignore_index=True) if not opportunity_snapshot.empty else old_opp
    alloc_out = pd.concat([old_alloc, allocation_snapshot], ignore_index=True) if not allocation_snapshot.empty else old_alloc

    # Keep history compact enough for notebook use.
    for df_name in ["course_out", "opp_out", "alloc_out"]:
        df = locals()[df_name]
        if not df.empty and "SnapshotTime" in df.columns:
            locals()[df_name] = df.tail(5000).reset_index(drop=True)

    try:
        with pd.ExcelWriter(history_file, engine="openpyxl") as writer:
            course_out.to_excel(writer, sheet_name="CourseSnapshots", index=False)
            opp_out.to_excel(writer, sheet_name="OpportunitySnapshots", index=False)
            alloc_out.to_excel(writer, sheet_name="AllocationSnapshots", index=False)
        print(f"Refresh snapshot appended: {history_file}")
    except Exception as exc:
        print(f"Refresh snapshot could not be saved. Close file if open: {history_file} ({exc})")
    return history_file


def load_refresh_history() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    history_file = _v2_paths()["history"]
    if not history_file.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    def read(sheet):
        try:
            return pd.read_excel(history_file, sheet_name=sheet, engine="openpyxl")
        except Exception:
            return pd.DataFrame()
    return read("CourseSnapshots"), read("OpportunitySnapshots"), read("AllocationSnapshots")


def fetch_trainer_kpi_summary(driver, wait=None) -> tuple[pd.DataFrame, dict[str, object]]:
    """
    Read the Trainer KPI page and return a structured snapshot.

    The page can vary by RMS role and may expose tables, cards, or plain text.
    This parser captures any visible structured content so the manager flow
    has a real KPI artifact instead of a bare navigation step.
    """
    from datetime import datetime
    from html.parser import HTMLParser

    url = globals().get('TRAINER_KPI_URL', 'https://rms.koenig-solutions.com/BadgeProject/frmTrainerKpi.aspx')
    if driver is None:
        return pd.DataFrame(), {'source_url': url, 'captured_at': datetime.now().isoformat(timespec='seconds'), 'error': 'no-driver'}

    try:
        driver.get(url)
    except Exception as exc:
        return pd.DataFrame(), {'source_url': url, 'captured_at': datetime.now().isoformat(timespec='seconds'), 'error': str(exc)}

    try:
        if wait is not None:
            try:
                wait.until(lambda d: d.execute_script("return document.readyState") == 'complete')
            except Exception:
                pass
    except Exception:
        pass

    html = ''
    try:
        html = driver.page_source or ''
    except Exception:
        html = ''

    frames: list[pd.DataFrame] = []

    try:
        if html:
            tables = _mk_tables_from_html(html)
            for idx, tbl in enumerate(tables):
                if tbl.empty:
                    continue
                tbl = tbl.copy()
                tbl['SourceType'] = 'table'
                tbl['SourceIndex'] = idx
                tbl['SourceUrl'] = url
                frames.append(tbl)
    except Exception:
        pass

    class _TextBlockParser(HTMLParser):
        def __init__(self):
            super().__init__()
            self.blocks = []
            self._stack = []
            self._current = []
            self._tag = ''
        def handle_starttag(self, tag, attrs):
            tag = tag.lower()
            if tag in {'div','li','p','span','td','th','h1','h2','h3','h4','h5','h6'}:
                self._stack.append(tag)
                self._tag = tag
                self._current = []
        def handle_data(self, data):
            if self._stack:
                txt = ' '.join(str(data).split())
                if txt:
                    self._current.append(txt)
        def handle_endtag(self, tag):
            tag = tag.lower()
            if self._stack and self._stack[-1] == tag:
                txt = ' '.join(' '.join(self._current).split())
                if txt and len(txt) > 1:
                    self.blocks.append({'SourceType':'text','Tag':tag,'Text':txt,'SourceUrl':url})
                self._stack.pop()
                self._current = []

    try:
        if html:
            parser = _TextBlockParser()
            parser.feed(html)
            if parser.blocks:
                frames.append(pd.DataFrame(parser.blocks))
    except Exception:
        pass

    try:
        labels = driver.find_elements('xpath', "//*[self::label or self::span or self::div or self::td or self::th or self::h1 or self::h2 or self::h3 or self::h4]")
        kv = []
        for el in labels[:600]:
            try:
                txt = _extract_text_safe(el)
                if not txt or len(txt) > 180:
                    continue
                low = txt.lower()
                if any(ch.isdigit() for ch in txt) or any(k in low for k in ['kpi','score','trainer','course','cert','badge','allocation','completion','pending','cleared']):
                    kv.append({'SourceType':'dom', 'Text': txt, 'SourceUrl': url})
            except Exception:
                continue
        if kv:
            frames.append(pd.DataFrame(kv))
    except Exception:
        pass

    if frames:
        out = pd.concat([f for f in frames if not f.empty], ignore_index=True, sort=False)
    else:
        out = pd.DataFrame(columns=['SourceType','Text','SourceUrl'])

    if out.empty:
        out = pd.DataFrame([{'SourceType':'meta','Text':'No structured KPI elements detected on page','SourceUrl':url}])

    summary = {
        'source_url': url,
        'captured_at': datetime.now().isoformat(timespec='seconds'),
        'row_count': int(len(out)),
        'column_count': int(len(out.columns)),
    }
    return out, summary



def build_predictive_demand_summary(opportunity: pd.DataFrame | None = None) -> pd.DataFrame:
    """
    Estimate rising/stable/declining demand from saved snapshots.
    With limited history, falls back to current opportunity/demand score.
    """
    course_hist, opp_hist, _ = load_refresh_history()
    opportunity = opportunity if opportunity is not None else pd.DataFrame()
    rows = []

    if not opp_hist.empty and {"CourseName", "DemandScore", "SnapshotTime"}.issubset(opp_hist.columns):
        hist = opp_hist.copy()
        hist["SnapshotTime"] = pd.to_datetime(hist["SnapshotTime"], errors="coerce")
        grouped = hist.dropna(subset=["SnapshotTime"]).groupby("CourseName")
        for course, grp in grouped:
            grp = grp.sort_values("SnapshotTime")
            first = _mk_safe_number(grp["DemandScore"].iloc[0], 0)
            last = _mk_safe_number(grp["DemandScore"].iloc[-1], 0)
            delta = last - first
            rows.append({
                "CourseName": course,
                "CurrentDemandScore": round(last, 3),
                "DemandDelta": round(delta, 3),
                "Trend": "Rising" if delta > 0.05 else ("Declining" if delta < -0.05 else "Stable"),
                "Snapshots": len(grp),
            })
    elif opportunity is not None and not opportunity.empty:
        base = opportunity.groupby("CourseName", dropna=False).agg(CurrentDemandScore=("DemandScore", "max")).reset_index()
        for _, row in base.iterrows():
            score = _mk_safe_number(row.get("CurrentDemandScore"), 0)
            rows.append({
                "CourseName": row.get("CourseName", ""),
                "CurrentDemandScore": round(score, 3),
                "DemandDelta": 0,
                "Trend": "High current demand" if score >= 0.75 else ("Moderate current demand" if score >= 0.45 else "Low current demand"),
                "Snapshots": 1,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    trend_order = {"Rising": 0, "High current demand": 1, "Stable": 2, "Moderate current demand": 3, "Declining": 4, "Low current demand": 5}
    out["_rank"] = out["Trend"].map(trend_order).fillna(9)
    return out.sort_values(["_rank", "CurrentDemandScore"], ascending=[True, False]).drop(columns=["_rank"]).reset_index(drop=True)


def build_opportunity_matrix(action_plan: pd.DataFrame, opportunity: pd.DataFrame, demand_summary: pd.DataFrame) -> pd.DataFrame:
    """
    Course opportunity categories: Quick Win, Strategic Upgrade, Watch, Low ROI.
    """
    if opportunity is None:
        opportunity = pd.DataFrame()
    rows = []
    if opportunity.empty:
        return pd.DataFrame(columns=["TrainerName", "CourseName", "Category", "Reason", "ManagerMove"])

    demand_lookup = {}
    if demand_summary is not None and not demand_summary.empty and "CourseName" in demand_summary.columns:
        demand_lookup = demand_summary.set_index("CourseName").to_dict("index")

    for _, row in opportunity.iterrows():
        trainer = row.get("TrainerName", "")
        course = row.get("CourseName", "")
        skill = _mk_safe_number(row.get("SkillMatchScore"), 0)
        demand = _mk_safe_number(row.get("DemandScore"), 0)
        cert = str(row.get("CertificationReadiness", ""))
        trend = demand_lookup.get(course, {}).get("Trend", "")
        coverage = _mk_safe_number(row.get("TrainerCourseRatio"), 0)

        if skill >= 0.7 and demand >= 0.65:
            category = "Quick Win"
            move = "Update RMS evidence and push for allocation now."
        elif demand >= 0.65 or trend in ["Rising", "High current demand"]:
            category = "Strategic Upgrade"
            move = "Put into 45/90-day growth path and track proof/cert completion."
        elif coverage >= 0.8 and demand < 0.5:
            category = "Low ROI"
            move = "Do not prioritize unless client demand appears."
        else:
            category = "Watch"
            move = "Keep as secondary target; revisit after next refresh."

        rows.append({
            "TrainerName": trainer,
            "CourseName": course,
            "Category": category,
            "DemandScore": round(demand, 3),
            "SkillMatchScore": round(skill, 3),
            "Trend": trend,
            "CertificationReadiness": cert,
            "Reason": f"demand={round(demand, 2)}, skill={round(skill, 2)}, trend={trend or 'n/a'}",
            "ManagerMove": move,
        })
    return pd.DataFrame(rows).sort_values(["Category", "DemandScore", "SkillMatchScore"], ascending=[True, False, False]).reset_index(drop=True)




def _mk_infer_required_skills(course_code: Any, course_name: Any, fallback_missing: Any = "") -> list[str]:
    """
    Infer course skills from the maintained catalog first, then keyword rules.
    This lets RMS courses, Microsoft Learn courses, MongoDB/Alteryx/dbt style courses,
    and future reportee data participate in the same simulator.
    """
    code = str(course_code or "").upper().strip()
    name = str(course_name or "")
    text = f"{code} {name}".upper()
    try:
        if "TRAINER_INTELLIGENCE_CATALOG" in globals() and not TRAINER_INTELLIGENCE_CATALOG.empty:
            catalog = TRAINER_INTELLIGENCE_CATALOG.copy()
            for _, row in catalog.iterrows():
                row_code = str(row.get("CourseCode", "")).upper()
                row_name = str(row.get("CourseName", "")).upper()
                if row_code and row_code in text:
                    return list(row.get("Skills", []) or [])
                if row_name and row_name in text:
                    return list(row.get("Skills", []) or [])
    except Exception:
        pass

    skills = []
    rules = [
        (["POWER BI", "PL-300", "DASHBOARD", "DAX"], ["Power BI", "DAX", "Visualization", "Data Modeling", "Power Query"]),
        (["FABRIC", "DP-700", "DP-600", "LAKEHOUSE"], ["Microsoft Fabric", "Lakehouse", "Pipelines", "SQL", "Data Modeling"]),
        (["SQL", "TRANSact", "T-SQL", "QUERYING DATA"], ["SQL", "T-SQL", "Data Modeling", "Query Optimization"]),
        (["AI", "AI-3025", "AI AGENT", "AZURE AI", "OPENAI"], ["Azure AI", "Prompt Engineering", "RAG", "Responsible AI", "Tool Calling"]),
        (["PYTHON"], ["Python", "Data Processing", "Automation"]),
        (["MONGODB"], ["MongoDB", "NoSQL", "Aggregation", "Indexing"]),
        (["ALTERYX"], ["Alteryx", "Data Preparation", "Workflow Automation", "Analytics"]),
        (["DBT"], ["dbt", "SQL", "Data Modeling", "Analytics Engineering"]),
        (["AWS"], ["AWS", "Cloud", "IAM", "Networking"]),
        (["AZURE", "AZ-"], ["Azure", "Cloud", "Identity", "Networking"]),
    ]
    for triggers, inferred in rules:
        if any(t.upper() in text for t in triggers):
            skills.extend(inferred)
    if not skills and fallback_missing:
        if isinstance(fallback_missing, list):
            skills.extend(map(str, fallback_missing))
        else:
            skills.extend([x.strip() for x in str(fallback_missing).split(",") if x.strip()])
    return list(dict.fromkeys([x for x in skills if str(x).strip()]))


def _mk_workload_stats() -> dict[str, dict[str, float]]:
    """Build workload proxy from current Trainer_Course_Map until RMS delivery-date history is captured."""
    stats: dict[str, dict[str, float]] = {}
    try:
        trainers = _mk_read_excel(_mk_paths()["trainer"], "Trainers")
        tcm = _mk_read_excel(_mk_paths()["course"], "Trainer_Course_Map")
        if trainers.empty:
            return stats
        if tcm.empty:
            for _, tr in trainers.iterrows():
                stats[str(tr.get("TrainerName", ""))] = {"RecentCourses": 0, "AssignmentLoad": 0, "RecencyScore": 1, "OverloadPenalty": 0}
            return stats
        tcm = tcm.copy()
        tcm["AssignmentsDelivered"] = pd.to_numeric(tcm.get("AssignmentsDelivered", 0), errors="coerce").fillna(0)
        raw = tcm.groupby("TrainerID").agg(RecentCourses=("CourseID", "count"), AssignmentLoad=("AssignmentsDelivered", "sum")).reset_index()
        max_courses = max(float(raw["RecentCourses"].max() or 1), 1)
        max_load = max(float(raw["AssignmentLoad"].max() or 1), 1)
        merged = trainers.merge(raw, on="TrainerID", how="left").fillna({"RecentCourses": 0, "AssignmentLoad": 0})
        for _, row in merged.iterrows():
            load_ratio = max(float(row.get("RecentCourses", 0)) / max_courses, float(row.get("AssignmentLoad", 0)) / max_load)
            # RMS model currently has workload but not delivery dates. Keep recency neutral
            # until dates are captured, and use workload only as a soft overload signal.
            stats[str(row.get("TrainerName", ""))] = {
                "RecentCourses": float(row.get("RecentCourses", 0)),
                "AssignmentLoad": float(row.get("AssignmentLoad", 0)),
                "RecencyScore": 0.5,
                "OverloadPenalty": round(min(load_ratio * 0.5, 0.5), 3),
            }
    except Exception:
        pass
    return stats


def calculate_autotall_score_from_existing_data(
    trainer: pd.Series,
    course_name: Any,
    course_code: Any = "",
    required_skills: list[str] | None = None,
    demand_score: float = 0.0,
    workload: dict[str, float] | None = None,
) -> tuple[float, dict[str, float | str]]:
    """AutoTall approximation using signals already present in the notebook models."""
    required_skills = required_skills or _mk_infer_required_skills(course_code, course_name)
    current_skills = trainer.get("CurrentSkills", []) or []
    if not isinstance(current_skills, list):
        current_skills = [x.strip() for x in str(current_skills).split(",") if x.strip()]

    current = {_mk_norm(x) for x in current_skills if _mk_norm(x)}
    current_text = " ".join(sorted(current))
    current_tokens = set(current_text.split())
    required = [_mk_norm(x) for x in required_skills if _mk_norm(x)]

    def has_skill(skill: Any) -> bool:
        norm = _mk_norm(skill)
        if not norm:
            return False
        if norm in current:
            return True
        if norm in current_text:
            return True
        tokens = set(norm.split())
        if tokens and tokens.issubset(current_tokens):
            return True
        aliases = {
            "power bi": ["power bi", "dashboard", "pl-300"],
            "dax": ["dax", "pl-300"],
            "sql": ["sql", "transact-sql", "t-sql", "querying data"],
            "t-sql": ["transact-sql", "sql", "querying data"],
            "microsoft fabric": ["fabric", "dp-600", "dp-700"],
            "lakehouse": ["lakehouse", "fabric", "dp-700"],
            "pipelines": ["pipeline", "data factory", "fabric"],
            "azure ai": ["azure ai", "ai-102", "ai-900"],
            "prompt engineering": ["prompt", "ai agent", "generative ai"],
            "python": ["python"],
            "data modeling": ["data modeling", "model", "semantic"],
            "visualization": ["visual", "dashboard", "power bi"],
        }
        for key, vals in aliases.items():
            if norm == key or key in norm:
                return any(v in current_text for v in vals)
        return False

    missing = [skill for skill in required_skills if _mk_norm(skill) and not has_skill(skill)]
    skill_match = 1 - (len(missing) / max(len(required), 1)) if required else 0.0

    feedback = min(max(_mk_safe_number(trainer.get("Feedback"), 4.2) / 5, 0), 1)
    availability = min(max(_mk_safe_number(trainer.get("Availability"), 0.65), 0), 1)
    workload = workload or {}
    recency = min(max(_mk_safe_number(workload.get("RecencyScore"), 0.5), 0), 1)
    overload = min(max(_mk_safe_number(workload.get("OverloadPenalty"), 0.5), 0), 1)
    demand = min(max(_mk_safe_number(demand_score, 0), 0), 1)

    score = (
        0.30 * skill_match +
        0.20 * demand +
        0.20 * feedback +
        0.15 * availability +
        0.10 * recency -
        0.05 * overload
    )
    score = round(max(0, min(1, score)), 4)
    return score, {
        "SkillMatch": round(skill_match, 3),
        "Demand": round(demand, 3),
        "Feedback": round(feedback, 3),
        "Availability": round(availability, 3),
        "Recency": round(recency, 3),
        "OverloadPenalty": round(overload, 3),
        "MissingSkills": ", ".join(missing),
    }


def build_autotall_allocation_simulator(
    profiles: pd.DataFrame,
    catalog: pd.DataFrame | None = None,
    opportunity: pd.DataFrame | None = None,
    demand_summary: pd.DataFrame | None = None,
    top_courses_per_trainer: int = 8,
) -> pd.DataFrame:
    """
    Reverse-engineered AutoTall simulator.
    Uses RMS opportunity cache when present and falls back to the maintained future-ready catalog.
    """
    if profiles is None or profiles.empty:
        return pd.DataFrame()
    opportunity = opportunity if opportunity is not None else pd.DataFrame()
    catalog = catalog if catalog is not None else pd.DataFrame()
    demand_summary = demand_summary if demand_summary is not None else pd.DataFrame()
    demand_lookup = {}
    if not demand_summary.empty and {"CourseName", "CurrentDemandScore"}.issubset(demand_summary.columns):
        demand_lookup = demand_summary.set_index("CourseName")["CurrentDemandScore"].to_dict()

    profile_lookup = {str(r.get("TrainerName", "")): r for _, r in profiles.iterrows()}
    workload_lookup = _mk_workload_stats()
    rows = []

    if opportunity is not None and not opportunity.empty and {"TrainerName", "CourseName"}.issubset(opportunity.columns):
        base = opportunity.copy()
        base["DemandScore"] = pd.to_numeric(base.get("DemandScore", 0), errors="coerce").fillna(0)
        base["OpportunityScore"] = pd.to_numeric(base.get("OpportunityScore", 0), errors="coerce").fillna(0)
        base = base.sort_values(["TrainerName", "OpportunityScore", "DemandScore"], ascending=[True, False, False])
        base = base.groupby("TrainerName", group_keys=False).head(max(top_courses_per_trainer, 1))
        for _, row in base.iterrows():
            trainer_name = str(row.get("TrainerName", ""))
            if trainer_name not in profile_lookup:
                continue
            course_name = row.get("CourseName", "")
            course_code = row.get("CourseCode", "")
            demand = demand_lookup.get(course_name, row.get("DemandScore", 0))
            skills = _mk_infer_required_skills(course_code, course_name, row.get("MissingSkills", ""))
            score, parts = calculate_autotall_score_from_existing_data(
                profile_lookup[trainer_name], course_name, course_code, skills, demand, workload_lookup.get(trainer_name, {})
            )
            missing_list = [x.strip() for x in str(parts.get("MissingSkills", "")).split(",") if x.strip()]
            best_skill = missing_list[0] if missing_list else "Update RMS evidence"
            improvement = round(0.30 / max(len(skills), 1), 4) if missing_list else 0.02
            rows.append({
                "TrainerName": trainer_name,
                "CourseName": course_name,
                "CourseCode": course_code,
                "AutoTallScore": score,
                "AutoTallBand": "Push now" if score >= 0.72 else ("Near ready" if score >= 0.58 else "Build first"),
                "BestSkillBoost": best_skill,
                "EstimatedBoostIfAdded": improvement,
                **parts,
                "Source": "RMS opportunity cache",
            })
    else:
        for _, trainer in profiles.iterrows():
            trainer_name = str(trainer.get("TrainerName", ""))
            for _, course in catalog.iterrows():
                course_name = course.get("CourseName", "")
                course_code = course.get("CourseCode", "")
                demand = demand_lookup.get(course_name, course.get("DemandScore", 0))
                skills = list(course.get("Skills", []) or [])
                score, parts = calculate_autotall_score_from_existing_data(
                    trainer, course_name, course_code, skills, demand, workload_lookup.get(trainer_name, {})
                )
                missing_list = [x.strip() for x in str(parts.get("MissingSkills", "")).split(",") if x.strip()]
                best_skill = missing_list[0] if missing_list else "Update RMS evidence"
                improvement = round(0.30 / max(len(skills), 1), 4) if missing_list else 0.02
                rows.append({
                    "TrainerName": trainer_name,
                    "CourseName": course_name,
                    "CourseCode": course_code,
                    "AutoTallScore": score,
                    "AutoTallBand": "Push now" if score >= 0.72 else ("Near ready" if score >= 0.58 else "Build first"),
                    "BestSkillBoost": best_skill,
                    "EstimatedBoostIfAdded": improvement,
                    **parts,
                    "Source": "Future-ready catalog",
                })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["TrainerName", "AutoTallScore", "EstimatedBoostIfAdded"], ascending=[True, False, False]).reset_index(drop=True)


def enrich_action_plan_v2(action_plan: pd.DataFrame, demand_summary: pd.DataFrame, opportunity_matrix: pd.DataFrame, autotall_simulator: pd.DataFrame | None = None) -> pd.DataFrame:
    if action_plan.empty:
        return action_plan
    out = action_plan.copy()
    demand_top = demand_summary.head(3)["CourseName"].tolist() if demand_summary is not None and not demand_summary.empty and "CourseName" in demand_summary.columns else []
    autotall_simulator = autotall_simulator if autotall_simulator is not None else pd.DataFrame()

    quick_lookup = {}
    if opportunity_matrix is not None and not opportunity_matrix.empty and {"TrainerName", "CourseName", "Category"}.issubset(opportunity_matrix.columns):
        quick = opportunity_matrix[opportunity_matrix["Category"].isin(["Quick Win", "Strategic Upgrade"])]
        if not quick.empty:
            quick_lookup = quick.groupby("TrainerName")["CourseName"].apply(lambda s: ", ".join(list(dict.fromkeys(map(str, s)))[:3])).to_dict()

    autotall_lookup = {}
    if not autotall_simulator.empty and {"TrainerName", "CourseName", "AutoTallScore"}.issubset(autotall_simulator.columns):
        sim = autotall_simulator.sort_values(["TrainerName", "AutoTallScore"], ascending=[True, False])
        autotall_lookup = sim.groupby("TrainerName").head(1).set_index("TrainerName").to_dict("index")

    out["TopOpportunityCourses"] = out.apply(
        lambda r: quick_lookup.get(r.get("TrainerName"), ", ".join(demand_top[:3]) or "No saved demand signal yet"),
        axis=1,
    )
    out["AutoTallBestCourse"] = out.apply(lambda r: autotall_lookup.get(r.get("TrainerName"), {}).get("CourseName", ""), axis=1)
    out["AutoTallScore"] = out.apply(lambda r: autotall_lookup.get(r.get("TrainerName"), {}).get("AutoTallScore", ""), axis=1)
    out["AutoTallBand"] = out.apply(lambda r: autotall_lookup.get(r.get("TrainerName"), {}).get("AutoTallBand", ""), axis=1)
    out["AutoTallBestSkillBoost"] = out.apply(lambda r: autotall_lookup.get(r.get("TrainerName"), {}).get("BestSkillBoost", ""), axis=1)
    out["AutoTallEstimatedBoost"] = out.apply(lambda r: autotall_lookup.get(r.get("TrainerName"), {}).get("EstimatedBoostIfAdded", ""), axis=1)
    out["90DayGrowthTarget"] = out.apply(
        lambda r: f"After {r.get('ConcreteExamTarget', r.get('UpgradeNext', 'target'))}, build delivery readiness for: {r.get('TopOpportunityCourses', 'next high-demand course')}",
        axis=1,
    )
    out["180DayPositioning"] = out.apply(
        lambda r: f"Position as primary backup/lead after proof, delivery notes, RMS skills, and trainer evidence are updated for: {r.get('TopOpportunityCourses', r.get('UpgradeCourseName', 'future-ready courses'))}",
        axis=1,
    )

    def readiness_score(row):
        cleared = _mk_safe_number(row.get("ClearedCertCount"), 0)
        pending = _mk_safe_number(row.get("PendingCertCount"), 0)
        skills = min(_mk_safe_number(row.get("SkillCount"), 0) / 20, 1)
        has_exam_target = 1 if str(row.get("ConcreteExamTarget", "")).strip() else 0
        has_opportunity = 1 if str(row.get("TopOpportunityCourses", "")).strip() and "No saved" not in str(row.get("TopOpportunityCourses", "")) else 0
        sim_score = _mk_safe_number(row.get("AutoTallScore"), 0)
        sim_component = min(sim_score, 1) * 15 if sim_score else 0
        score = 30 * min(cleared, 1) + 10 * min(pending, 1) + 20 * skills + 15 * has_exam_target + 10 * has_opportunity + sim_component
        return int(round(min(score, 100)))

    out["AllocationReadinessScore"] = out.apply(readiness_score, axis=1)
    out["AllocationReadinessBand"] = out["AllocationReadinessScore"].apply(
        lambda s: "Ready now" if _mk_safe_number(s, 0) >= 75 else ("Build proof fast" if _mk_safe_number(s, 0) >= 50 else "Evidence gap")
    )
    out["ManagerRisk"] = out.apply(
        lambda r: "High risk: no cleared certification proof" if str(r.get("ClearedCertCount", "0")) == "0" else "Manageable: has some certification proof",
        axis=1,
    )
    return out


def display_unified_manager_cockpit(
    action_plan: pd.DataFrame,
    team_strategy: pd.DataFrame,
    allocation: pd.DataFrame | None = None,
    demand_summary: pd.DataFrame | None = None,
    opportunity_matrix: pd.DataFrame | None = None,
    autotall_simulator: pd.DataFrame | None = None,
    title: str = "Manager Cockpit",
) -> pd.DataFrame:
    """
    Single-screen manager view for team strategy, prediction, trainer growth path,
    certification/proof target, and allocation readiness.
    """
    if action_plan.empty:
        print("No manager action plan available.")
        return action_plan

    allocation = allocation if allocation is not None else pd.DataFrame()
    demand_summary = demand_summary if demand_summary is not None else pd.DataFrame()
    opportunity_matrix = opportunity_matrix if opportunity_matrix is not None else pd.DataFrame()
    autotall_simulator = autotall_simulator if autotall_simulator is not None else pd.DataFrame()
    trainer_rows = action_plan.fillna("").astype(str).to_dict("records")
    team_rows = team_strategy.fillna("").astype(str).to_dict("records") if not team_strategy.empty else []
    allocation_rows = allocation.fillna("").astype(str).head(12).to_dict("records") if not allocation.empty else []
    demand_rows = demand_summary.fillna("").astype(str).head(8).to_dict("records") if not demand_summary.empty else []
    matrix_rows = opportunity_matrix.fillna("").astype(str).head(80).to_dict("records") if not opportunity_matrix.empty else []
    simulator_rows = autotall_simulator.fillna("").astype(str).head(80).to_dict("records") if not autotall_simulator.empty else []
    payload_json = json.dumps(
        {"trainers": trainer_rows, "team": team_rows, "allocation": allocation_rows, "demand": demand_rows, "matrix": matrix_rows, "simulator": simulator_rows},
        ensure_ascii=False,
        default=str,
    ).replace("</", "<\\/")

    html = f"""
    <div id="managerCockpit" style="font-family:Segoe UI, Arial, sans-serif; color:#0f172a;">
      <style>
        #managerCockpit * {{box-sizing:border-box;}}
        #managerCockpit .topbar {{display:flex; justify-content:space-between; align-items:end; gap:12px; margin:6px 0 12px; flex-wrap:wrap;}}
        #managerCockpit label {{font-size:12px; color:#475569; display:block; margin-bottom:4px;}}
        #managerCockpit select {{border:1px solid #cbd5e1; border-radius:7px; padding:9px 10px; min-width:280px; background:white; font-size:14px;}}
        #managerCockpit .summary {{display:grid; grid-template-columns:repeat(4,minmax(130px,1fr)); gap:8px; margin:10px 0 12px;}}
        #managerCockpit .summaryBox {{border:1px solid #e5e7eb; background:#f8fafc; border-radius:7px; padding:9px;}}
        #managerCockpit .summaryBox b {{display:block; font-size:19px;}}
        #managerCockpit .summaryBox span {{font-size:11px; color:#64748b;}}
        #managerCockpit .layout {{display:grid; grid-template-columns:minmax(320px, 36%) minmax(480px, 64%); gap:14px; align-items:start;}}
        #managerCockpit .panel, #managerCockpit .trainerCard {{border:1px solid #d1d5db; border-radius:8px; background:#fff; padding:12px;}}
        #managerCockpit .panel h3 {{margin:0 0 6px; font-size:17px;}}
        #managerCockpit .muted {{font-size:12px; color:#64748b;}}
        #managerCockpit .teamItem {{border-top:1px solid #e5e7eb; padding:9px 0;}}
        #managerCockpit .teamItem:first-child {{border-top:0;}}
        #managerCockpit .tag {{display:inline-block; border-radius:999px; padding:2px 8px; font-size:11px; border:1px solid #bfdbfe; background:#eff6ff; color:#1d4ed8; margin-bottom:5px;}}
        #managerCockpit .tagGreen {{border-color:#86efac; background:#dcfce7; color:#166534;}}
        #managerCockpit .tagYellow {{border-color:#fcd34d; background:#fef3c7; color:#92400e;}}
        #managerCockpit .trainerHead {{display:flex; justify-content:space-between; gap:8px; align-items:start; flex-wrap:wrap; margin-bottom:10px;}}
        #managerCockpit .trainerHead h2 {{font-size:22px; margin:0;}}
        #managerCockpit .decision {{display:inline-block; border-radius:999px; padding:4px 10px; font-size:12px; border:1px solid #99f6e4; background:#ecfdf5; color:#065f46;}}
        #managerCockpit .decision.warn {{border-color:#fcd34d; background:#fffbeb; color:#92400e;}}
        #managerCockpit .kpis {{display:grid; grid-template-columns:repeat(4,minmax(90px,1fr)); gap:8px; margin:8px 0 12px;}}
        #managerCockpit .kpi, #managerCockpit .miniBox {{background:#f8fafc; border:1px solid #e5e7eb; border-radius:7px; padding:9px;}}
        #managerCockpit .kpi b {{display:block; font-size:20px;}}
        #managerCockpit .kpi span {{font-size:11px; color:#64748b;}}
        #managerCockpit .miniGrid {{display:grid; grid-template-columns:1fr 1fr; gap:8px; margin:8px 0 10px;}}
        #managerCockpit .progress {{height:8px; background:#e5e7eb; border-radius:99px; overflow:hidden; margin-top:5px;}}
        #managerCockpit .progress span {{display:block; height:100%; background:#10b981;}}
        #managerCockpit .section {{border-top:1px solid #e5e7eb; padding-top:10px; margin-top:10px;}}
        #managerCockpit .section h4 {{margin:0 0 6px; font-size:12px; color:#475569; text-transform:uppercase; letter-spacing:.02em;}}
        #managerCockpit ul {{margin:6px 0 0 18px; padding:0;}}
        #managerCockpit li {{margin:4px 0; line-height:1.35;}}
        #managerCockpit .upgrade {{font-size:18px; font-weight:800;}}
        #managerCockpit .target {{font-weight:800; color:#b45309;}}
        #managerCockpit .empty {{padding:12px; color:#64748b; background:#f8fafc; border-radius:7px;}}
        @media (max-width: 980px) {{
          #managerCockpit .layout {{grid-template-columns:1fr;}}
          #managerCockpit .kpis, #managerCockpit .summary {{grid-template-columns:repeat(2,minmax(90px,1fr));}}
          #managerCockpit .miniGrid {{grid-template-columns:1fr;}}
        }}
      </style>

      <div class="topbar">
        <div>
          <h2 style="margin:0;">{title}</h2>
          <div class="muted">One-screen view for team strategy, trainer upgrades, 45/90/180-day growth path, and allotment focus.</div>
        </div>
        <div><label>Trainer</label><select id="mcTrainer"></select></div>
      </div>
      <div class="summary" id="mcSummary"></div>
      <div class="layout">
        <div class="panel">
          <h3>Team Strategy Board</h3>
          <div class="muted">Common strengths, cohort upgrades, allocation signals, and demand trend.</div>
          <div id="mcTeam"></div>
          <div class="section"><h4>Allocation Signals</h4><div id="mcAllocation"></div></div>
          <div class="section"><h4>Demand Prediction</h4><div id="mcDemand"></div></div>
        </div>
        <div id="mcTrainerDetails"></div>
      </div>
    </div>
    <script>
    (function() {{
      const data = {payload_json};
      const root = document.getElementById('managerCockpit');
      const sel = root.querySelector('#mcTrainer');
      const summaryBox = root.querySelector('#mcSummary');
      const teamBox = root.querySelector('#mcTeam');
      const allocBox = root.querySelector('#mcAllocation');
      const demandBox = root.querySelector('#mcDemand');
      const detailBox = root.querySelector('#mcTrainerDetails');
      function esc(v) {{ return String(v || '').replace(/[&<>"']/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[ch])); }}
      function splitPipes(v) {{ return String(v || '').split('|').map(x => x.trim()).filter(Boolean); }}
      function splitComma(v) {{ return String(v || '').split(',').map(x => x.trim()).filter(Boolean); }}
      const trainers = [...new Set(data.trainers.map(r => r.TrainerName).filter(Boolean))];
      sel.innerHTML = '<option value="">All trainers</option>' + trainers.map(t => `<option value="${{esc(t)}}">${{esc(t)}}</option>`).join('');

      function renderSummary() {{
        const ready = data.trainers.filter(r => Number(r.AllocationReadinessScore || 0) >= 75).length;
        const proof = data.trainers.filter(r => String(r.ManagerRisk || '').toLowerCase().includes('high risk')).length;
        const quick = (data.matrix || []).filter(r => r.Category === 'Quick Win').length;
        const strategic = (data.matrix || []).filter(r => r.Category === 'Strategic Upgrade').length;
        const pushNow = (data.simulator || []).filter(r => r.AutoTallBand === 'Push now').length;
        summaryBox.innerHTML = `
          <div class="summaryBox"><b>${{esc(data.trainers.length)}}</b><span>trainers tracked</span></div>
          <div class="summaryBox"><b>${{esc(ready)}}</b><span>ready for allotment push</span></div>
          <div class="summaryBox"><b>${{esc(proof)}}</b><span>need cert/proof attention</span></div>
          <div class="summaryBox"><b>${{esc(pushNow || (quick + strategic))}}</b><span>allocation push signals</span></div>`;
      }}

      function renderTeam() {{
        const rows = data.team || [];
        if (!rows.length) {{ teamBox.innerHTML = '<div class="empty">No common team signal yet. Run option 1 after RMS data is refreshed.</div>'; return; }}
        teamBox.innerHTML = rows.slice(0, 8).map(r => `<div class="teamItem">
          <span class="tag ${{String(r.Priority).includes('upgrade') ? 'tagGreen' : ''}}">${{esc(r.Priority)}} &middot; ${{esc(r.TrainerCount)}} trainer(s)</span>
          <div><b>${{esc(r.Focus)}}</b></div><div class="muted">${{esc(r.Trainers)}}</div><div style="margin-top:4px;">${{esc(r.ManagerMove)}}</div>
        </div>`).join('');
      }}

      function renderAllocation() {{
        const rows = data.allocation || [];
        if (!rows.length) {{ allocBox.innerHTML = '<div class="empty">No AutoTall allocation table was captured yet. Internal coverage signals are used in trainer plans.</div>'; return; }}
        allocBox.innerHTML = rows.slice(0, 6).map(r => `<div class="teamItem">
          <span class="tag tagYellow">${{esc(r.OpportunitySignal || 'Allocation signal')}}</span>
          <div><b>${{esc(r.CourseName)}}</b></div><div class="muted">My share: ${{esc(r.MyReporteeShare)}} &middot; Total allocations: ${{esc(r.TotalAllocations)}}</div>
        </div>`).join('');
      }}

      function renderDemand() {{
        const rows = data.demand || [];
        if (!rows.length) {{ demandBox.innerHTML = '<div class="empty">No demand history yet. Each option 1 refresh will make this smarter.</div>'; return; }}
        demandBox.innerHTML = rows.slice(0, 5).map(r => `<div class="teamItem">
          <span class="tag ${{String(r.Trend).includes('Rising') || String(r.Trend).includes('High') ? 'tagGreen' : ''}}">${{esc(r.Trend || 'Demand signal')}}</span>
          <div><b>${{esc(r.CourseName)}}</b></div><div class="muted">Demand: ${{esc(r.CurrentDemandScore)}} &middot; snapshots: ${{esc(r.Snapshots)}}</div>
        </div>`).join('');
      }}

      function trainerCard(r) {{
        const actions = splitPipes(r.ThisWeekManagerAction);
        const gaps = splitComma(r.SkillGapToClose);
        const decisionClass = String(r.ManagerDecision).toLowerCase().includes('need') ? 'warn' : '';
        const readiness = Math.max(0, Math.min(100, Number(r.AllocationReadinessScore || 0)));
        return `<div class="trainerCard">
          <div class="trainerHead"><div><h2>${{esc(r.TrainerName)}}</h2><span class="decision ${{decisionClass}}">${{esc(r.ManagerDecision)}}</span></div><div class="tag tagGreen">${{esc(r.Exam45DayTarget)}}</div></div>
          <div class="kpis"><div class="kpi"><b>${{esc(r.SkillCount)}}</b><span>skills/courses</span></div><div class="kpi"><b>${{esc(r.PastCourseCount)}}</b><span>delivered</span></div><div class="kpi"><b>${{esc(r.ClearedCertCount)}}</b><span>cleared</span></div><div class="kpi"><b>${{esc(r.PendingCertCount)}}</b><span>pending</span></div></div>
          <div class="miniGrid"><div class="miniBox"><b>Allocation readiness: ${{esc(r.AllocationReadinessScore || 0)}}/100</b><div class="progress"><span style="width:${{readiness}}%;"></span></div><div class="muted">${{esc(r.AllocationReadinessBand || '')}}</div></div><div class="miniBox"><b>Risk</b><div>${{esc(r.ManagerRisk || '')}}</div></div></div>
          <div class="section"><h4>AutoTall Simulator</h4><ul><li><b>${{esc(r.AutoTallBestCourse || 'No simulator course yet')}}</b> <span class="tag">${{esc(r.AutoTallBand || '')}}</span></li><li>Estimated score: <b>${{esc(r.AutoTallScore || 'n/a')}}</b></li><li>Highest boost skill: <b>${{esc(r.AutoTallBestSkillBoost || 'Update RMS evidence')}}</b> ${{r.AutoTallEstimatedBoost ? '(+' + esc(r.AutoTallEstimatedBoost) + ')' : ''}}</li></ul></div>
          <div class="section"><h4>Upgrade Next</h4><div class="upgrade">${{esc(r.UpgradeNext)}} <span style="font-weight:400;">&middot; ${{esc(r.UpgradeCourseName)}}</span></div><ul><li>${{esc(r.WhyThisMatters)}}</li></ul></div>
          <div class="section"><h4>Growth Path</h4><ul><li><b>45 days:</b> ${{esc(r.Exam45DayTarget)}}</li><li><b>90 days:</b> ${{esc(r['90DayGrowthTarget'] || '')}}</li><li><b>180 days:</b> ${{esc(r['180DayPositioning'] || '')}}</li></ul></div>
          <div class="section"><h4>Skill Gap To Close</h4><ul>${{(gaps.length ? gaps : ['No major skill gap detected; update evidence and proof.']).map(g => `<li>${{esc(g)}}</li>`).join('')}}</ul></div>
          <div class="section"><h4>Exam / Proof Move</h4><ul><li><span class="target">${{esc(r.Exam45DayTarget)}}</span></li><li><b>${{esc(r.ConcreteExamTarget)}}</b> &middot; ${{esc(r.ExamTargetName)}}</li><li>${{esc(r.CertificationMove)}}</li><li><i>${{esc(r.ProofPack)}}</i></li></ul></div>
          <div class="section"><h4>Allotment Strategy</h4><ul><li>${{esc(r.AllotmentStrategy)}}</li><li><b>Best opportunity courses:</b> ${{esc(r.TopOpportunityCourses || '')}}</li></ul></div>
          <div class="section"><h4>This Week Manager Actions</h4><ul>${{actions.map(a => `<li>${{esc(a)}}</li>`).join('')}}</ul></div>
        </div>`;
      }}

      function renderDetails() {{
        const selected = sel.value;
        const visible = data.trainers.filter(r => !selected || r.TrainerName === selected);
        detailBox.innerHTML = visible.length ? visible.map(trainerCard).join('<div style="height:10px;"></div>') : '<div class="empty">No trainer found.</div>';
      }}

      renderSummary(); renderTeam(); renderAllocation(); renderDemand(); renderDetails();
      sel.addEventListener('input', renderDetails);
    }})();
    </script>
    """
    _mk_display_html(html)
    return action_plan


def display_manager_action_board(action_plan: pd.DataFrame, title: str = "Manager Action Board") -> pd.DataFrame:
    """
    Simple decision dashboard for layman/manager use.
    """
    if action_plan.empty:
        print("No manager action plan available.")
        return action_plan

    rows = action_plan.fillna("").astype(str).to_dict("records")
    payload_json = json.dumps({"rows": rows}, ensure_ascii=False).replace("</", "<\\/")
    html = f"""
    <div id="managerActionBoard" style="font-family:Segoe UI, Arial, sans-serif; color:#1f2937;">
      <style>
        #managerActionBoard .bar {{display:flex; gap:10px; flex-wrap:wrap; align-items:end; margin:12px 0;}}
        #managerActionBoard label {{font-size:12px; color:#4b5563; display:block; margin-bottom:4px;}}
        #managerActionBoard select {{border:1px solid #cbd5e1; border-radius:6px; padding:7px 8px; min-width:220px; background:white;}}
        #managerActionBoard .grid {{display:block; margin-top:12px; max-width:1100px;}}
        #managerActionBoard .card {{border:1px solid #d1d5db; border-radius:8px; padding:12px; background:#fff; margin-bottom:10px;}}
        #managerActionBoard .card h4 {{margin:0 0 6px; font-size:16px; color:#111827;}}
        #managerActionBoard .decision {{display:inline-block; border-radius:999px; padding:3px 9px; background:#ecfdf5; border:1px solid #99f6e4; font-size:12px; margin-bottom:8px;}}
        #managerActionBoard .label {{font-size:11px; color:#64748b; text-transform:uppercase; margin-top:8px;}}
        #managerActionBoard .value {{font-size:13px; line-height:1.35;}}
        #managerActionBoard .kpis {{display:grid; grid-template-columns:repeat(4,minmax(90px,140px)); gap:6px; margin:8px 0;}}
        #managerActionBoard .kpi {{background:#f8fafc; border:1px solid #e5e7eb; border-radius:6px; padding:7px;}}
        #managerActionBoard .kpi b {{display:block; font-size:16px;}}
        #managerActionBoard .kpi span {{font-size:11px; color:#64748b;}}
      </style>
      <h3 style="margin:4px 0 2px;">{title}</h3>
      <div style="font-size:12px;color:#64748b;">Use this first: it tells you what to push for each trainer and how to position them for allocation.</div>
      <div class="bar"><div><label>Trainer</label><select id="mabTrainer"></select></div></div>
      <div id="mabCards" class="grid"></div>
    </div>
    <script>
    (function() {{
      const data = {payload_json};
      const root = document.getElementById('managerActionBoard');
      const sel = root.querySelector('#mabTrainer');
      const cards = root.querySelector('#mabCards');
      function esc(v) {{ return String(v || '').replace(/[&<>"']/g, ch => ({{'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}}[ch])); }}
      const trainers = [...new Set(data.rows.map(r => r.TrainerName).filter(Boolean))];
      sel.innerHTML = '<option value="">All trainers</option>' + trainers.map(t => `<option value="${{esc(t)}}">${{esc(t)}}</option>`).join('');
      function card(r) {{
        return `<div class="card">
          <h4>${{esc(r.TrainerName)}}</h4>
          <div class="decision">${{esc(r.ManagerDecision)}}</div>
          <div class="kpis">
            <div class="kpi"><b>${{esc(r.SkillCount)}}</b><span>skills/courses</span></div>
            <div class="kpi"><b>${{esc(r.PastCourseCount)}}</b><span>delivered</span></div>
            <div class="kpi"><b>${{esc(r.ClearedCertCount)}}</b><span>cleared</span></div>
            <div class="kpi"><b>${{esc(r.PendingCertCount)}}</b><span>pending</span></div>
          </div>
          <div class="label">Upgrade next</div><div class="value"><b>${{esc(r.UpgradeNext)}}</b><br>${{esc(r.UpgradeCourseName)}}</div>
          <div class="label">Why this matters</div><div class="value">${{esc(r.WhyThisMatters)}}</div>
          <div class="label">Skill gap to close</div><div class="value">${{esc(r.SkillGapToClose)}}</div>
          <div class="label">Certification move</div><div class="value">${{esc(r.CertificationMove)}}</div>
          <div class="label">45-day exam/proof target</div><div class="value"><b>${{esc(r.Exam45DayTarget)}}</b></div>
          <div class="label">Allotment strategy</div><div class="value">${{esc(r.AllotmentStrategy)}}</div>
          <div class="label">This week's manager action</div><div class="value">${{esc(r.ThisWeekManagerAction)}}</div>
        </div>`;
      }}
      function render() {{
        const visible = data.rows.filter(r => !sel.value || r.TrainerName === sel.value);
        cards.innerHTML = visible.map(card).join('');
      }}
      sel.addEventListener('input', render);
      render();
    }})();
    </script>
    """
    _mk_display_html(html)
    return action_plan


def _mk_signal_paths() -> Dict[str, Path]:
    base = _mk_paths()["output"]
    return {
        "popular": base / "RMS_Popular_Courses_Latest.xlsx",
        "allocation": base / "AutoTall_Allocation_Latest.xlsx",
        "opportunity": base / "RMS_Opportunity_Latest.xlsx",
    }


def refresh_market_signal_cache(driver=None) -> Dict[str, Path]:
    """
    Option 1 uses this to refresh RMS market/allocation signals once and save them as Excel.
    Option 3 reads these saved files; it does not scrape/login again.
    """
    paths = _mk_signal_paths()
    saved = {}

    popular = fetch_rms_popular_courses_with_driver(driver) if driver is not None else pd.DataFrame()
    if popular.empty:
        popular = pd.DataFrame()
    try:
        popular.to_excel(paths["popular"], index=False)
        saved["popular"] = paths["popular"]
    except Exception as exc:
        print(f"Popular-course cache save failed. Close the Excel file if it is open: {paths['popular']} ({exc})")

    allocation = pd.DataFrame()
    if driver is not None:
        allocation = analyze_autotall_allocations(fetch_autotall_dashboard_with_driver(driver))
    try:
        allocation.to_excel(paths["allocation"], index=False)
        saved["allocation"] = paths["allocation"]
    except Exception as exc:
        print(f"AutoTall cache save failed. Close the Excel file if it is open: {paths['allocation']} ({exc})")

    opportunity = build_rms_popularity_opportunity_analysis(popular)
    try:
        opportunity.to_excel(paths["opportunity"], index=False)
        saved["opportunity"] = paths["opportunity"]
    except Exception as exc:
        print(f"Opportunity cache save failed. Close the Excel file if it is open: {paths['opportunity']} ({exc})")

    print("Market intelligence cache refreshed:")
    for name, path in saved.items():
        print(f" - {name}: {path}")
    return saved


def load_market_signal_cache() -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load saved option-1 intelligence signals for option 3.
    If cache is missing, opportunity falls back to internal delivered-course frequency.
    """
    paths = _mk_signal_paths()
    opportunity = pd.DataFrame()
    allocation = pd.DataFrame()

    for candidate in [paths["opportunity"]]:
        if candidate.exists():
            try:
                opportunity = pd.read_excel(candidate, engine="openpyxl")
                break
            except Exception:
                pass

    for candidate in [paths["allocation"]]:
        if candidate.exists():
            try:
                allocation = pd.read_excel(candidate, engine="openpyxl")
                break
            except Exception:
                pass

    if opportunity.empty:
        print("No saved RMS opportunity cache found. Using current trainer/course model as fallback.")
        opportunity = build_rms_popularity_opportunity_analysis(pd.DataFrame())
    else:
        print("Using saved RMS opportunity cache from option 1.")

    if allocation.empty:
        print("No saved AutoTall cache found. Allocation strategy will use internal coverage signals.")
    else:
        print("Using saved AutoTall allocation cache from option 1.")

    return opportunity, allocation


In [ ]:
"""
Cell 11 - Menu utilities (status + menu print)
"""

from __future__ import annotations
from pathlib import Path

def print_menu_status() -> None:
    """
    Print the current availability of normalized models.
    """
    try:
        trainer_file = MY_REPORTEES_FOLDER / "RMS_Normalized_Trainer_Model.xlsx"
        course_file  = MY_REPORTEES_FOLDER / "RMS_Normalized_Course_Model.xlsx"
        cert_file    = MY_REPORTEES_FOLDER / "RMS_Normalized_Certification_Model.xlsx"
    except NameError:
        trainer_file = Path("RMS_Normalized_Trainer_Model.xlsx")
        course_file  = Path("RMS_Normalized_Course_Model.xlsx")
        cert_file    = Path("RMS_Normalized_Certification_Model.xlsx")

    print("\nModel Status:")
    print(f" - Trainer Model: {'FOUND' if trainer_file.exists() else 'MISSING'} {trainer_file}")
    print(f" - Course Model : {'FOUND' if course_file.exists()  else 'MISSING'} {course_file}")
    print(f" - Cert Model   : {'FOUND' if cert_file.exists()    else 'MISSING'} {cert_file}")


def show_menu(show_dashboard_option: bool = True) -> None:
    """
    Print the console menu. Call this before reading user's choice.
    """
    WIDTH = 72

    def hr(ch: str = "-") -> None:
        print(ch * WIDTH)

    def title(txt: str) -> None:
        hr("=")
        print(txt.center(WIDTH))
        hr("=")

    title("RMS GOVERNANCE + TRAINER INTELLIGENCE CONSOLE")
    print_menu_status()

    print("\nAvailable Actions:\n")
    print(" 1. Refresh Governance Data")
    print("    - One RMS login, refreshes Excel models and market/allocation signal cache")

    print("\n 2. Trainer Solved Courses Dashboard")
    print("    - HTML dashboard with trainer, certification, exam-state, and course filters")

    print("\n 3. Unified Trainer Intelligence + RMS Opportunity + AutoTall Analysis")
    print("    - Trainer readiness, exams, recommendations, popular-course ratios, and allocation signals")
    print("    - Reads saved Excel intelligence from option 1; no RMS login/scrape here")

    print("\n 4. Exit Console")
    print()
    hr()


def cleanup_old_timestamped_outputs(delete: bool = False) -> list[Path]:
    """
    Find old timestamped output workbooks that are no longer needed.
    Pass delete=True only when you want to remove them.
    Core model files and *_Latest.xlsx files are preserved.
    """
    folder = Path(MY_REPORTEES_FOLDER)

    patterns = [
        "Trainer_Intelligence_Output_20*.xlsx",
        "Trainer_Solved_Courses_Dashboard_20*.xlsx",
        "Unified_Manager_Intelligence_20*.xlsx",
        "RMS_Popularity_Opportunity_20*.xlsx",
        "AutoTall_Allocation_Intelligence_20*.xlsx",
    ]
    files = []
    for pattern in patterns:
        files.extend(folder.glob(pattern))
    files = sorted(set(files))

    if not files:
        print("No old timestamped output workbooks found.")
        return []

    if delete:
        deleted = []
        for file in files:
            try:
                file.unlink()
                deleted.append(file)
            except Exception as exc:
                print(f"Could not delete {file}: {exc}")
        print(f"Deleted {len(deleted)} old timestamped output workbook(s).")
        return deleted

    print("Old timestamped output workbooks found. Run cleanup_old_timestamped_outputs(delete=True) to remove them:")
    for file in files:
        print(f" - {file.name}")
    return files





def ensure_certification_model_shell() -> Path:
    """
    Create the certification workbook shell if RMS certification extraction fails before
    producing a file. This keeps options 2/3 from getting stuck on a missing cert model.
    """
    cert_file = Path(MY_REPORTEES_FOLDER) / "RMS_Normalized_Certification_Model.xlsx"
    if cert_file.exists():
        return cert_file
    cert_file.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(columns=["TrainerID", "CourseID", "ExamName", "ExamCode", "Result", "ApprovalStatus", "Vendor", "ExamURL"])
    with pd.ExcelWriter(cert_file, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Trainer_Certification_Map", index=False)
    print(f"Created empty certification model shell: {cert_file}")
    return cert_file


def _empty_df(columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame(columns=columns)


def generate_managed_latest_workbooks(clean_old: bool = True) -> dict[str, Path]:
    """
    Regenerate the managed Latest workbooks from the current normalized models.
    If a report/cache workbook is missing, create it with the correct sheets/columns.
    The three normalized RMS model files are still produced by option 1 from RMS data.
    """
    outputs: dict[str, Path] = {}
    folder = Path(MY_REPORTEES_FOLDER)
    folder.mkdir(parents=True, exist_ok=True)

    try:
        ensure_future_reportee_template()
    except Exception as exc:
        print(f"Future reportee template check skipped: {exc}")

    solved = _empty_df([
        "TrainerID", "TrainerName", "CourseID", "CourseName", "AssignmentsDelivered",
        "QubitScore", "MinScoreRequired", "QubitQuestionCount", "IsFutureLive",
        "CertificationVendor", "CertificationName", "CertificationStatus", "ExamURL",
        "Result", "ApprovalStatus", "TrainerCertificationState", "HasValidCertificationLink",
    ])
    try:
        loaded_solved = load_trainer_course_solved_view()
        if not loaded_solved.empty:
            solved = loaded_solved
    except Exception as exc:
        print(f"Trainer solved latest workbook will be created as empty shell: {exc}")

    path = folder / "Trainer_Solved_Courses_Dashboard_Latest.xlsx"
    try:
        solved.to_excel(path, index=False)
        outputs["trainer_solved_dashboard"] = path
    except Exception as exc:
        print(f"Trainer solved latest workbook save failed. Close the file if open: {path} ({exc})")

    profiles = _empty_df(["TrainerID", "TrainerName", "CurrentSkills", "PastCourses", "ClearedCertifications", "PendingCertifications", "AssignedCertifications", "Feedback", "Availability"])
    exam_status = _empty_df(["TrainerName", "ExamCode", "ExamName", "CourseName", "Result", "ApprovalStatus", "ExamStatus"])
    catalog = TRAINER_INTELLIGENCE_CATALOG.copy() if "TRAINER_INTELLIGENCE_CATALOG" in globals() else _empty_df(["CourseCode", "CourseName", "Category", "DemandScore"])
    top_recommendations = _empty_df(["TrainerName", "RecommendedCourse", "CourseName", "Category", "AssignmentProbability", "Priority"])
    weekly_actions = _empty_df(["TrainerName", "NextBestCourse", "CourseName", "Priority", "AssignmentProbability", "NextActions"])
    opportunity = _empty_df(["TrainerName", "CourseName", "CourseCode", "DemandScore", "TrainerCourseRatio", "SkillMatchScore", "CertificationReadiness", "OpportunityScore", "RecommendedNextAction"])
    allocation = _empty_df(["CourseName", "SegmentOrCriteria", "TotalAllocations", "UniqueTrainers", "MyReporteeAllocations", "MyReporteeShare", "OpportunitySignal"])
    action_plan = _empty_df([
        "TrainerName", "ManagerDecision", "UpgradeNext", "UpgradeCourseName", "WhyThisMatters",
        "SkillGapToClose", "CertificationMove", "Exam45DayTarget", "ConcreteExamTarget",
        "ExamTargetName", "ProofPack", "ManagerReviewCadence", "AllotmentStrategy",
        "ThisWeekManagerAction", "90DayGrowthTarget", "180DayPositioning", "TopOpportunityCourses",
        "AllocationReadinessScore", "AllocationReadinessBand", "ManagerRisk",
    ])
    team_strategy = _empty_df(["Priority", "Focus", "CourseName", "TrainerCount", "Trainers", "ManagerMove"])
    demand_summary = _empty_df(["CourseName", "CurrentDemandScore", "DemandDelta", "Trend", "Snapshots"])
    opportunity_matrix = _empty_df(["TrainerName", "CourseName", "Category", "DemandScore", "SkillMatchScore", "Trend", "CertificationReadiness", "Reason", "ManagerMove"])
    autotall_simulator = _empty_df([
        "TrainerName", "CourseName", "CourseCode", "AutoTallScore", "AutoTallBand",
        "BestSkillBoost", "EstimatedBoostIfAdded", "SkillMatch", "Demand", "Feedback",
        "Availability", "Recency", "OverloadPenalty", "MissingSkills", "Source",
    ])

    try:
        profiles, exam_status, catalog = build_current_trainer_intelligence()
        recommendations = build_next_course_recommendations(profiles, catalog)
        top_recommendations = recommendations.groupby("TrainerName", group_keys=False).head(5) if not recommendations.empty else recommendations
        weekly_actions = build_weekly_next_actions(recommendations) if not recommendations.empty else weekly_actions
    except Exception as exc:
        print(f"Trainer intelligence data unavailable; creating workbook shell: {exc}")

    try:
        opportunity, allocation = load_market_signal_cache()
    except Exception as exc:
        print(f"Market signal cache unavailable; creating workbook shell: {exc}")

    cache_paths = _mk_signal_paths() if "_mk_signal_paths" in globals() else {}
    try:
        popular_path = cache_paths.get("popular")
        if popular_path and not Path(popular_path).exists():
            _empty_df(["CourseName", "CourseCode", "RMSPopularityRaw", "RMSCategory", "PopularityRatio", "DemandScore"]).to_excel(popular_path, index=False)
        opportunity_path = cache_paths.get("opportunity")
        if opportunity_path:
            opportunity.to_excel(opportunity_path, index=False)
            outputs["opportunity_cache"] = opportunity_path
        allocation_path = cache_paths.get("allocation")
        if allocation_path:
            allocation.to_excel(allocation_path, index=False)
            outputs["allocation_cache"] = allocation_path
    except Exception as exc:
        print(f"Cache shell/latest save skipped. Close cache Excel files if open: {exc}")

    try:
        action_plan = build_manager_action_plan(
            profiles=profiles,
            recommendations=top_recommendations,
            weekly_actions=weekly_actions,
            opportunity=opportunity,
            allocation=allocation,
        )
        demand_summary = build_predictive_demand_summary(opportunity)
        opportunity_matrix = build_opportunity_matrix(action_plan, opportunity, demand_summary)
        autotall_simulator = build_autotall_allocation_simulator(profiles, catalog, opportunity, demand_summary)
        action_plan = enrich_action_plan_v2(action_plan, demand_summary, opportunity_matrix, autotall_simulator)
        team_strategy = build_team_strategy_summary(profiles, action_plan)
    except Exception as exc:
        print(f"Manager action plan unavailable; creating workbook shell: {exc}")

    trainer_intel_path = folder / "Trainer_Intelligence_Output_Latest.xlsx"
    try:
        with pd.ExcelWriter(trainer_intel_path, engine="openpyxl") as writer:
            profiles.to_excel(writer, sheet_name="CurrentReadiness", index=False)
            exam_status.to_excel(writer, sheet_name="ExamStatus", index=False)
            top_recommendations.to_excel(writer, sheet_name="TopRecommendations", index=False)
            weekly_actions.to_excel(writer, sheet_name="WeeklyActions", index=False)
            catalog.to_excel(writer, sheet_name="DemandCatalog", index=False)
        outputs["trainer_intelligence"] = trainer_intel_path
    except Exception as exc:
        print(f"Trainer intelligence latest workbook save failed. Close the file if open: {trainer_intel_path} ({exc})")

    unified_path = folder / "Unified_Manager_Intelligence_Latest.xlsx"
    try:
        with pd.ExcelWriter(unified_path, engine="openpyxl") as writer:
            action_plan.to_excel(writer, sheet_name="ManagerActionBoard", index=False)
            team_strategy.to_excel(writer, sheet_name="TeamStrategy", index=False)
            profiles.to_excel(writer, sheet_name="TrainerReadiness", index=False)
            exam_status.to_excel(writer, sheet_name="ExamStatus", index=False)
            top_recommendations.to_excel(writer, sheet_name="TopRecommendations", index=False)
            weekly_actions.to_excel(writer, sheet_name="WeeklyActions", index=False)
            opportunity.to_excel(writer, sheet_name="OpportunitySignals", index=False)
            allocation.to_excel(writer, sheet_name="AutoTallSignals", index=False)
            demand_summary.to_excel(writer, sheet_name="DemandPrediction", index=False)
            opportunity_matrix.to_excel(writer, sheet_name="OpportunityMatrix", index=False)
            autotall_simulator.to_excel(writer, sheet_name="AutoTallSimulator", index=False)
        outputs["unified_manager_intelligence"] = unified_path
    except Exception as exc:
        print(f"Unified manager latest workbook save failed. Close the file if open: {unified_path} ({exc})")

    if clean_old:
        try:
            cleanup_old_timestamped_outputs(delete=True)
        except Exception as exc:
            print(f"Old timestamp cleanup skipped: {exc}")

    if outputs:
        print("Managed latest workbooks are ready:")
        for name, path in outputs.items():
            print(f" - {name}: {path}")
    return outputs


In [ ]:
"""
Cell 12 - Orchestrate full flow and route menu actions.
"""

from __future__ import annotations
from pathlib import Path

try:
    from IPython.display import clear_output
except Exception:
    def clear_output(wait: bool = False):
        print("\n" + "-" * 72 + "\n")



def _console_model_folder() -> Path:
    """Return the reportee model folder, even if earlier path cells were not run."""
    try:
        return Path(MY_REPORTEES_FOLDER)
    except NameError:
        candidates = [
            Path.home() / "OneDrive - Koenig Solutions Ltd" / "SkillEdge" / "myReportees",
            Path.home() / "OneDrive - Koenig Solutions Ltd" / "Documents" / "My Reportees",
            Path.home() / "OneDrive - Koenig Solutions Ltd" / "My Reportees",
            Path.home() / "OneDrive" / "Documents" / "My Reportees",
            Path.home() / "OneDrive" / "My Reportees",
        ]
        for folder in candidates:
            if folder.exists():
                return folder
        return Path.home() / "OneDrive - Koenig Solutions Ltd" / "SkillEdge" / "myReportees"


def _console_required_missing(choice: str) -> list[str]:
    missing = []
    if choice == "1":
        for name in ["login_and_get_driver", "show_trainer_information", "extract_trainer_course_mapping", "extract_trainer_certification_results"]:
            if name not in globals():
                missing.append(name)
    elif choice == "2":
        if "trainer_course_html_dashboard" not in globals():
            missing.append("trainer_course_html_dashboard")
    elif choice == "3":
        for name in ["trainer_intelligence_html_dashboard", "rms_popularity_opportunity_console", "autotall_allocation_console"]:
            if name not in globals():
                missing.append(name)
    return missing


def _driver_is_alive(driver) -> bool:
    if driver is None:
        return False
    try:
        _ = driver.current_url
        return True
    except Exception:
        return False


def _ensure_rms_session(driver, wait, manager_name: str | None = None):
    """Reuse the current RMS browser session. If there is no live session, open Chrome once."""
    if _driver_is_alive(driver):
        return driver, wait
    if not globals().get("SELENIUM_AVAILABLE", True):
        return None, None
    if "login_and_get_driver" not in globals():
        return None, None
    try:
        driver, wait, manager_name_final, meta = login_and_get_driver(
            manager_name=manager_name,
            headless=False,
            timeout_sec=30,
            threshold=90,
        )
        return driver, wait
    except Exception:
        return None, None


def main() -> None:
    import os
    import webbrowser

    dashboard_path = Path(r"C:\Users\Aishw\OneDrive - Koenig Solutions Ltd\trainings\AI-103T00-ENU-PowerPoint\rms_governance_dashboard.html")
    dashboard_path.parent.mkdir(parents=True, exist_ok=True)
    if not dashboard_path.exists():
        dashboard_path.write_text(r"""<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\" />
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
  <title>RMS Governance Console</title>
  <link href=\"https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css\" rel=\"stylesheet\">
  <style>
    body { font-family: Segoe UI, Arial, sans-serif; background: linear-gradient(180deg, #f8fafc 0%, #eef2ff 100%); }
    .hero { background: linear-gradient(135deg, #0f172a 0%, #1e293b 55%, #334155 100%); color: white; }
    .soft-card { box-shadow: 0 10px 30px rgba(15,23,42,.08); }
    .mono { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; }
  </style>
</head>
<body>
  <div class=\"container py-4\">
    <div class=\"hero rounded-4 p-4 mb-4 soft-card\">
      <div class=\"d-flex justify-content-between align-items-center gap-3 flex-wrap\">
        <div>
          <div class=\"text-uppercase small opacity-75\">RMS Governance Console</div>
          <h1 class=\"display-6 mb-0\">Trainer Intelligence</h1>
          <div class=\"opacity-75\">Launch succeeded. Use the notebook cell again to refresh or rebuild the dashboard.</div>
        </div>
        <span class=\"badge text-bg-light text-dark rounded-pill px-3 py-2\">Bootstrap 5</span>
      </div>
    </div>
    <div class=\"card soft-card\">
      <div class=\"card-body\">
        <h2 class=\"h4\">Dashboard file created</h2>
        <p class=\"mb-0\">The notebook recreated this file because it was missing. Replace this placeholder with the full dashboard content when ready.</p>
      </div>
    </div>
  </div>
</body>
</html>""", encoding='utf-8')

    try:
        import subprocess
        subprocess.Popen(["cmd", "/c", "start", "", str(dashboard_path)], shell=False)
    except Exception:
        try:
            os.startfile(str(dashboard_path))
        except Exception:
            webbrowser.open(dashboard_path.as_uri(), new=1, autoraise=True)

    print(f"Opened dashboard in browser: {dashboard_path}")


In [ ]:
"""
Cell 13 - Run the console main loop (interactive).
"""

# In notebooks, run this cell to launch the menu:
main()